In [ ]:
"""
=============================================================================
Shared configuration and helpers for the sciMeDseq -> reference pipeline
=============================================================================

Every step imports this with:

    from pipeline_config import *

Edit parameters here only. The steps themselves contain no configuration.
=============================================================================
"""

import os
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp

warnings.filterwarnings('ignore', category=FutureWarning)
sc.settings.verbosity = 1

# ----------------------------- paths -----------------------------------------
DCM_MATRIX = ('/media/erlantz/71DE167A75E99BB4/Pool32/Combi/Matrix/'
              'Pool32B_NPC_noDup_rcMatrix_DCM_geneEnsemble.tsv')
DCM_META   = ('/media/erlantz/71DE167A75E99BB4/Pool32/Combi/stats/'
              'stats_intestine.csv')
REF_H5AD   = ('/media/erlantz/71DE167A75E99BB4/Pool40_rebuttal/Jona_py/'
              'JK_Anal_ysis/haber_ref.h5ad')
OUT_DIR    = ('/media/erlantz/71DE167A75E99BB4/Pool32/Combi/Matrix/'
              'gene_scoring/supervised_pipeline')
INTEG_DIR  = ('/media/erlantz/71DE167A75E99BB4/Pool32/Combi/Python/'
              'integration/')
FIG_DIR = os.path.join(OUT_DIR, 'figures')
OBJ_DIR = os.path.join(OUT_DIR, 'objects')
TAB_DIR = os.path.join(OUT_DIR, 'tables')
for _d in (FIG_DIR, OBJ_DIR, TAB_DIR, INTEG_DIR):
    os.makedirs(_d, exist_ok=True)

# intermediate objects, one per step
H5_DCM_CLEAN  = os.path.join(OBJ_DIR, '01_dcm_clean.h5ad')
H5_COMBINED   = os.path.join(OBJ_DIR, '02_combined.h5ad')
H5_INTEGRATED = os.path.join(OBJ_DIR, '02b_combined_integrated.h5ad')
H5_CLASSIFIED = os.path.join(OBJ_DIR, '03_combined_classified.h5ad')
CSV_GENESETS  = os.path.join(TAB_DIR, 'final_gene_sets.csv')

# ----------------------------- QC thresholds ---------------------------------
MIN_GENES          = 200     # as in the original pipeline
MIN_CELLS_PER_GENE = 0
MAX_MITO_PCT       = 20.0    # set to None to skip
QC_NMADS           = 5       # MAD cut for count-depth outliers

# background / ambient
BG_MIN_COUNTS  = 1           # sub-threshold barcodes need at least this many counts
BG_MAX_GENES   = MIN_GENES   # ...and fewer than this many genes, to be background
AMBIENT_NMADS  = 3           # MAD cut on background similarity
AMBIENT_ACTION = 'flag'      # 'flag' | 'correct' | 'none'
RUN_DECONTX    = False       # optional, needs pydecontx / decontx-scanpy

# doublets - rate measured directly, already corrected for invisible collisions
RUN_SCRUBLET          = False
EXPECTED_DOUBLET_RATE = 0.0716   # collision experiment, includes same-species pairs
SCRUBLET_BATCH_COL    = None     # e.g. 'plate' if collisions are per-plate
DOUBLET_CALL_MODE     = 'rate'   # 'rate' (measured rate) | 'scrublet_auto'
DOUBLET_NGENES_NMADS  = None        # complexity signal; tiebreaker in 'rate' mode

# ----------------------------- QC / outliers ---------------------------------
MIN_GENES          = 200
MIN_CELLS_PER_GENE = 0
MAX_MITO_PCT       = 20.0
QC_NMADS           = 3       # two-sided MAD cut on log depth
MAX_GENES          = None    # or a hard cap, e.g. 6000
AMBIENT_ACTION     = 'none'
RUN_SCRUBLET       = False
DOUBLET_NGENES_NMADS = None

OUTLIER_METHOD = 'iqr'   # 'iqr' | 'mad' | 'none'
IQR_K          = 1.5     # Tukey's fence
OUTLIER_ON_LOG = True    # apply the fence to log1p(counts), not raw counts

# ----------------------------- integration -----------------------------------
DIMENSIONS   = [20]
BEST_DIM     = 20
UMAP_COLOR_BY = 'fixed_leiden'   # 'fixed_leiden' | 'per_dim_leiden'
N_TOP_GENES             = 2000
NEIGHBORS_WITHIN_BATCH  = 1
PYNNDESCENT_N_NEIGHBORS = 3
ANNOT_IGNORE_UNKNOWN    = False  # True: ignore query cells in the majority vote
LEIDEN_RESOLUTION = 1.0    # held fixed across all dimensions


# ----------------------------- labels ----------------------------------------
LABEL_COL   = 'cell_type_broad'
LABEL_ORDER = ['Stem', 'TA', 'Enterocyte', 'Paneth', 'Goblet', 'Tuft', 'EEC']
NAME_MAP    = {'Enteroendocrine': 'EEC', 'Endocrine': 'EEC'}
# leiden_annot_<dim> stores "<cluster>_<celltype>"; the parsed "_type" column
# holds the bare cell type and is what comparisons use.
INTEG_COL   = f'leiden_annot_{BEST_DIM}_type'

CT_COLORS = {'Stem': '#9370DB', 'TA': '#D2B48C', 'Enterocyte': '#FF8C00',
             'Paneth': '#DC143C', 'Goblet': '#2E8B57', 'Tuft': '#FF1493',
             'EEC': '#4169E1', 'unassigned': '#BBBBBB',
             'not_in_reference': '#444444'}

# ----------------------------- marker sets -----------------------------------
LITERATURE_CSV = '/media/erlantz/71DE167A75E99BB4/Pool32/Combi/Matrix/gene_scoring/marker_gene_sets/literature_top_markers.csv'   # two columns: cell_type, gene. Overrides the dict below.
LITERATURE_MARKERS = {
    'Stem':       ['Lgr5', 'Ascl2', 'Olfm4', 'Axin2', 'Slc12a2', 'Rgmb', 'Smoc2'],
    'TA':         ['Mki67', 'Top2a', 'Ccnb1', 'Birc5', 'Ube2c', 'Pcna', 'Mcm5'],
    'Enterocyte': ['Alpi', 'Apoa1', 'Apoa4', 'Fabp1', 'Fabp2', 'Krt20', 'Lct'],
    'Paneth':     ['Lyz1', 'Defa17', 'Defa22', 'Defa24', 'Ang4', 'Mmp7', 'Itln1'],
    'Goblet':     ['Muc2', 'Tff3', 'Agr2', 'Clca1', 'Fcgbp', 'Zg16', 'Spdef'],
    'Tuft':       ['Dclk1', 'Trpm5', 'Gfi1b', 'Il25', 'Tas1r3', 'Avil', 'Sh2d6'],
    'EEC':        ['Chga', 'Chgb', 'Tac1', 'Neurod1', 'Neurog3', 'Cck', 'Sct'],
}

DE_METHOD      = 't-test'
DE_LOGFC_MIN   = 1.0
DE_PADJ_MAX    = 1e-4
DE_UNIQUE_ONLY = True    # keep genes significant for exactly one cell type
DE_TOP_N       = 50      # per cell type, after filtering

# ----------------------------- classifier ------------------------------------
SCALE_MODE       = 'zscore'  # 'zscore' (within dataset) or 'none'
TARGET_PRECISION = 0.95
MAHAL_ALPHA      = 0.01
N_SPLITS         = 5
SEED             = 0

np.random.seed(SEED)

# ----------------------------- logreg markers (original recipe) --------------
DE_ON_FULL_REFERENCE = True   # True reproduces the original (all reference
                              # genes); False restricts to genes shared with DCM
LOGREG_SCORE_MIN  = 0.5
LOGREG_PADJ_MAX   = 0.05
LOGREG_LOGFC_MIN  = 1.0
LOGREG_PCT_MIN    = 0.2       # gene expressed in >=50% of the cell type
WRITE_LOGREG_GENESETS = False # True overwrites final_gene_sets.csv for step5

# ----------------------------- GMM on module scores --------------------------
GMM_MODE        = 'unsupervised'  # 'reference_anchored' | 'unsupervised'
GMM_COVARIANCE  = 'full'                # 'full' | 'tied' | 'diag'
GMM_REG_COVAR   = 1e-4
GMM_N_INIT      = 20
GMM_MAX_ITER    = 500
BAYES_FDR       = 0.05                  # target Bayesian FDR
GMM_OUTLIER_PCT = 1.0                   # reference log-density percentile for
                                        # calling a query cell not_in_reference

# ----------------------------- literature gene sets --------------------------
LITERATURE_GENESET_CSV = ('/media/erlantz/71DE167A75E99BB4/Pool32/Combi/Matrix/'
                          'gene_scoring/marker_gene_sets/literature_top_markers.csv')
LIT_SOURCE_FILTER = None    # None = all; or 'literature' / 'top_markers'
LIT_REQUIRE_SPECIFIC = True # drop rows flagged non-specific, if the column exists
LIT_MAX_PER_TYPE = None     # None = keep all; or an int cap per cell type

GENE_THRESHOLDS = [200, 500, 750, 1000, 2000]
# =============================================================================
# HELPERS
# =============================================================================
def banner(text):
    print('\n' + '=' * 72)
    print(text)
    print('=' * 72)


def save_fig(fig, name):
    """Save at 600 dpi as both PDF and PNG, then close."""
    stem = os.path.splitext(name)[0]
    for ext in ('pdf', 'png'):
        fig.savefig(os.path.join(FIG_DIR, f'{stem}.{ext}'),
                    dpi=600, bbox_inches='tight')
    plt.close(fig)


def mad_outlier(values, nmads, direction='both'):
    """Boolean mask of MAD-based outliers. direction: 'both' | 'upper' | 'lower'."""
    x = np.asarray(values, dtype=float)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    if mad == 0:
        return np.zeros(x.shape, dtype=bool)
    scaled = 1.4826 * mad
    if direction == 'upper':
        return x > med + nmads * scaled
    if direction == 'lower':
        return x < med - nmads * scaled
    return np.abs(x - med) > nmads * scaled

def iqr_outlier(values, k=1.5, direction='both'):
    """Tukey's fence. Outlier if below Q1 - k*IQR or above Q3 + k*IQR.
    direction: 'both' | 'upper' | 'lower'."""
    x = np.asarray(values, dtype=float)
    q1, q3 = np.nanpercentile(x, [25, 75])
    iqr = q3 - q1
    if iqr == 0:
        return np.zeros(x.shape, dtype=bool)
    lo, hi = q1 - k * iqr, q3 + k * iqr
    if direction == 'upper':
        return x > hi
    if direction == 'lower':
        return x < lo
    return (x < lo) | (x > hi)
    
def sanitize_columns(ad, label='adata'):
    """h5py treats '/' as a path separator, so obs/var column names cannot
    contain it. Rename in place and report what changed."""
    for attr in ('obs', 'var'):
        df = getattr(ad, attr)
        bad = [c for c in df.columns if '/' in str(c)]
        if not bad:
            continue
        mapping = {}
        for c in bad:
            new = str(c).replace('/', '_')
            while new in df.columns or new in mapping.values():
                new += '_'
            mapping[c] = new
        setattr(ad, attr, df.rename(columns=mapping))
        for old, new in mapping.items():
            print(f"  Renamed {label}.{attr} column '{old}' -> '{new}' "
                  f"(illegal '/' for HDF5)")
    return ad
def coerce_obs_strings(ad, label='adata'):
    """Make object-dtype obs columns uniformly string. Filling missing columns
    with 'Unknown' creates mixed str/bool/float columns that h5py cannot write."""
    fixed = []
    for col in ad.obs.columns:
        s = ad.obs[col]
        if s.dtype == object:
            types = {type(v).__name__ for v in s.dropna().unique()[:1000]}
            if types - {'str'}:
                fixed.append(f'{col} ({"/".join(sorted(types))})')
            ad.obs[col] = s.astype(str).replace({'nan': 'Unknown',
                                                 'None': 'Unknown',
                                                 '<NA>': 'Unknown'})
    if fixed:
        print(f'  Coerced mixed-type {label}.obs columns to string: '
              f'{", ".join(fixed)}')
    return ad

def require_input(path, produced_by):
    """Fail early and clearly if a previous step has not been run."""
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'Missing input:\n    {path}\n'
            f'Run {produced_by} first.'
        )
    return path


def load_step_input(path, produced_by):
    require_input(path, produced_by)
    ad = sc.read_h5ad(path)
    print(f'  Loaded {os.path.basename(path)}: '
          f'{ad.n_obs} cells x {ad.n_vars} genes')
    return ad


def scale_block(df, mode=None):
    """Within-dataset scaling of a cells x classes score matrix."""
    mode = SCALE_MODE if mode is None else mode
    if mode == 'none':
        return df.copy()
    if mode == 'zscore':
        return ((df - df.mean(axis=0)) /
                df.std(axis=0).replace(0, np.nan)).fillna(0.0)
    raise ValueError("SCALE_MODE must be 'zscore' or 'none'")


def load_gene_sets():
    """Read the gene sets written by step4 and rebuild the CLASSES order."""
    require_input(CSV_GENESETS, 'step4_markers.py')
    df = pd.read_csv(CSV_GENESETS)
    gene_sets = {ct: g['gene'].tolist() for ct, g in df.groupby('cell_type')}
    classes = [ct for ct in LABEL_ORDER if ct in gene_sets]
    return gene_sets, classes

In [ ]:
"""
=============================================================================
STEP 1 - sciMeDseq (DCM) loading, QC, ambient background, doublets
=============================================================================

  in : DCM_MATRIX (tsv), DCM_META (csv)
  out: 01_dcm_clean.h5ad
       figures/stage1_qc_knee_complexity.pdf
       figures/stage2_ambient_background.pdf
       figures/stage3_doublets.pdf
       tables/background_top_genes.csv
       tables/dcm_qc_per_barcode.csv

Rerun when any QC / ambient / doublet parameter changes.
=============================================================================
"""
import os
import sys

try:
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__))
                    if '__file__' in globals() else os.getcwd())
    from pipeline_config import *  # noqa: F401,F403
except ModuleNotFoundError:
    if 'H5_DCM_CLEAN' not in globals():
        raise RuntimeError(
            'Config not loaded. Either run the pipeline_config cell at the top '
            'of this notebook, or save pipeline_config.py somewhere on sys.path.'
        ) from None
    print('  Using config from the notebook session.')

# %%
# =============================================================================
# 1.1 - LOAD MATRIX AND METADATA
# =============================================================================
banner('STEP 1.1 - LOADING sciMeDseq (DCM) DATA')

counts = pd.read_csv(DCM_MATRIX, index_col=0, sep='\t')
if 'region_ID' in counts.columns:
    counts = counts.drop(columns='region_ID')
print(f'  Matrix: {counts.shape[0]} genes x {counts.shape[1]} barcodes')

meta = pd.read_csv(DCM_META, index_col=0, sep=',')
print(f'  Metadata: {meta.shape[0]} rows x {meta.shape[1]} columns')

# align metadata to matrix columns explicitly; passing obs=meta directly would
# silently assume identical row order
bar_matrix = pd.Index(counts.columns.astype(str))
meta.index = meta.index.astype(str)

shared = bar_matrix.intersection(meta.index)
print(f'  Barcodes in both matrix and metadata: {len(shared)} / {len(bar_matrix)}')
if len(shared) == 0:
    raise ValueError(
        'No barcode overlap between matrix columns and metadata index. '
        'Check the index column of the stats CSV and the matrix header.'
    )
if len(shared) < len(bar_matrix):
    print(f'  WARNING: {len(bar_matrix) - len(shared)} barcodes have no metadata '
          f'and are dropped')

counts = counts.loc[:, shared]
meta = meta.reindex(shared)
assert (counts.columns == meta.index).all(), 'barcode alignment failed'

adata_dcm = sc.AnnData(
    X=sp.csr_matrix(counts.T.values.astype(np.float32)),
    obs=meta.copy(),
    var=pd.DataFrame(index=counts.index.astype(str)),
)
adata_dcm.var_names_make_unique()
adata_dcm.obs['orig.ident'] = 'sciMeDseq'
adata_dcm.obs['Dox'] = '48h'
adata_dcm.layers['raw_counts'] = adata_dcm.X.copy()
adata_dcm = sanitize_columns(adata_dcm, 'adata_dcm')
print(f'  AnnData: {adata_dcm.n_obs} barcodes x {adata_dcm.n_vars} genes')

# --- QC metrics (mito prefix checked in one consistent case) -----------------
adata_dcm.var['mt'] = adata_dcm.var_names.str.lower().str.startswith('mt-')
n_mt = int(adata_dcm.var['mt'].sum())
print(f'  Mitochondrial genes detected: {n_mt}')
if n_mt == 0:
    print('  NOTE: no mt- genes found (Ensembl IDs?). Mito filter disabled.')

sc.pp.calculate_qc_metrics(
    adata_dcm, qc_vars=['mt'] if n_mt > 0 else [],
    percent_top=[20], log1p=False, inplace=True,
)

print('\n  Pre-filter QC:')
print(adata_dcm.obs[['total_counts', 'n_genes_by_counts']].describe().to_string())

rank_order = np.argsort(-adata_dcm.obs['total_counts'].values)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(np.arange(1, adata_dcm.n_obs + 1),
             adata_dcm.obs['total_counts'].values[rank_order], lw=1)
axes[0].set_xscale('log'); axes[0].set_yscale('log')
axes[0].set_xlabel('Barcode rank'); axes[0].set_ylabel('Total counts')
axes[0].set_title('Knee plot', fontweight='bold')

axes[1].hist(adata_dcm.obs['n_genes_by_counts'], bins=80, color='#7B2D8E', alpha=0.8)
axes[1].axvline(MIN_GENES, color='k', ls='--', lw=1.5, label=f'min_genes={MIN_GENES}')
axes[1].set_xlabel('Genes detected'); axes[1].set_ylabel('Barcodes')
axes[1].set_title('Complexity', fontweight='bold'); axes[1].legend(fontsize=8)
for ax in axes:
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
fig.tight_layout()
save_fig(fig, 'stage1_qc_knee_complexity.pdf')


# %%
# =============================================================================
# 1.2 - AMBIENT BACKGROUND FROM SUB-THRESHOLD BARCODES
# =============================================================================
banner('STEP 1.2 - AMBIENT BACKGROUND')

n_genes_vec = adata_dcm.obs['n_genes_by_counts'].values
tot_vec = adata_dcm.obs['total_counts'].values

bg_mask = (n_genes_vec < BG_MAX_GENES) & (tot_vec >= BG_MIN_COUNTS)
cell_mask = n_genes_vec >= MIN_GENES

print(f'  Background barcodes (<{BG_MAX_GENES} genes, >={BG_MIN_COUNTS} counts): '
      f'{bg_mask.sum()}')
print(f'  Candidate cells (>={MIN_GENES} genes): {cell_mask.sum()}')

if bg_mask.sum() < 50:
    print('  WARNING: too few background barcodes for a stable profile.')
    print('  If the matrix was pre-filtered upstream, regenerate it without the')
    print('  count cut - the discarded barcodes are the only empty-droplet')
    print('  analogue available in plate data. Ambient scoring skipped.')
    HAVE_BACKGROUND = False
else:
    HAVE_BACKGROUND = True

    raw = adata_dcm.layers['raw_counts']
    bg_counts = np.asarray(raw[bg_mask].sum(axis=0)).ravel()
    bg_profile = bg_counts / bg_counts.sum()
    adata_dcm.var['background_freq'] = bg_profile

    cell_raw = raw[cell_mask]
    cell_tot = np.asarray(cell_raw.sum(axis=1)).ravel()
    cell_tot[cell_tot == 0] = 1.0
    cell_freq = np.asarray(cell_raw.todense()) / cell_tot[:, None]

    # 1) correlation to the background profile
    keep_g = bg_profile > 0
    bg_log = np.log1p(bg_profile[keep_g] * 1e4)
    cf_log = np.log1p(cell_freq[:, keep_g] * 1e4)
    bg_c = bg_log - bg_log.mean()
    cf_c = cf_log - cf_log.mean(axis=1, keepdims=True)
    denom = np.linalg.norm(cf_c, axis=1) * np.linalg.norm(bg_c)
    denom[denom == 0] = np.nan
    bg_corr = (cf_c @ bg_c) / denom

    # 2) fraction of a cell's counts in the top background genes
    top_bg = np.argsort(-bg_profile)[:100]
    frac_top_bg = cell_freq[:, top_bg].sum(axis=1)

    adata_dcm.obs['bg_corr'] = np.nan
    adata_dcm.obs['frac_top_bg'] = np.nan
    adata_dcm.obs.loc[adata_dcm.obs_names[cell_mask], 'bg_corr'] = bg_corr
    adata_dcm.obs.loc[adata_dcm.obs_names[cell_mask], 'frac_top_bg'] = frac_top_bg

    amb_flag = (mad_outlier(bg_corr, AMBIENT_NMADS, 'upper') |
                mad_outlier(frac_top_bg, AMBIENT_NMADS, 'upper'))
    adata_dcm.obs['ambient_flag'] = False
    adata_dcm.obs.loc[adata_dcm.obs_names[cell_mask], 'ambient_flag'] = amb_flag

    print(f'  Background correlation: median {np.nanmedian(bg_corr):.3f}, '
          f'IQR {np.nanpercentile(bg_corr, 25):.3f}-'
          f'{np.nanpercentile(bg_corr, 75):.3f}')
    print(f'  Counts in top-100 background genes: '
          f'median {np.nanmedian(frac_top_bg):.3f}')
    print(f'  Flagged as background-dominated: {amb_flag.sum()} '
          f'({100 * amb_flag.mean():.1f}% of candidate cells)')

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].hist(bg_corr, bins=60, color='#8B4513', alpha=0.8)
    axes[0].set_xlabel('Correlation to background profile')
    axes[0].set_ylabel('Cells')
    axes[0].set_title('Background similarity', fontweight='bold')
    axes[1].hist(frac_top_bg, bins=60, color='#FF8C00', alpha=0.8)
    axes[1].set_xlabel('Fraction of counts in top-100 background genes')
    axes[1].set_ylabel('Cells')
    axes[1].set_title('Background load', fontweight='bold')
    axes[2].scatter(np.log10(cell_tot), bg_corr, s=6, alpha=0.4,
                    c=np.where(amb_flag, '#DC143C', '#4169E1'))
    axes[2].set_xlabel('log10 total counts')
    axes[2].set_ylabel('Correlation to background')
    axes[2].set_title('Flagged cells in red', fontweight='bold')
    for ax in axes:
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    fig.tight_layout()
    save_fig(fig, 'stage2_ambient_background.pdf')

    pd.DataFrame({
        'gene': adata_dcm.var_names[top_bg],
        'background_freq': bg_profile[top_bg],
    }).to_csv(os.path.join(TAB_DIR, 'background_top_genes.csv'), index=False)
    print('  Top background genes written to background_top_genes.csv')
    print(f'  First 10: {list(adata_dcm.var_names[top_bg][:10])}')

if RUN_DECONTX:
    print('\n  Running decontX (optional)...')
    try:
        import pydecontx as dx  # decontx-scanpy / py-decontx
        ad_tmp = adata_dcm[cell_mask].copy()
        ad_tmp.X = ad_tmp.layers['raw_counts'].copy()
        sc.pp.normalize_total(ad_tmp, target_sum=1e4)
        sc.pp.log1p(ad_tmp)
        sc.pp.pca(ad_tmp, n_comps=min(30, ad_tmp.n_obs - 1))
        sc.pp.neighbors(ad_tmp)
        sc.tl.leiden(ad_tmp, key_added='leiden_decontx', resolution=1.0)
        ad_tmp.X = ad_tmp.layers['raw_counts'].copy()
        dx.decontx(ad_tmp, cluster_key='leiden_decontx')
        theta = np.asarray(ad_tmp.obs['decontX_contamination'], dtype=float)
        adata_dcm.obs['decontX_contamination'] = np.nan
        adata_dcm.obs.loc[adata_dcm.obs_names[cell_mask],
                          'decontX_contamination'] = theta
        print(f'    decontX contamination: median {np.median(theta):.3f}')
        print('    NOTE: decontX assumes multinomial transcription counts. For DCM '
              'deposition counts treat theta as a diagnostic, not ground truth.')
    except Exception as exc:  # noqa: BLE001
        print(f'    decontX unavailable or failed ({exc}); skipping.')
        print('    Install with: pip install decontx-scanpy')


# %%
# =============================================================================
# 1.3 - DOUBLETS (calibrated to the measured collision rate)
# =============================================================================
banner('STEP 1.3 - DOUBLETS')

print(f'  Expected doublet rate: {EXPECTED_DOUBLET_RATE:.4f} '
      f'(measured collision rate, invisible collisions included)')

adata_dcm.obs['doublet_score'] = np.nan
adata_dcm.obs['predicted_doublet'] = False
adata_dcm.obs['doublet_flag'] = False

n_cells = int(cell_mask.sum())
n_expected = int(round(EXPECTED_DOUBLET_RATE * n_cells))
print(f'  Cells: {n_cells}  ->  expected doublets: {n_expected}')

high_complexity = np.zeros(adata_dcm.n_obs, dtype=bool)
if DOUBLET_NGENES_NMADS is not None:
    high_complexity[cell_mask] = mad_outlier(
        n_genes_vec[cell_mask], DOUBLET_NGENES_NMADS, 'upper'
    )
    print(f'  High-complexity cells: {int(high_complexity.sum())}')
else:
    print('  Complexity flag disabled (it tracks depth, not doublets, here).')
adata_dcm.obs['high_complexity'] = high_complexity


scores = None
cut = None
if RUN_SCRUBLET and n_cells >= 100:
    ad_dbl = adata_dcm[cell_mask].copy()
    ad_dbl.X = ad_dbl.layers['raw_counts'].copy()
    try:
        scrublet_fn = getattr(sc.pp, 'scrublet', None) or sc.external.pp.scrublet
        scrublet_kw = dict(
            expected_doublet_rate=EXPECTED_DOUBLET_RATE,
            batch_key=SCRUBLET_BATCH_COL,
            random_state=SEED,
        )
        if DOUBLET_CALL_MODE == 'rate':
            # skip the automatic threshold (it needs skimage); rate mode ranks
            # by score and cuts at the measured collision rate instead
            scrublet_kw['threshold'] = 1.0
        scrublet_fn(ad_dbl, **scrublet_kw)
        adata_dcm.obs.loc[ad_dbl.obs_names, 'doublet_score'] = \
            ad_dbl.obs['doublet_score'].values
        adata_dcm.obs.loc[ad_dbl.obs_names, 'predicted_doublet'] = \
            ad_dbl.obs['predicted_doublet'].values
        scores = adata_dcm.obs['doublet_score'].values
        n_auto = int(ad_dbl.obs['predicted_doublet'].sum())
        print(f'  Scrublet automatic threshold: {n_auto} doublets '
              f'({100 * n_auto / n_cells:.1f}%)')
        if n_auto == 0 and DOUBLET_CALL_MODE != 'rate':
            print('  NOTE: automatic threshold found none - the score distribution '
                  'is not bimodal at this depth. Rate mode is the safer call.')
        elif n_auto > 0 and abs(n_auto - n_expected) > 0.5 * max(n_expected, 1):
            print(f'  NOTE: automatic call ({n_auto}) diverges from the measured '
                  f'rate ({n_expected}). Inspect stage3_doublets.pdf before '
                  f'trusting either.')
    except Exception as exc:  # noqa: BLE001
        print(f'  Scrublet failed ({exc}); falling back to complexity only.')
elif RUN_SCRUBLET:
    print(f'  Only {n_cells} cells - too few for a stable Scrublet null.')

if DOUBLET_CALL_MODE == 'rate' and scores is not None and n_expected > 0:
    idx = np.where(cell_mask)[0]
    s = scores[idx]
    order_d = np.lexsort((~high_complexity[idx], -s))  # score first, complexity ties
    chosen = idx[order_d[:n_expected]]
    flag = np.zeros(adata_dcm.n_obs, dtype=bool)
    flag[chosen] = True
    adata_dcm.obs['doublet_flag'] = flag
    cut = float(s[order_d[n_expected - 1]])
    print(f'  Rate mode: removing the top {n_expected} cells by doublet score '
          f'(score cutoff {cut:.4f})')
    print(f'  Of those, {int(high_complexity[chosen].sum())} also carry the '
          f'high-complexity flag')
elif scores is not None:
    adata_dcm.obs['doublet_flag'] = (
        adata_dcm.obs['predicted_doublet'].astype(bool) | high_complexity
    )
    print(f'  Scrublet-auto mode: {int(adata_dcm.obs["doublet_flag"].sum())} cells')
else:
    adata_dcm.obs['doublet_flag'] = high_complexity
    print(f'  Complexity-only mode: {int(high_complexity.sum())} cells')

print(f'  Final doublet flag: {int(adata_dcm.obs["doublet_flag"].sum())} cells')

if adata_dcm.obs['doublet_score'].notna().any():
    sub = adata_dcm.obs.loc[cell_mask]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].hist(sub['doublet_score'].dropna(), bins=60, color='#003366', alpha=0.8)
    if cut is not None:
        axes[0].axvline(cut, color='#DC143C', ls='--', lw=1.5,
                        label=f'rate cutoff ({n_expected} cells)')
        axes[0].legend(fontsize=8)
    axes[0].set_xlabel('Scrublet doublet score'); axes[0].set_ylabel('Cells')
    axes[0].set_title('Doublet scores', fontweight='bold')

    axes[1].scatter(sub['n_genes_by_counts'], sub['doublet_score'], s=6, alpha=0.4,
                    c=np.where(sub['doublet_flag'], '#DC143C', '#4169E1'))
    axes[1].set_xlabel('Genes detected'); axes[1].set_ylabel('Doublet score')
    axes[1].set_title('Removed cells in red', fontweight='bold')

    sorted_scores = np.sort(sub['doublet_score'].dropna().values)[::-1]
    axes[2].plot(np.arange(1, len(sorted_scores) + 1), sorted_scores, lw=1.2)
    axes[2].axvline(n_expected, color='#DC143C', ls='--', lw=1.5,
                    label=f'measured rate = {n_expected}')
    axes[2].set_xlabel('Cell rank'); axes[2].set_ylabel('Doublet score')
    axes[2].set_title('Score rank vs collision rate', fontweight='bold')
    axes[2].legend(fontsize=8)

    for ax in axes:
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    fig.tight_layout()
    save_fig(fig, 'stage3_doublets.pdf')


# %%
# =============================================================================
# 1.4 - FILTER AND NORMALISE
# =============================================================================
banner('STEP 1.4 - FILTERING AND NORMALISATION')

# --- diagnostic: what each fence would remove --------------------------------
sub_c = np.log1p(tot_vec[cell_mask])
sub_g = np.log1p(n_genes_vec[cell_mask])
print(f'  Candidate cells: {int(cell_mask.sum())}')
for name, v in [('total_counts', sub_c), ('n_genes', sub_g)]:
    q1, q3 = np.nanpercentile(v, [25, 75])
    iqr = q3 - q1
    for k in [1.5, 3.0]:
        n_lo = int((v < q1 - k * iqr).sum())
        n_hi = int((v > q3 + k * iqr).sum())
        print(f'    {name:<13} k={k}: {n_lo} low, {n_hi} high, '
              f'{n_lo + n_hi} total')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, v, name in [(axes[0], sub_c, 'log1p total_counts'),
                    (axes[1], sub_g, 'log1p n_genes')]:
    ax.boxplot(v, vert=False, whis=IQR_K, widths=0.5,
               flierprops=dict(marker='o', markersize=3, alpha=0.5,
                               markerfacecolor='#DC143C', markeredgecolor='none'))
    q1, q3 = np.nanpercentile(v, [25, 75])
    iqr = q3 - q1
    ax.axvline(q1 - IQR_K * iqr, color='#DC143C', ls='--', lw=1.2)
    ax.axvline(q3 + IQR_K * iqr, color='#DC143C', ls='--', lw=1.2)
    ax.set_xlabel(name); ax.set_yticks([])
    ax.set_title(f'Tukey fence, k={IQR_K}', fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
fig.tight_layout()
save_fig(fig, 'stage1_iqr_outliers.pdf')

# --- the actual filter chain -------------------------------------------------
keep = cell_mask.copy()
print(f'\n  Start (>= {MIN_GENES} genes):            {int(keep.sum())}')

keep &= ~adata_dcm.obs['doublet_flag'].values
print(f'  After doublet removal:                 {int(keep.sum())}')

if MAX_MITO_PCT is not None and n_mt > 0:
    keep &= (adata_dcm.obs['pct_counts_mt'].values <= MAX_MITO_PCT)
    print(f'  After mito <= {MAX_MITO_PCT}%:                  {int(keep.sum())}')

depth_counts = np.log1p(tot_vec) if OUTLIER_ON_LOG else tot_vec
depth_genes = np.log1p(n_genes_vec) if OUTLIER_ON_LOG else n_genes_vec

is_out = np.zeros(adata_dcm.n_obs, dtype=bool)
if OUTLIER_METHOD == 'iqr':
    out_counts = iqr_outlier(depth_counts[cell_mask], IQR_K, 'both')
    out_genes = iqr_outlier(depth_genes[cell_mask], IQR_K, 'both')
    is_out[cell_mask] = out_counts | out_genes
    q1, q3 = np.nanpercentile(depth_counts[cell_mask], [25, 75])
    lo, hi = q1 - IQR_K * (q3 - q1), q3 + IQR_K * (q3 - q1)
    bounds = ((np.expm1(lo), np.expm1(hi)) if OUTLIER_ON_LOG else (lo, hi))
    print(f'  Tukey fence on total_counts (k={IQR_K}): '
          f'[{bounds[0]:.0f}, {bounds[1]:.0f}]')
    print(f'    flagged by counts: {int(out_counts.sum())}, '
          f'by genes: {int(out_genes.sum())}, union: {int(is_out.sum())}')
elif OUTLIER_METHOD == 'mad':
    is_out[cell_mask] = mad_outlier(depth_counts[cell_mask], QC_NMADS, 'both')
    print(f'  MAD outliers (k={QC_NMADS}): {int(is_out.sum())}')
else:
    print('  Outlier removal disabled.')

adata_dcm.obs['depth_outlier'] = is_out
n_new = int((is_out & keep).sum())
keep &= ~is_out
print(f'  After outlier removal:                 {int(keep.sum())} '
      f'({n_new} newly removed)')

if AMBIENT_ACTION == 'flag' and HAVE_BACKGROUND:
    keep &= ~adata_dcm.obs['ambient_flag'].values
    print(f'  After ambient flag removal:            {int(keep.sum())}')

qc_table = adata_dcm.obs.copy()
qc_table['passed_qc'] = keep
qc_table.to_csv(os.path.join(TAB_DIR, 'dcm_qc_per_barcode.csv'))

adata_dcm = adata_dcm[keep].copy()
if MIN_CELLS_PER_GENE and MIN_CELLS_PER_GENE > 0:
    n_before = adata_dcm.n_vars
    sc.pp.filter_genes(adata_dcm, min_cells=MIN_CELLS_PER_GENE)
    print(f'  Gene filter (min_cells={MIN_CELLS_PER_GENE}): '
          f'{n_before} -> {adata_dcm.n_vars}')
else:
    print(f'  Gene filter disabled; keeping all {adata_dcm.n_vars} genes')
print(f'\n  Final: {adata_dcm.n_obs} cells x {adata_dcm.n_vars} genes')

# normalise (defensive copies at every assignment)
adata_dcm.X = adata_dcm.layers['raw_counts'].copy()
sc.pp.normalize_total(adata_dcm, target_sum=1e4)
sc.pp.log1p(adata_dcm)
adata_dcm.layers['lognorm'] = adata_dcm.X.copy()

sc.pp.scale(adata_dcm)
adata_dcm.layers['lognorm_scaled'] = adata_dcm.X.copy()
adata_dcm.X = adata_dcm.layers['lognorm'].copy()   # leave X as lognorm

adata_dcm.write_h5ad(H5_DCM_CLEAN)
print(f'\n  Saved {os.path.basename(H5_DCM_CLEAN)}')
print('  Next: step2_combine.py')

In [ ]:
# %%
# --- how many cells would each MAD threshold remove? -------------------------
lt = np.log1p(tot_vec[cell_mask])
lg = np.log1p(n_genes_vec[cell_mask])
print(f'  Candidate cells: {cell_mask.sum()}')
for k in [2, 2.5, 3, 4, 5]:
    a = mad_outlier(lt, k, 'both')
    b = mad_outlier(lg, k, 'both')
    print(f'    {k:>3} MADs: {int(a.sum()):>3} by counts, '
          f'{int(b.sum()):>3} by genes, {int((a | b).sum()):>3} by either')

In [ ]:
from scipy.stats import spearmanr
qc = pd.read_csv(os.path.join(TAB_DIR, 'dcm_qc_per_barcode.csv'), index_col=0)
q = qc[qc['bg_corr'].notna()]
for v in ['total_counts', 'n_genes_by_counts']:
    r, p = spearmanr(q['bg_corr'], q[v])
    print(f'bg_corr vs {v:<18}: rho = {r:+.3f} (p = {p:.2g})')

In [ ]:
#No ambient

In [ ]:
"""
=============================================================================
STEP 1 - sciMeDseq (DCM) loading, QC, ambient background, doublets
=============================================================================

  in : DCM_MATRIX (tsv), DCM_META (csv)
  out: 01_dcm_clean.h5ad
       figures/stage1_qc_knee_complexity.pdf
       figures/stage2_ambient_background.pdf
       figures/stage3_doublets.pdf
       tables/background_top_genes.csv
       tables/dcm_qc_per_barcode.csv

Rerun when any QC / ambient / doublet parameter changes.
=============================================================================
"""
import os
import sys

try:
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__))
                    if '__file__' in globals() else os.getcwd())
    from pipeline_config import *  # noqa: F401,F403
except ModuleNotFoundError:
    if 'H5_DCM_CLEAN' not in globals():
        raise RuntimeError(
            'Config not loaded. Either run the pipeline_config cell at the top '
            'of this notebook, or save pipeline_config.py somewhere on sys.path.'
        ) from None
    print('  Using config from the notebook session.')

# %%
# =============================================================================
# 1.1 - LOAD MATRIX AND METADATA
# =============================================================================
banner('STEP 1.1 - LOADING sciMeDseq (DCM) DATA')

counts = pd.read_csv(DCM_MATRIX, index_col=0, sep='\t')
if 'region_ID' in counts.columns:
    counts = counts.drop(columns='region_ID')
print(f'  Matrix: {counts.shape[0]} genes x {counts.shape[1]} barcodes')

meta = pd.read_csv(DCM_META, index_col=0, sep=',')
print(f'  Metadata: {meta.shape[0]} rows x {meta.shape[1]} columns')

# align metadata to matrix columns explicitly; passing obs=meta directly would
# silently assume identical row order
bar_matrix = pd.Index(counts.columns.astype(str))
meta.index = meta.index.astype(str)

shared = bar_matrix.intersection(meta.index)
print(f'  Barcodes in both matrix and metadata: {len(shared)} / {len(bar_matrix)}')
if len(shared) == 0:
    raise ValueError(
        'No barcode overlap between matrix columns and metadata index. '
        'Check the index column of the stats CSV and the matrix header.'
    )
if len(shared) < len(bar_matrix):
    print(f'  WARNING: {len(bar_matrix) - len(shared)} barcodes have no metadata '
          f'and are dropped')

counts = counts.loc[:, shared]
meta = meta.reindex(shared)
assert (counts.columns == meta.index).all(), 'barcode alignment failed'

adata_dcm = sc.AnnData(
    X=sp.csr_matrix(counts.T.values.astype(np.float32)),
    obs=meta.copy(),
    var=pd.DataFrame(index=counts.index.astype(str)),
)
adata_dcm.var_names_make_unique()
adata_dcm.obs['orig.ident'] = 'sciMeDseq'
adata_dcm.obs['Dox'] = '48h'
adata_dcm.layers['raw_counts'] = adata_dcm.X.copy()
adata_dcm = sanitize_columns(adata_dcm, 'adata_dcm')
print(f'  AnnData: {adata_dcm.n_obs} barcodes x {adata_dcm.n_vars} genes')

# --- QC metrics (mito prefix checked in one consistent case) -----------------
adata_dcm.var['mt'] = adata_dcm.var_names.str.lower().str.startswith('mt-')
n_mt = int(adata_dcm.var['mt'].sum())
print(f'  Mitochondrial genes detected: {n_mt}')
if n_mt == 0:
    print('  NOTE: no mt- genes found (Ensembl IDs?). Mito filter disabled.')

sc.pp.calculate_qc_metrics(
    adata_dcm, qc_vars=['mt'] if n_mt > 0 else [],
    percent_top=[20], log1p=False, inplace=True,
)

print('\n  Pre-filter QC:')
print(adata_dcm.obs[['total_counts', 'n_genes_by_counts']].describe().to_string())

rank_order = np.argsort(-adata_dcm.obs['total_counts'].values)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(np.arange(1, adata_dcm.n_obs + 1),
             adata_dcm.obs['total_counts'].values[rank_order], lw=1)
axes[0].set_xscale('log'); axes[0].set_yscale('log')
axes[0].set_xlabel('Barcode rank'); axes[0].set_ylabel('Total counts')
axes[0].set_title('Knee plot', fontweight='bold')

axes[1].hist(adata_dcm.obs['n_genes_by_counts'], bins=80, color='#7B2D8E', alpha=0.8)
axes[1].axvline(MIN_GENES, color='k', ls='--', lw=1.5, label=f'min_genes={MIN_GENES}')
axes[1].set_xlabel('Genes detected'); axes[1].set_ylabel('Barcodes')
axes[1].set_title('Complexity', fontweight='bold'); axes[1].legend(fontsize=8)
for ax in axes:
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
fig.tight_layout()
save_fig(fig, 'stage1_qc_knee_complexity.pdf')


# %%
# =============================================================================
# 1.2 - AMBIENT BACKGROUND FROM SUB-THRESHOLD BARCODES
# =============================================================================
banner('STEP 1.2 - AMBIENT BACKGROUND')

n_genes_vec = adata_dcm.obs['n_genes_by_counts'].values
tot_vec = adata_dcm.obs['total_counts'].values

bg_mask = (n_genes_vec < BG_MAX_GENES) & (tot_vec >= BG_MIN_COUNTS)
cell_mask = n_genes_vec >= MIN_GENES

print(f'  Background barcodes (<{BG_MAX_GENES} genes, >={BG_MIN_COUNTS} counts): '
      f'{bg_mask.sum()}')
print(f'  Candidate cells (>={MIN_GENES} genes): {cell_mask.sum()}')

if bg_mask.sum() < 50:
    print('  WARNING: too few background barcodes for a stable profile.')
    print('  If the matrix was pre-filtered upstream, regenerate it without the')
    print('  count cut - the discarded barcodes are the only empty-droplet')
    print('  analogue available in plate data. Ambient scoring skipped.')
    HAVE_BACKGROUND = False
else:
    HAVE_BACKGROUND = True

    raw = adata_dcm.layers['raw_counts']
    bg_counts = np.asarray(raw[bg_mask].sum(axis=0)).ravel()
    bg_profile = bg_counts / bg_counts.sum()
    adata_dcm.var['background_freq'] = bg_profile

    cell_raw = raw[cell_mask]
    cell_tot = np.asarray(cell_raw.sum(axis=1)).ravel()
    cell_tot[cell_tot == 0] = 1.0
    cell_freq = np.asarray(cell_raw.todense()) / cell_tot[:, None]

    # 1) correlation to the background profile
    keep_g = bg_profile > 0
    bg_log = np.log1p(bg_profile[keep_g] * 1e4)
    cf_log = np.log1p(cell_freq[:, keep_g] * 1e4)
    bg_c = bg_log - bg_log.mean()
    cf_c = cf_log - cf_log.mean(axis=1, keepdims=True)
    denom = np.linalg.norm(cf_c, axis=1) * np.linalg.norm(bg_c)
    denom[denom == 0] = np.nan
    bg_corr = (cf_c @ bg_c) / denom

    # 2) fraction of a cell's counts in the top background genes
    top_bg = np.argsort(-bg_profile)[:100]
    frac_top_bg = cell_freq[:, top_bg].sum(axis=1)

    adata_dcm.obs['bg_corr'] = np.nan
    adata_dcm.obs['frac_top_bg'] = np.nan
    adata_dcm.obs.loc[adata_dcm.obs_names[cell_mask], 'bg_corr'] = bg_corr
    adata_dcm.obs.loc[adata_dcm.obs_names[cell_mask], 'frac_top_bg'] = frac_top_bg

    amb_flag = (mad_outlier(bg_corr, AMBIENT_NMADS, 'upper') |
                mad_outlier(frac_top_bg, AMBIENT_NMADS, 'upper'))
    adata_dcm.obs['ambient_flag'] = False
    adata_dcm.obs.loc[adata_dcm.obs_names[cell_mask], 'ambient_flag'] = amb_flag

    print(f'  Background correlation: median {np.nanmedian(bg_corr):.3f}, '
          f'IQR {np.nanpercentile(bg_corr, 25):.3f}-'
          f'{np.nanpercentile(bg_corr, 75):.3f}')
    print(f'  Counts in top-100 background genes: '
          f'median {np.nanmedian(frac_top_bg):.3f}')
    print(f'  Flagged as background-dominated: {amb_flag.sum()} '
          f'({100 * amb_flag.mean():.1f}% of candidate cells)')

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].hist(bg_corr, bins=60, color='#8B4513', alpha=0.8)
    axes[0].set_xlabel('Correlation to background profile')
    axes[0].set_ylabel('Cells')
    axes[0].set_title('Background similarity', fontweight='bold')
    axes[1].hist(frac_top_bg, bins=60, color='#FF8C00', alpha=0.8)
    axes[1].set_xlabel('Fraction of counts in top-100 background genes')
    axes[1].set_ylabel('Cells')
    axes[1].set_title('Background load', fontweight='bold')
    axes[2].scatter(np.log10(cell_tot), bg_corr, s=6, alpha=0.4,
                    c=np.where(amb_flag, '#DC143C', '#4169E1'))
    axes[2].set_xlabel('log10 total counts')
    axes[2].set_ylabel('Correlation to background')
    axes[2].set_title('Flagged cells in red', fontweight='bold')
    for ax in axes:
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    fig.tight_layout()
    save_fig(fig, 'stage2_ambient_background.pdf')

    pd.DataFrame({
        'gene': adata_dcm.var_names[top_bg],
        'background_freq': bg_profile[top_bg],
    }).to_csv(os.path.join(TAB_DIR, 'background_top_genes.csv'), index=False)
    print('  Top background genes written to background_top_genes.csv')
    print(f'  First 10: {list(adata_dcm.var_names[top_bg][:10])}')

if RUN_DECONTX:
    print('\n  Running decontX (optional)...')
    try:
        import pydecontx as dx  # decontx-scanpy / py-decontx
        ad_tmp = adata_dcm[cell_mask].copy()
        ad_tmp.X = ad_tmp.layers['raw_counts'].copy()
        sc.pp.normalize_total(ad_tmp, target_sum=1e4)
        sc.pp.log1p(ad_tmp)
        sc.pp.pca(ad_tmp, n_comps=min(30, ad_tmp.n_obs - 1))
        sc.pp.neighbors(ad_tmp)
        sc.tl.leiden(ad_tmp, key_added='leiden_decontx', resolution=1.0)
        ad_tmp.X = ad_tmp.layers['raw_counts'].copy()
        dx.decontx(ad_tmp, cluster_key='leiden_decontx')
        theta = np.asarray(ad_tmp.obs['decontX_contamination'], dtype=float)
        adata_dcm.obs['decontX_contamination'] = np.nan
        adata_dcm.obs.loc[adata_dcm.obs_names[cell_mask],
                          'decontX_contamination'] = theta
        print(f'    decontX contamination: median {np.median(theta):.3f}')
        print('    NOTE: decontX assumes multinomial transcription counts. For DCM '
              'deposition counts treat theta as a diagnostic, not ground truth.')
    except Exception as exc:  # noqa: BLE001
        print(f'    decontX unavailable or failed ({exc}); skipping.')
        print('    Install with: pip install decontx-scanpy')


# %%
# =============================================================================
# 1.3 - DOUBLETS (calibrated to the measured collision rate)
# =============================================================================
banner('STEP 1.3 - DOUBLETS')

print(f'  Expected doublet rate: {EXPECTED_DOUBLET_RATE:.4f} '
      f'(measured collision rate, invisible collisions included)')

adata_dcm.obs['doublet_score'] = np.nan
adata_dcm.obs['predicted_doublet'] = False
adata_dcm.obs['doublet_flag'] = False

n_cells = int(cell_mask.sum())
n_expected = int(round(EXPECTED_DOUBLET_RATE * n_cells))
print(f'  Cells: {n_cells}  ->  expected doublets: {n_expected}')

high_complexity = np.zeros(adata_dcm.n_obs, dtype=bool)
if DOUBLET_NGENES_NMADS is not None:
    high_complexity[cell_mask] = mad_outlier(
        n_genes_vec[cell_mask], DOUBLET_NGENES_NMADS, 'upper'
    )
    print(f'  High-complexity cells: {int(high_complexity.sum())}')
else:
    print('  Complexity flag disabled (it tracks depth, not doublets, here).')
adata_dcm.obs['high_complexity'] = high_complexity


scores = None
cut = None
if RUN_SCRUBLET and n_cells >= 100:
    ad_dbl = adata_dcm[cell_mask].copy()
    ad_dbl.X = ad_dbl.layers['raw_counts'].copy()
    try:
        scrublet_fn = getattr(sc.pp, 'scrublet', None) or sc.external.pp.scrublet
        scrublet_kw = dict(
            expected_doublet_rate=EXPECTED_DOUBLET_RATE,
            batch_key=SCRUBLET_BATCH_COL,
            random_state=SEED,
        )
        if DOUBLET_CALL_MODE == 'rate':
            # skip the automatic threshold (it needs skimage); rate mode ranks
            # by score and cuts at the measured collision rate instead
            scrublet_kw['threshold'] = 1.0
        scrublet_fn(ad_dbl, **scrublet_kw)
        adata_dcm.obs.loc[ad_dbl.obs_names, 'doublet_score'] = \
            ad_dbl.obs['doublet_score'].values
        adata_dcm.obs.loc[ad_dbl.obs_names, 'predicted_doublet'] = \
            ad_dbl.obs['predicted_doublet'].values
        scores = adata_dcm.obs['doublet_score'].values
        n_auto = int(ad_dbl.obs['predicted_doublet'].sum())
        print(f'  Scrublet automatic threshold: {n_auto} doublets '
              f'({100 * n_auto / n_cells:.1f}%)')
        if n_auto == 0 and DOUBLET_CALL_MODE != 'rate':
            print('  NOTE: automatic threshold found none - the score distribution '
                  'is not bimodal at this depth. Rate mode is the safer call.')
        elif n_auto > 0 and abs(n_auto - n_expected) > 0.5 * max(n_expected, 1):
            print(f'  NOTE: automatic call ({n_auto}) diverges from the measured '
                  f'rate ({n_expected}). Inspect stage3_doublets.pdf before '
                  f'trusting either.')
    except Exception as exc:  # noqa: BLE001
        print(f'  Scrublet failed ({exc}); falling back to complexity only.')
elif RUN_SCRUBLET:
    print(f'  Only {n_cells} cells - too few for a stable Scrublet null.')

if DOUBLET_CALL_MODE == 'rate' and scores is not None and n_expected > 0:
    idx = np.where(cell_mask)[0]
    s = scores[idx]
    order_d = np.lexsort((~high_complexity[idx], -s))  # score first, complexity ties
    chosen = idx[order_d[:n_expected]]
    flag = np.zeros(adata_dcm.n_obs, dtype=bool)
    flag[chosen] = True
    adata_dcm.obs['doublet_flag'] = flag
    cut = float(s[order_d[n_expected - 1]])
    print(f'  Rate mode: removing the top {n_expected} cells by doublet score '
          f'(score cutoff {cut:.4f})')
    print(f'  Of those, {int(high_complexity[chosen].sum())} also carry the '
          f'high-complexity flag')
elif scores is not None:
    adata_dcm.obs['doublet_flag'] = (
        adata_dcm.obs['predicted_doublet'].astype(bool) | high_complexity
    )
    print(f'  Scrublet-auto mode: {int(adata_dcm.obs["doublet_flag"].sum())} cells')
else:
    adata_dcm.obs['doublet_flag'] = high_complexity
    print(f'  Complexity-only mode: {int(high_complexity.sum())} cells')

print(f'  Final doublet flag: {int(adata_dcm.obs["doublet_flag"].sum())} cells')

if adata_dcm.obs['doublet_score'].notna().any():
    sub = adata_dcm.obs.loc[cell_mask]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].hist(sub['doublet_score'].dropna(), bins=60, color='#003366', alpha=0.8)
    if cut is not None:
        axes[0].axvline(cut, color='#DC143C', ls='--', lw=1.5,
                        label=f'rate cutoff ({n_expected} cells)')
        axes[0].legend(fontsize=8)
    axes[0].set_xlabel('Scrublet doublet score'); axes[0].set_ylabel('Cells')
    axes[0].set_title('Doublet scores', fontweight='bold')

    axes[1].scatter(sub['n_genes_by_counts'], sub['doublet_score'], s=6, alpha=0.4,
                    c=np.where(sub['doublet_flag'], '#DC143C', '#4169E1'))
    axes[1].set_xlabel('Genes detected'); axes[1].set_ylabel('Doublet score')
    axes[1].set_title('Removed cells in red', fontweight='bold')

    sorted_scores = np.sort(sub['doublet_score'].dropna().values)[::-1]
    axes[2].plot(np.arange(1, len(sorted_scores) + 1), sorted_scores, lw=1.2)
    axes[2].axvline(n_expected, color='#DC143C', ls='--', lw=1.5,
                    label=f'measured rate = {n_expected}')
    axes[2].set_xlabel('Cell rank'); axes[2].set_ylabel('Doublet score')
    axes[2].set_title('Score rank vs collision rate', fontweight='bold')
    axes[2].legend(fontsize=8)

    for ax in axes:
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    fig.tight_layout()
    save_fig(fig, 'stage3_doublets.pdf')

# %%
# =============================================================================
# 1.4 - FILTER AND NORMALISE
# =============================================================================
banner('STEP 1.4 - FILTERING AND NORMALISATION')

# --- diagnostic: what each fence would remove --------------------------------
sub_c = np.log1p(tot_vec[cell_mask])
sub_g = np.log1p(n_genes_vec[cell_mask])
print(f'  Candidate cells: {int(cell_mask.sum())}')
for name, v in [('total_counts', sub_c), ('n_genes', sub_g)]:
    q1, q3 = np.nanpercentile(v, [25, 75])
    iqr = q3 - q1
    for k in [1.5, 3.0]:
        n_lo = int((v < q1 - k * iqr).sum())
        n_hi = int((v > q3 + k * iqr).sum())
        print(f'    {name:<13} k={k}: {n_lo} low, {n_hi} high, '
              f'{n_lo + n_hi} total')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, v, name in [(axes[0], sub_c, 'log1p total_counts'),
                    (axes[1], sub_g, 'log1p n_genes')]:
    ax.boxplot(v, vert=False, whis=IQR_K, widths=0.5,
               flierprops=dict(marker='o', markersize=3, alpha=0.5,
                               markerfacecolor='#DC143C', markeredgecolor='none'))
    q1, q3 = np.nanpercentile(v, [25, 75])
    iqr = q3 - q1
    ax.axvline(q1 - IQR_K * iqr, color='#DC143C', ls='--', lw=1.2)
    ax.axvline(q3 + IQR_K * iqr, color='#DC143C', ls='--', lw=1.2)
    ax.set_xlabel(name); ax.set_yticks([])
    ax.set_title(f'Tukey fence, k={IQR_K}', fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
fig.tight_layout()
save_fig(fig, 'stage1_iqr_outliers.pdf')

# --- ambient correction, BEFORE any subsetting -------------------------------
if AMBIENT_ACTION == 'correct' and HAVE_BACKGROUND:
    print('\n  Subtracting the background profile from raw counts...')
    raw_d = adata_dcm.layers['raw_counts'].todense()
    tot = np.asarray(raw_d.sum(axis=1)).ravel()
    rho = np.clip(adata_dcm.obs['frac_top_bg'].values /
                  max(np.nanmedian(adata_dcm.obs['frac_top_bg'].values), 1e-9)
                  * 0.1, 0, 0.5)
    rho = np.nan_to_num(rho)
    bgp = adata_dcm.var['background_freq'].values
    bgp = bgp / bgp.sum() if bgp.sum() > 0 else bgp
    adata_dcm.layers['raw_counts'] = sp.csr_matrix(
        np.clip(np.asarray(raw_d) - (rho * tot)[:, None] * bgp[None, :], 0, None))
    print(f'  Mean removed fraction: {rho.mean():.3f}')
elif AMBIENT_ACTION == 'correct':
    print('\n  AMBIENT_ACTION="correct" but no background profile; skipping.')

# --- the filter chain --------------------------------------------------------
keep = cell_mask.copy()
print(f'\n  Start (>= {MIN_GENES} genes):            {int(keep.sum())}')

keep &= ~adata_dcm.obs['doublet_flag'].values
print(f'  After doublet removal:                 {int(keep.sum())}')

if MAX_MITO_PCT is not None and n_mt > 0:
    keep &= (adata_dcm.obs['pct_counts_mt'].values <= MAX_MITO_PCT)
    print(f'  After mito <= {MAX_MITO_PCT}%:                  {int(keep.sum())}')

depth_counts = np.log1p(tot_vec) if OUTLIER_ON_LOG else tot_vec
depth_genes = np.log1p(n_genes_vec) if OUTLIER_ON_LOG else n_genes_vec

is_out = np.zeros(adata_dcm.n_obs, dtype=bool)
if OUTLIER_METHOD == 'iqr':
    out_counts = iqr_outlier(depth_counts[cell_mask], IQR_K, 'both')
    out_genes = iqr_outlier(depth_genes[cell_mask], IQR_K, 'both')
    is_out[cell_mask] = out_counts | out_genes
    q1, q3 = np.nanpercentile(depth_counts[cell_mask], [25, 75])
    lo, hi = q1 - IQR_K * (q3 - q1), q3 + IQR_K * (q3 - q1)
    bounds = (np.expm1(lo), np.expm1(hi)) if OUTLIER_ON_LOG else (lo, hi)
    print(f'  Tukey fence on total_counts (k={IQR_K}): '
          f'[{bounds[0]:.0f}, {bounds[1]:.0f}]')
    print(f'    flagged by counts: {int(out_counts.sum())}, '
          f'by genes: {int(out_genes.sum())}, union: {int(is_out.sum())}')
elif OUTLIER_METHOD == 'mad':
    is_out[cell_mask] = mad_outlier(depth_counts[cell_mask], QC_NMADS, 'both')
    print(f'  MAD outliers (k={QC_NMADS}): {int(is_out.sum())}')
else:
    print('  Outlier removal disabled.')

adata_dcm.obs['depth_outlier'] = is_out
n_new = int((is_out & keep).sum())
keep &= ~is_out
print(f'  After outlier removal:                 {int(keep.sum())} '
      f'({n_new} newly removed)')

if AMBIENT_ACTION == 'flag' and HAVE_BACKGROUND:
    keep &= ~adata_dcm.obs['ambient_flag'].values
    print(f'  After ambient flag removal:            {int(keep.sum())}')

qc_table = adata_dcm.obs.copy()
qc_table['passed_qc'] = keep
qc_table.to_csv(os.path.join(TAB_DIR, 'dcm_qc_per_barcode.csv'))

adata_dcm = adata_dcm[keep].copy()
if MIN_CELLS_PER_GENE and MIN_CELLS_PER_GENE > 0:
    n_before = adata_dcm.n_vars
    sc.pp.filter_genes(adata_dcm, min_cells=MIN_CELLS_PER_GENE)
    print(f'  Gene filter (min_cells={MIN_CELLS_PER_GENE}): '
          f'{n_before} -> {adata_dcm.n_vars}')
else:
    print(f'  Gene filter disabled; keeping all {adata_dcm.n_vars} genes')
print(f'\n  Final: {adata_dcm.n_obs} cells x {adata_dcm.n_vars} genes')

# normalise (defensive copies at every assignment)
adata_dcm.X = adata_dcm.layers['raw_counts'].copy()
sc.pp.normalize_total(adata_dcm, target_sum=1e4)
sc.pp.log1p(adata_dcm)
adata_dcm.layers['lognorm'] = adata_dcm.X.copy()

sc.pp.scale(adata_dcm)
adata_dcm.layers['lognorm_scaled'] = adata_dcm.X.copy()
adata_dcm.X = adata_dcm.layers['lognorm'].copy()

adata_dcm.write_h5ad(H5_DCM_CLEAN)
print(f'\n  Saved {os.path.basename(H5_DCM_CLEAN)}')
print('  Next: step2_combine.py')

In [ ]:
"""
=============================================================================
STEP 2 - Reference loading and combined object
=============================================================================

  in : 01_dcm_clean.h5ad, REF_H5AD
  out: 02_combined.h5ad

The reference is assumed already normalised and is NOT re-normalised. A sanity
check catches the case where the lognorm layer turns out to hold raw counts.

Rerun when the reference file, LABEL_ORDER or NAME_MAP changes.
=============================================================================
"""

import os
import sys

try:
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__))
                    if '__file__' in globals() else os.getcwd())
    from pipeline_config import *  # noqa: F401,F403
except ModuleNotFoundError:
    if 'H5_DCM_CLEAN' not in globals():
        raise RuntimeError(
            'Config not loaded. Either run the pipeline_config cell at the top '
            'of this notebook, or save pipeline_config.py somewhere on sys.path.'
        ) from None
    print('  Using config from the notebook session.')

# %%
banner('STEP 2 - REFERENCE AND COMBINED OBJECT')

adata_dcm = load_step_input(H5_DCM_CLEAN, 'step1_dcm_qc.py')

adata_ref = sc.read_h5ad(REF_H5AD)
adata_ref.var_names_make_unique()
adata_ref = sanitize_columns(adata_ref, 'adata_ref')
print(f'  Reference: {adata_ref.n_obs} cells x {adata_ref.n_vars} genes')
print(f'  Reference layers: {list(adata_ref.layers.keys())}')

if LABEL_COL not in adata_ref.obs.columns:
    raise ValueError(f'{LABEL_COL} not in reference obs: '
                     f'{list(adata_ref.obs.columns)}')

adata_ref.obs[LABEL_COL] = adata_ref.obs[LABEL_COL].astype(str).replace(NAME_MAP)
adata_ref = adata_ref[adata_ref.obs[LABEL_COL].isin(LABEL_ORDER)].copy()
print(f'  Reference after label harmonisation: {adata_ref.n_obs} cells')
print(adata_ref.obs[LABEL_COL].value_counts().reindex(LABEL_ORDER).to_string())

# %%
# --- normalisation: take the reference as it is ------------------------------
if 'lognorm' not in adata_ref.layers:
    adata_ref.layers['lognorm'] = adata_ref.X.copy()
    print('  No lognorm layer; using adata_ref.X as-is (assumed log-normalised).')
else:
    print('  Using existing adata_ref.layers["lognorm"].')

_xr = adata_ref.layers['lognorm']
_sample = (_xr[:500].toarray() if sp.issparse(_xr) else np.asarray(_xr[:500]))
_mx = float(np.nanmax(_sample))
_is_int = bool(np.allclose(_sample, np.round(_sample)))
print(f'  Reference lognorm: max {_mx:.2f}, integer-valued: {_is_int}')
if _is_int and _mx > 50:
    raise ValueError(
        'Reference lognorm layer looks like raw counts. Normalise it first or '
        'point REF_H5AD at the normalised object.'
    )
if _mx < 3:
    print('  WARNING: max value is very low. If this object was scaled or logged '
          'twice, module scores will not be comparable to the DCM side.')

# lognorm_scaled drives the PCA in step3 and must exist in BOTH objects, since
# sc.concat keeps only the intersection of layer keys.
if 'lognorm_scaled' not in adata_ref.layers:
    adata_ref.X = adata_ref.layers['lognorm'].copy()
    sc.pp.scale(adata_ref)
    adata_ref.layers['lognorm_scaled'] = adata_ref.X.copy()
    adata_ref.X = adata_ref.layers['lognorm'].copy()
    print('  Created lognorm_scaled layer for the reference.')
else:
    print('  Using existing adata_ref.layers["lognorm_scaled"].')

if 'orig.ident' not in adata_ref.obs.columns:
    adata_ref.obs['orig.ident'] = 'reference'

# %%
# --- combine -----------------------------------------------------------------
shared_genes = adata_ref.var_names.intersection(adata_dcm.var_names)
print(f'\n  Shared genes: {len(shared_genes)}')
if len(shared_genes) < 500:
    print('  WARNING: very few shared genes. Check that both objects use the same '
          'identifier type (symbols vs Ensembl IDs).')

adata_dcm.obs[LABEL_COL] = 'Unknown'

# Make object columns uniformly string BEFORE filling gaps, so a column that
# is bool in one object and 'Unknown' in the other ends up str in both.
adata_dcm = coerce_obs_strings(adata_dcm, 'adata_dcm')
adata_ref = coerce_obs_strings(adata_ref, 'adata_ref')

all_cols = set(adata_dcm.obs.columns) | set(adata_ref.obs.columns)
for ad in (adata_dcm, adata_ref):
    for col in all_cols:
        if col not in ad.obs.columns:
            ad.obs[col] = 'Unknown'
    ad.X = ad.layers['lognorm'].copy()
    
adata_combined = sc.concat(
    [adata_dcm, adata_ref], join='inner', merge='unique',
    label='dataset', keys=['sciMeDseq', 'reference'], index_unique=None,
)
adata_combined.obs_names_make_unique()
adata_combined.layers['lognorm'] = adata_combined.X.copy()
adata_combined = sanitize_columns(adata_combined, 'adata_combined')
adata_combined = coerce_obs_strings(adata_combined, 'adata_combined')
print(f'  Combined: {adata_combined.n_obs} cells x {adata_combined.n_vars} genes')
print(f'  Combined layers: {list(adata_combined.layers.keys())}')
print(adata_combined.obs['dataset'].value_counts().to_string())

for lyr in ('lognorm', 'lognorm_scaled'):
    if lyr not in adata_combined.layers:
        raise ValueError(
            f"'{lyr}' did not survive the concat - it must be present in BOTH "
            f"input objects. DCM layers: {list(adata_dcm.layers.keys())}, "
            f"reference layers: {list(adata_ref.layers.keys())}"
        )

adata_combined.write_h5ad(H5_COMBINED)
print(f'\n  Saved {os.path.basename(H5_COMBINED)}')
print('  Next: step3_integrate.py')

In [ ]:
"""
=============================================================================
STEP 3 - BBKNN integration across dimensions, with per-dimension diagnostics
=============================================================================

  in : 02_combined.h5ad
  out: 02b_combined_integrated.h5ad
       figures/step3_umap_grid_<colour mode>.pdf/.png
       figures/step3_umap_scimedseq_highlight.pdf/.png
       figures/step3_composition_counts.pdf/.png
       figures/step3_composition_percent.pdf/.png
       tables/step3_composition_by_dim.csv
       per-dimension scanpy PNGs under INTEG_DIR

Runs BBKNN + UMAP + Leiden at every value in DIMENSIONS and keeps all of them:

    obsm['X_umap_dim<d>']        the embedding at that dimension
    obs['leiden_dim<d>']         the clustering at that dimension
    obs['leiden_annot_<d>']      "<cluster>_<celltype>", as in the original code
    obs['leiden_annot_<d>_type'] bare cell type, used by every later comparison

obsm['X_umap'] and obs['leiden'] are set to BEST_DIM at the end, so everything
downstream keeps working unchanged.

UMAP_COLOR_BY
-------------
'fixed_leiden'   colour every panel by the BEST_DIM clustering. Colours mean
                 the same thing in every panel, so you can see how the
                 embedding rearranges a fixed set of groups.
'per_dim_leiden' colour each panel by its own clustering. Matches the original
                 loop, but cluster 3 at dim 5 is unrelated to cluster 3 at
                 dim 50, so panels are not comparable.

Slow: one BBKNN + UMAP + Leiden pass per dimension.
=============================================================================
"""

import os
import sys

try:
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__))
                    if '__file__' in globals() else os.getcwd())
    from pipeline_config import *  # noqa: F401,F403
except ModuleNotFoundError:
    if 'H5_DCM_CLEAN' not in globals():
        raise RuntimeError(
            'Config not loaded. Either run the pipeline_config cell at the top '
            'of this notebook, or save pipeline_config.py somewhere on sys.path.'
        ) from None
    print('  Using config from the notebook session.')

# %%
banner('STEP 3 - INTEGRATION ACROSS DIMENSIONS')

adata_combined = load_step_input(H5_COMBINED, 'step2_combine.py')

sc.settings.figdir = INTEG_DIR
sc.set_figure_params(dpi=300, dpi_save=600, fontsize=6)

if 'orig.ident' not in adata_combined.obs.columns:
    raise ValueError('orig.ident missing in adata_combined.obs; needed for '
                     'batch-aware HVG')
for lyr in ('lognorm', 'lognorm_scaled'):
    if lyr not in adata_combined.layers:
        raise ValueError(f"adata_combined.layers must contain '{lyr}'")
if BEST_DIM not in DIMENSIONS:
    raise ValueError(f'BEST_DIM={BEST_DIM} must be one of DIMENSIONS={DIMENSIONS}')

print(f'  Dimensions to run: {DIMENSIONS}')
print(f'  BEST_DIM: {BEST_DIM}')
print(f'  UMAP colouring: {UMAP_COLOR_BY}')
print(f'  Leiden resolution: {LEIDEN_RESOLUTION} (fixed across dimensions)')
is_q = (adata_combined.obs['dataset'] == 'sciMeDseq').values
is_r = ~is_q
print(f'  sciMeDseq: {int(is_q.sum())}   reference: {int(is_r.sum())}')

# %%
# --- HVG and PCA (computed once, shared by every dimension) ------------------
adata_combined.X = adata_combined.layers['lognorm'].copy()
sc.pp.highly_variable_genes(adata_combined, n_top_genes=N_TOP_GENES,
                            batch_key='orig.ident')
print(f"\n  HVGs selected: {int(adata_combined.var['highly_variable'].sum())}")

adata_combined.X = adata_combined.layers['lognorm_scaled'].copy()
sc.tl.pca(adata_combined)
n_pcs_avail = adata_combined.obsm['X_pca'].shape[1]
print(f'  PCs available: {n_pcs_avail}')
too_big = [d for d in DIMENSIONS if d > n_pcs_avail]
if too_big:
    raise ValueError(f'DIMENSIONS {too_big} exceed the {n_pcs_avail} PCs '
                     f'computed. Increase n_comps in sc.tl.pca or lower them.')


# %%
# =============================================================================
# 3.1 - LOOP OVER DIMENSIONS
# =============================================================================
banner('STEP 3.1 - DIMENSION SWEEP')

for dim in DIMENSIONS:
    print(f'\n--- dim = {dim} ---')

    try:
        sc.external.pp.bbknn(
            adata_combined,
            batch_key='orig.ident',
            n_pcs=dim,
            neighbors_within_batch=NEIGHBORS_WITHIN_BATCH,
            pynndescent_n_neighbors=PYNNDESCENT_N_NEIGHBORS,
        )
    except Exception as exc:  # noqa: BLE001
        print(f'  BBKNN failed: {exc}')
        print('  Falling back to sc.pp.neighbors()')
        sc.pp.neighbors(adata_combined, n_pcs=dim)

    sc.tl.umap(adata_combined)
    sc.tl.leiden(adata_combined, key_added=f'leiden_dim{dim}',
                 resolution=LEIDEN_RESOLUTION, random_state=SEED)

    # keep this dimension's embedding and clustering
    adata_combined.obsm[f'X_umap_dim{dim}'] = adata_combined.obsm['X_umap'].copy()
    leiden_str = adata_combined.obs[f'leiden_dim{dim}'].astype(str)
    print(f'  Clusters: {leiden_str.nunique()}')

    sc.tl.embedding_density(adata_combined, groupby='orig.ident',
                            key_added='batch_dense')
    sc.pl.embedding_density(adata_combined, key='batch_dense',
                            save=f'_{dim}_batch_density.png', show=False)

    # majority reference label per cluster
    annot_col = f'leiden_annot_{dim}'
    cluster_annotations = {}
    for cluster in sorted(leiden_str.unique(),
                          key=lambda x: int(x) if x.isdigit() else x):
        cells = adata_combined.obs.loc[leiden_str == cluster, LABEL_COL].astype(str)
        if ANNOT_IGNORE_UNKNOWN:
            cells = cells[cells != 'Unknown']
        vc = cells.value_counts()
        cluster_annotations[cluster] = vc.idxmax() if len(vc) > 0 else 'Unknown'

    n_unknown = sum(v == 'Unknown' for v in cluster_annotations.values())
    if n_unknown:
        print(f'  NOTE: {n_unknown}/{len(cluster_annotations)} clusters take '
              f'"Unknown" as majority - no transferable label there.')

    adata_combined.obs[annot_col] = (
        leiden_str + '_' + leiden_str.map(cluster_annotations).fillna('Unknown')
    )
    adata_combined.obs[f'{annot_col}_type'] = (
        adata_combined.obs[annot_col].astype(str).str.split('_').str[-1]
    )

    plot_cols = [c for c in ['Dox', 'Sample', LABEL_COL, annot_col,
                             f'leiden_dim{dim}']
                 if c in adata_combined.obs.columns]
    sc.pl.umap(adata_combined, color=plot_cols, frameon=False,
               legend_loc='on data', save=f'_{dim}_umap.png', show=False)

# --- set the canonical embedding and clustering to BEST_DIM ------------------
adata_combined.obsm['X_umap'] = adata_combined.obsm[f'X_umap_dim{BEST_DIM}'].copy()
adata_combined.obs['leiden'] = adata_combined.obs[f'leiden_dim{BEST_DIM}'].copy()
adata_combined.X = adata_combined.layers['lognorm'].copy()
print(f'\n  obsm["X_umap"] and obs["leiden"] set to dim {BEST_DIM}')

if INTEG_COL not in adata_combined.obs.columns:
    raise ValueError(f'{INTEG_COL} was not created - check BEST_DIM={BEST_DIM}')

print(f'\n  {INTEG_COL} on sciMeDseq cells:')
print(adata_combined.obs.loc[is_q, INTEG_COL].value_counts().to_string())


# %%
# =============================================================================
# 3.2 - UMAP GRID ACROSS DIMENSIONS
# =============================================================================
banner('STEP 3.2 - UMAP GRID')

if UMAP_COLOR_BY == 'fixed_leiden':
    colour_series = adata_combined.obs[f'leiden_dim{BEST_DIM}'].astype(str)
    colour_note = f'Leiden from dim {BEST_DIM}, held fixed'
elif UMAP_COLOR_BY == 'per_dim_leiden':
    colour_series = None
    colour_note = "each panel's own Leiden"
else:
    raise ValueError("UMAP_COLOR_BY must be 'fixed_leiden' or 'per_dim_leiden'")
print(f'  Colouring: {colour_note}')

palette = (list(plt.get_cmap('tab20').colors) +
           list(plt.get_cmap('tab20b').colors) +
           list(plt.get_cmap('tab20c').colors))

if colour_series is not None:
    lev = sorted(colour_series.unique(), key=lambda x: int(x) if x.isdigit() else x)
    fixed_colours = {c: palette[i % len(palette)] for i, c in enumerate(lev)}

ncol = 5
nrow = int(np.ceil(len(DIMENSIONS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 4.2 * nrow))
axes = np.atleast_1d(axes).ravel()

for ax, dim in zip(axes, DIMENSIONS):
    U = adata_combined.obsm[f'X_umap_dim{dim}']
    if colour_series is not None:
        cs, cmap_d = colour_series, fixed_colours
    else:
        cs = adata_combined.obs[f'leiden_dim{dim}'].astype(str)
        lv = sorted(cs.unique(), key=lambda x: int(x) if x.isdigit() else x)
        cmap_d = {c: palette[i % len(palette)] for i, c in enumerate(lv)}
    for c, colour in cmap_d.items():
        m = (cs == c).values
        if m.sum():
            ax.scatter(U[m, 0], U[m, 1], s=1.5, alpha=0.7, color=colour,
                       linewidths=0, rasterized=True)
    ax.set_title(f'dim {dim}   ({cs.nunique()} clusters)',
                 fontweight='bold', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)

for ax in axes[len(DIMENSIONS):]:
    ax.axis('off')

fig.suptitle(f'BBKNN embeddings across dimensions - coloured by {colour_note}',
             fontweight='bold', fontsize=12, y=1.005)
fig.tight_layout()
save_fig(fig, f'step3_umap_grid_{UMAP_COLOR_BY}.pdf')


# %%
# =============================================================================
# 3.3 - sciMeDseq HIGHLIGHTED ON EACH EMBEDDING
# =============================================================================
banner('STEP 3.3 - sciMeDseq HIGHLIGHT')

CATS = LABEL_ORDER + ['Unknown']
CAT_COLORS = {**CT_COLORS, 'Unknown': '#999999'}

fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 4.6 * nrow))
axes = np.atleast_1d(axes).ravel()

for ax, dim in zip(axes, DIMENSIONS):
    U = adata_combined.obsm[f'X_umap_dim{dim}']
    qlab = adata_combined.obs[f'leiden_annot_{dim}_type'].astype(str).values

    # reference cells, faint background
    ax.scatter(U[is_r, 0], U[is_r, 1], s=2, color='#CCCCCC', alpha=0.25,
               linewidths=0, rasterized=True, zorder=1)
    # sciMeDseq cells, by transferred cell type
    for cat in CATS:
        m = is_q & (qlab == cat)
        if m.sum():
            ax.scatter(U[m, 0], U[m, 1], s=18, alpha=0.9,
                       color=CAT_COLORS.get(cat, '#777777'),
                       edgecolors='white', linewidths=0.25,
                       rasterized=True, zorder=3)
    ax.set_title(f'dim {dim}', fontweight='bold', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)

for ax in axes[len(DIMENSIONS):]:
    ax.axis('off')

present_cats = [c for c in CATS
                if (adata_combined.obs.loc[is_q, INTEG_COL].astype(str) == c).any()]
handles = ([plt.Line2D([0], [0], marker='o', ls='', markersize=6,
                       color='#CCCCCC', alpha=0.5,
                       label=f'reference (n={int(is_r.sum())})')] +
           [plt.Line2D([0], [0], marker='o', ls='', markersize=7,
                       color=CAT_COLORS.get(c, '#777777'), label=c)
            for c in present_cats])
fig.legend(handles=handles, loc='lower center', ncol=min(9, len(handles)),
           fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.suptitle('sciMeDseq cells by transferred cell type, reference in background',
             fontweight='bold', fontsize=12, y=1.005)
fig.tight_layout()
save_fig(fig, 'step3_umap_scimedseq_highlight.pdf')


# %%
# =============================================================================
# 3.4 - COMPOSITION OF THE sciMeDseq CELLS PER DIMENSION
# =============================================================================
banner('STEP 3.4 - COMPOSITION PER DIMENSION')

comp = pd.DataFrame({
    dim: (adata_combined.obs.loc[is_q, f'leiden_annot_{dim}_type']
          .astype(str).value_counts().reindex(CATS).fillna(0).astype(int))
    for dim in DIMENSIONS
}).T
comp.index.name = 'dim'

print('  Counts:')
print(comp.to_string())
comp.to_csv(os.path.join(TAB_DIR, 'step3_composition_by_dim.csv'))

pct = comp.div(comp.sum(axis=1).replace(0, np.nan), axis=0) * 100
print('\n  Percentages:')
print(pct.round(1).to_string())
pct.round(2).to_csv(os.path.join(TAB_DIR, 'step3_composition_by_dim_percent.csv'))

n_types = (comp > 0).sum(axis=1)
print('\n  Distinct cell types recovered per dimension:')
print(n_types.to_string())
missing_any = [c for c in LABEL_ORDER if (comp[c] == 0).any()]
if missing_any:
    print(f'\n  Types absent at one or more dimensions: {missing_any}')
    print('  A type that never appears cannot be recovered at that dimension,')
    print('  whatever the classifier does downstream.')


def composition_bars(table, as_pct):
    fig, ax = plt.subplots(figsize=(1.15 * len(table) + 4.5, 5.6))
    xpos = np.arange(len(table))
    bottom = np.zeros(len(table))
    for cat in CATS:
        vals = table[cat].values.astype(float)
        ax.bar(xpos, vals, bottom=bottom, width=0.7,
               color=CAT_COLORS.get(cat, '#777777'),
               edgecolor='white', linewidth=0.6, label=cat)
        span = 100.0 if as_pct else table.sum(axis=1).max()
        for i, (v, b) in enumerate(zip(vals, bottom)):
            if v > 0.05 * span:
                txt = f'{v:.0f}%' if as_pct else f'{int(v)}'
                ax.text(xpos[i], b + v / 2, txt, ha='center', va='center',
                        fontsize=7.5, color='white', fontweight='bold')
        bottom += vals
    if not as_pct:
        for i, t in enumerate(table.sum(axis=1).values):
            ax.text(xpos[i], bottom[i], f'n={int(t)}', ha='center', va='bottom',
                    fontsize=8, fontweight='bold')
    ax.set_xticks(xpos)
    ax.set_xticklabels([str(d) for d in table.index])
    ax.set_xlabel('BBKNN dimensions (n_pcs)')
    ax.set_ylabel('Percent of sciMeDseq cells' if as_pct else 'sciMeDseq cells')
    ax.set_title('sciMeDseq composition by integration dimension',
                 fontweight='bold')
    if as_pct:
        ax.set_ylim(0, 105)
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8,
              frameon=False, title='cell type', title_fontsize=9)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    fig.tight_layout()
    save_fig(fig, f'step3_composition_{"percent" if as_pct else "counts"}.pdf')


composition_bars(comp, as_pct=False)
composition_bars(pct.fillna(0), as_pct=True)

# --- cluster count per dimension at fixed resolution --------------------------
n_clusters = pd.Series(
    {dim: adata_combined.obs[f'leiden_dim{dim}'].nunique() for dim in DIMENSIONS},
    name='n_clusters')
n_clusters.index.name = 'dim'

print(f'\n  Clusters per dimension at resolution {LEIDEN_RESOLUTION}:')
print(pd.concat([n_clusters, n_types.rename('n_celltypes_recovered')],
                axis=1).to_string())

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(n_clusters.index, n_clusters.values, 'o-', color='#4169E1',
        label='Leiden clusters')
ax.plot(n_types.index, n_types.values, 's-', color='#DC143C',
        label='cell types recovered')
ax.axvline(BEST_DIM, color='k', ls=':', lw=1.2, label=f'BEST_DIM={BEST_DIM}')
ax.set_xlabel('BBKNN dimensions (n_pcs)')
ax.set_ylabel('Count')
ax.set_title(f'Granularity at fixed resolution {LEIDEN_RESOLUTION}',
             fontweight='bold')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
fig.tight_layout()
save_fig(fig, 'step3_clusters_per_dim.pdf')

# %%
# =============================================================================
# 3.5 - SAVE
# =============================================================================
banner('STEP 3.5 - SAVE')

adata_combined.write_h5ad(H5_INTEGRATED)
print(f'  Saved {os.path.basename(H5_INTEGRATED)}')
print(f'  Stored embeddings: '
      f'{[k for k in adata_combined.obsm if k.startswith("X_umap")]}')
print(f'  Figures: {FIG_DIR}')
print(f'  Tables:  {TAB_DIR}')
print('  Next: step3b_integration_qc.py, then step4 / step4c')

In [ ]:
"""
=============================================================================
STEP 3B - Integration diagnostics (read-only)
=============================================================================

  in : 02b_combined_integrated.h5ad
  out: figures under INTEG_DIR and FIG_DIR, tables under TAB_DIR
       nothing is written back to the h5ad

Sections:
  3B.1  the original multi-panel UMAP, for comparison with the old notebook
  3B.2  did the two assays actually mix?
  3B.3  cluster composition
  3B.4  is the embedding driven by depth rather than identity?
  3B.5  Leiden clusters vs reference cell types, annotated counts
  3B.6  UMAP coloured by Leiden cluster

Section 3B.4 matters here because the doublet score turned out to correlate
with depth at rho = 0.65. If the UMAP does too, the integration is arranging
cells by library size rather than biology.

Safe to rerun at any time; it changes nothing.
=============================================================================
"""

import os
import sys

try:
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__))
                    if '__file__' in globals() else os.getcwd())
    from pipeline_config import *  # noqa: F401,F403
except ModuleNotFoundError:
    if 'H5_DCM_CLEAN' not in globals():
        raise RuntimeError(
            'Config not loaded. Either run the pipeline_config cell at the top '
            'of this notebook, or save pipeline_config.py somewhere on sys.path.'
        ) from None
    print('  Using config from the notebook session.')

# %%
banner('STEP 3B - INTEGRATION DIAGNOSTICS')

adata = load_step_input(H5_INTEGRATED, 'step3_integrate.py')

sc.settings.figdir = INTEG_DIR
sc.set_figure_params(dpi=300, dpi_save=600, fontsize=6)

ANNOT_COL = f'leiden_annot_{BEST_DIM}'
is_q = (adata.obs['dataset'] == 'sciMeDseq').values
is_r = ~is_q
umap = adata.obsm['X_umap']
leiden = adata.obs['leiden'].astype(str)
labels = adata.obs[LABEL_COL].astype(str)
clusters = sorted(leiden.unique(), key=lambda x: int(x) if x.isdigit() else x)

print(f'  Query (sciMeDseq): {is_q.sum()}   Reference: {is_r.sum()}')
print(f'  Leiden clusters: {len(clusters)}')


# %%
# =============================================================================
# 3B.1 - THE ORIGINAL MULTI-PANEL UMAP
# =============================================================================
banner('3B.1 - UMAP PANELS (as in the original notebook)')

plot_cols = [c for c in ['Dox', 'Sample', LABEL_COL, ANNOT_COL, 'leiden',
                         'dataset', 'orig.ident']
             if c in adata.obs.columns]
print(f'  Colouring by: {plot_cols}')

sc.pl.umap(adata, color=plot_cols, frameon=False, legend_loc='on data',
           ncols=3, save=f'_{BEST_DIM}_diagnostics_ondata.png', show=False)

# same panels with the legend beside the plot, easier to read for many classes
sc.pl.umap(adata, color=plot_cols, frameon=False, ncols=3,
           save=f'_{BEST_DIM}_diagnostics_legend.png', show=False)


# %%
# =============================================================================
# 3B.2 - DID THE ASSAYS MIX?
# =============================================================================
banner('3B.2 - BATCH MIXING')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(umap[is_r, 0], umap[is_r, 1], s=3, alpha=0.35,
                c='#BBBBBB', label=f'reference (n={is_r.sum()})', rasterized=True)
axes[0].scatter(umap[is_q, 0], umap[is_q, 1], s=10, alpha=0.85,
                c='#DC143C', label=f'sciMeDseq (n={is_q.sum()})', rasterized=True)
axes[0].legend(fontsize=8, markerscale=2)
axes[0].set_title('Overlay', fontweight='bold')

for ct in LABEL_ORDER:
    m = is_r & (labels == ct).values
    if m.sum():
        axes[1].scatter(umap[m, 0], umap[m, 1], s=3, alpha=0.6,
                        c=CT_COLORS.get(ct, '#777777'), label=f'{ct} ({m.sum()})',
                        rasterized=True)
axes[1].legend(fontsize=6, markerscale=3, loc='best', frameon=False)
axes[1].set_title('Reference only, by cell_type_broad', fontweight='bold')

if INTEG_COL in adata.obs.columns:
    qlab = adata.obs[INTEG_COL].astype(str)
    for ct in sorted(qlab[is_q].unique()):
        m = is_q & (qlab == ct).values
        axes[2].scatter(umap[m, 0], umap[m, 1], s=12, alpha=0.85,
                        c=CT_COLORS.get(ct, '#777777'), label=f'{ct} ({m.sum()})',
                        rasterized=True)
    axes[2].legend(fontsize=6, markerscale=2, loc='best', frameon=False)
axes[2].set_title(f'sciMeDseq only, by {INTEG_COL}', fontweight='bold')

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')
fig.tight_layout()
save_fig(fig, 'step3b_umap_by_dataset.pdf')

# --- kNN mixing score --------------------------------------------------------
# For each cell: what fraction of its graph neighbours come from the other
# dataset? Perfect mixing would put this near the other dataset's global share.
if 'connectivities' in adata.obsp:
    binary = (adata.obsp['connectivities'] > 0).astype(np.int8)
    n_neigh = np.asarray(binary.sum(axis=1)).ravel()
    n_q_neigh = np.asarray(binary[:, is_q].sum(axis=1)).ravel()
    frac_q_neigh = np.divide(n_q_neigh, np.maximum(n_neigh, 1))
    global_q = is_q.mean()

    print(f'  Global sciMeDseq share: {global_q:.3f}')
    print(f'  Mean query-neighbour fraction, query cells:     '
          f'{frac_q_neigh[is_q].mean():.3f}')
    print(f'  Mean query-neighbour fraction, reference cells: '
          f'{frac_q_neigh[is_r].mean():.3f}')
    print('  (both near the global share = well mixed; query near 1.0 = the')
    print('   assays are sitting in separate regions of the graph)')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(frac_q_neigh[is_q], bins=40, alpha=0.7, color='#DC143C',
                 density=True, label='sciMeDseq cells')
    axes[0].hist(frac_q_neigh[is_r], bins=40, alpha=0.7, color='#4169E1',
                 density=True, label='reference cells')
    axes[0].axvline(global_q, color='k', ls='--', lw=1.5,
                    label=f'global share {global_q:.3f}')
    axes[0].set_xlabel('Fraction of neighbours that are sciMeDseq')
    axes[0].set_ylabel('Density')
    axes[0].set_title('kNN batch mixing', fontweight='bold')
    axes[0].legend(fontsize=8)
    axes[0].spines['top'].set_visible(False)
    axes[0].spines['right'].set_visible(False)

    sca = axes[1].scatter(umap[:, 0], umap[:, 1], s=4, c=frac_q_neigh,
                          cmap='coolwarm', vmin=0, vmax=1, rasterized=True)
    plt.colorbar(sca, ax=axes[1], label='fraction sciMeDseq neighbours')
    axes[1].set_xticks([]); axes[1].set_yticks([])
    for s in axes[1].spines.values():
        s.set_visible(False)
    axes[1].set_title('Mixing on the embedding', fontweight='bold')
    fig.tight_layout()
    save_fig(fig, 'step3b_knn_mixing.pdf')
else:
    print('  No connectivities in .obsp; skipping the kNN mixing score.')

try:
    sc.tl.embedding_density(adata, groupby='orig.ident', key_added='batch_dense')
    sc.pl.embedding_density(adata, key='batch_dense',
                            save=f'_{BEST_DIM}_step3b_density.png', show=False)
except Exception as exc:  # noqa: BLE001
    print(f'  embedding_density failed ({exc}); skipping.')


# %%
# =============================================================================
# 3B.3 - CLUSTER COMPOSITION
# =============================================================================
banner('3B.3 - CLUSTER COMPOSITION')

comp = pd.crosstab(leiden, adata.obs['dataset'].astype(str)).reindex(clusters)
comp = comp.fillna(0).astype(int)
comp['total'] = comp.sum(axis=1)
comp['pct_sciMeDseq'] = (100 * comp.get('sciMeDseq', 0) / comp['total']).round(1)
print(comp.to_string())
comp.to_csv(os.path.join(TAB_DIR, f'step3b_cluster_composition_dim{BEST_DIM}.csv'))

query_only = comp[comp['pct_sciMeDseq'] >= 90]
if len(query_only):
    print(f'\n  {len(query_only)} cluster(s) are >=90% sciMeDseq: '
          f'{list(query_only.index)}')
    print('  These have essentially no reference cells to transfer a label from.')

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

frac = comp[[c for c in ['reference', 'sciMeDseq'] if c in comp.columns]]
frac = frac.div(frac.sum(axis=1), axis=0)
bottom = np.zeros(len(frac))
for col, colour in [('reference', '#4169E1'), ('sciMeDseq', '#DC143C')]:
    if col in frac.columns:
        axes[0].bar(frac.index, frac[col], bottom=bottom, color=colour, label=col)
        bottom += frac[col].values
axes[0].axhline(0.5, color='k', ls=':', lw=1)
axes[0].set_xlabel('Leiden cluster'); axes[0].set_ylabel('Fraction of cells')
axes[0].set_title(f'Cluster composition (dim {BEST_DIM})', fontweight='bold')
axes[0].legend(fontsize=8)
axes[0].tick_params(axis='x', rotation=90, labelsize=6)

xt_all = pd.crosstab(leiden, labels).reindex(clusters).fillna(0).astype(int)
keep_cols = [c for c in LABEL_ORDER if c in xt_all.columns]
if 'Unknown' in xt_all.columns:
    keep_cols = keep_cols + ['Unknown']
xt_all = xt_all[keep_cols]
sns.heatmap(xt_all.div(xt_all.sum(axis=1).replace(0, np.nan), axis=0),
            annot=False, cmap='Blues', vmin=0, vmax=1, ax=axes[1],
            cbar_kws={'label': 'Fraction of cluster'})
axes[1].set_xlabel(LABEL_COL); axes[1].set_ylabel('Leiden cluster')
axes[1].set_title('Cluster vs reference label (all cells)', fontweight='bold')
axes[1].tick_params(labelsize=6)

fig.tight_layout()
save_fig(fig, 'step3b_cluster_composition.pdf')


# %%
# =============================================================================
# 3B.4 - IS THE EMBEDDING DRIVEN BY DEPTH?
# =============================================================================
banner('3B.4 - DEPTH CHECK')

# Depth columns may have been coerced to string when the objects were merged
# (reference cells had no value), so parse back to numeric and use query cells.
depth = {}
for col in ['total_counts', 'n_genes_by_counts']:
    if col in adata.obs.columns:
        v = pd.to_numeric(adata.obs[col], errors='coerce').values
        if np.isfinite(v[is_q]).sum() > 10:
            depth[col] = v

if not depth:
    print('  No usable depth columns on the query cells; skipping.')
else:
    from scipy.stats import spearmanr

    fig, axes = plt.subplots(1, 2 * len(depth), figsize=(7 * len(depth), 4.5))
    axes = np.atleast_1d(axes)
    k = 0
    for col, v in depth.items():
        vq = v[is_q]
        ok = np.isfinite(vq)
        lv = np.log10(np.maximum(vq, 1))
        for j, comp_axis in enumerate(['UMAP1', 'UMAP2']):
            coord = umap[is_q, j]
            rho, p = spearmanr(lv[ok], coord[ok])
            sca = axes[k].scatter(umap[is_q, 0], umap[is_q, 1], s=14, c=lv,
                                  cmap='RdBu_r', rasterized=True)
            if j == 0:
                plt.colorbar(sca, ax=axes[k], label=f'log10 {col}')
            axes[k].set_xticks([]); axes[k].set_yticks([])
            for s in axes[k].spines.values():
                s.set_visible(False)
            axes[k].set_title(f'{col}\nrho vs {comp_axis} = {rho:+.2f}',
                              fontweight='bold', fontsize=9)
            print(f'  {col:<20} vs {comp_axis}: rho = {rho:+.3f} (p = {p:.2g})')
            k += 1

    fig.tight_layout()
    save_fig(fig, 'step3b_depth_on_umap.pdf')

    print('\n  |rho| above ~0.5 on either axis means the embedding is arranging')
    print('  sciMeDseq cells largely by library size, and any label transferred')
    print('  from neighbouring reference cells inherits that.')


# %%
# =============================================================================
# 3B.5 - LEIDEN CLUSTERS VS REFERENCE CELL TYPES (annotated counts)
# =============================================================================
banner('3B.5 - CLUSTER x REFERENCE CELL TYPE')

# counts of REFERENCE cells only; the query total is shown as a side bar
ct_ref = pd.crosstab(leiden[is_r], labels[is_r])
ct_ref = ct_ref[[c for c in LABEL_ORDER if c in ct_ref.columns]]
ct_ref = ct_ref.reindex(clusters).fillna(0).astype(int)

n_query = leiden[is_q].value_counts().reindex(clusters).fillna(0).astype(int)
n_ref = ct_ref.sum(axis=1)

# colour by row fraction so small clusters stay readable
frac_ref = ct_ref.div(n_ref.replace(0, np.nan), axis=0)

h = max(4.0, 0.32 * len(ct_ref))
fig, (ax, axb) = plt.subplots(
    1, 2, figsize=(1.05 * len(ct_ref.columns) + 6.5, h),
    gridspec_kw={'width_ratios': [len(ct_ref.columns), 2.2], 'wspace': 0.04},
)

sns.heatmap(frac_ref, annot=ct_ref, fmt='d', cmap='Blues', vmin=0, vmax=1,
            linewidths=0.4, linecolor='white', ax=ax, annot_kws={'size': 7},
            cbar_kws={'label': "Fraction of the cluster's reference cells",
                      'shrink': 0.6})
ax.set_xlabel(LABEL_COL)
ax.set_ylabel(f'Leiden cluster (dim {BEST_DIM})')
ax.set_title('Reference cells per cluster (numbers = cell counts)',
             fontweight='bold')
ax.tick_params(axis='y', rotation=0, labelsize=7)
ax.tick_params(axis='x', rotation=45, labelsize=8)
for lbl in ax.get_xticklabels():
    lbl.set_ha('right')

ypos = np.arange(len(ct_ref)) + 0.5
axb.barh(ypos, n_query.values, height=0.72, color='#DC143C', alpha=0.85)
axb.set_ylim(len(ct_ref), 0)
axb.set_yticks([])
axb.set_xlabel('sciMeDseq cells')
axb.set_title('Query', fontweight='bold', fontsize=9)
axb.spines['top'].set_visible(False)
axb.spines['right'].set_visible(False)
for yi, v in zip(ypos, n_query.values):
    if v > 0:
        axb.text(v, yi, f' {v}', va='center', ha='left', fontsize=6.5)
axb.set_xlim(0, max(1, n_query.max() * 1.25))

fig.tight_layout()
save_fig(fig, f'step3b_leiden_vs_reference_counts_dim{BEST_DIM}.pdf')

out = ct_ref.copy()
out['n_reference'] = n_ref
out['n_sciMeDseq'] = n_query
out['pct_sciMeDseq'] = (100 * n_query / (n_ref + n_query)).round(1)
out['majority'] = ct_ref.idxmax(axis=1).where(n_ref > 0, 'no_reference')
out['purity'] = (ct_ref.max(axis=1) / n_ref.replace(0, np.nan)).round(3)

print('\n' + out.to_string())
out.to_csv(os.path.join(TAB_DIR,
                        f'step3b_leiden_vs_reference_counts_dim{BEST_DIM}.csv'))

no_ref = out[out['n_reference'] == 0]
thin = out[(out['n_reference'] > 0) & (out['n_reference'] < 10)]
impure = out[out['purity'] < 0.6]
print(f'\n  Clusters with no reference cells: {len(no_ref)} '
      f'{list(no_ref.index) if len(no_ref) else ""}')
print(f'  Clusters with fewer than 10 reference cells: {len(thin)} '
      f'{list(thin.index) if len(thin) else ""}')
print(f'  sciMeDseq cells in those clusters: '
      f'{int(no_ref["n_sciMeDseq"].sum() + thin["n_sciMeDseq"].sum())} '
      f'of {int(n_query.sum())}')
print(f'  Clusters below 0.6 purity (majority vote is not clear-cut): '
      f'{len(impure)} {list(impure.index) if len(impure) else ""}')


# %%
# =============================================================================
# 3B.6 - UMAP BY LEIDEN CLUSTER
# =============================================================================
banner('3B.6 - UMAP BY LEIDEN CLUSTER')

palette = (list(plt.get_cmap('tab20').colors) +
           list(plt.get_cmap('tab20b').colors))
colours = {c: palette[i % len(palette)] for i, c in enumerate(clusters)}

if ANNOT_COL in adata.obs.columns:
    annot_type = adata.obs[ANNOT_COL].astype(str).str.split('_').str[-1]
    ctype = {c: annot_type[leiden == c].iloc[0] for c in clusters}
else:
    ctype = {c: '' for c in clusters}

n_r_cl = leiden[is_r].value_counts()
n_q_cl = leiden[is_q].value_counts()


def _centroid(mask):
    return (np.median(umap[mask, 0]), np.median(umap[mask, 1]))


fig, axes = plt.subplots(2, 2, figsize=(15, 13))

ax = axes[0, 0]
for c in clusters:
    m = (leiden == c).values
    ax.scatter(umap[m, 0], umap[m, 1], s=3, alpha=0.7, color=colours[c],
               rasterized=True)
    x, y = _centroid(m)
    ax.text(x, y, c, fontsize=9, fontweight='bold', ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.75))
ax.set_title(f'Leiden clusters, all cells (dim {BEST_DIM})', fontweight='bold')

ax = axes[0, 1]
ax.scatter(umap[:, 0], umap[:, 1], s=2, color='#EEEEEE', rasterized=True)
for c in clusters:
    m = (leiden == c).values & is_r
    if m.sum():
        ax.scatter(umap[m, 0], umap[m, 1], s=3, alpha=0.7, color=colours[c],
                   rasterized=True)
ax.set_title(f'Reference only (n={is_r.sum()})', fontweight='bold')

ax = axes[1, 0]
ax.scatter(umap[:, 0], umap[:, 1], s=2, color='#EEEEEE', rasterized=True)
for c in clusters:
    m = (leiden == c).values & is_q
    if m.sum():
        ax.scatter(umap[m, 0], umap[m, 1], s=22, alpha=0.9, color=colours[c],
                   edgecolors='white', linewidths=0.3, rasterized=True)
        x, y = _centroid(m)
        ax.text(x, y, c, fontsize=8, fontweight='bold', ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.12', fc='white', ec='none',
                          alpha=0.8))
ax.set_title(f'sciMeDseq only (n={is_q.sum()})', fontweight='bold')

ax = axes[1, 1]
ax.axis('off')
handles = [
    plt.Line2D([0], [0], marker='o', ls='', markersize=6, color=colours[c],
               label=f'{c}: {ctype[c]}  (ref {int(n_r_cl.get(c, 0))} / '
                     f'sci {int(n_q_cl.get(c, 0))})')
    for c in clusters
]
ax.legend(handles=handles, loc='center', ncol=2 if len(clusters) > 18 else 1,
          fontsize=7.5, frameon=False,
          title='cluster: transferred type  (cell counts)', title_fontsize=9)

for ax in axes.ravel()[:3]:
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel('UMAP1', fontsize=8); ax.set_ylabel('UMAP2', fontsize=8)
    for s in ax.spines.values():
        s.set_visible(False)

fig.tight_layout()
save_fig(fig, f'step3b_umap_leiden_clusters_dim{BEST_DIM}.pdf')

# scanpy version, for direct comparison with the original notebook
sc.pl.umap(adata, color='leiden', legend_loc='on data', frameon=False,
           title=f'leiden (dim {BEST_DIM})',
           save=f'_{BEST_DIM}_step3b_leiden.png', show=False)

print(f'\n  Figures: {INTEG_DIR} and {FIG_DIR}')
print(f'  Tables:  {TAB_DIR}')
print('  Nothing was written back to the h5ad.')

In [ ]:
"""
=============================================================================
STEP 4 - Marker sets from the reference
=============================================================================

  in : 02b_combined_integrated.h5ad
  out: tables/final_gene_sets.csv
       tables/reference_de_markers.csv

DE is run on the reference half of the combined object, which already holds
only the shared genes and the same lognorm layer. That keeps this step
independent of the original Haber file.

Rerun when LITERATURE_MARKERS, DE_* or LABEL_ORDER change. Cheap - no
integration involved.
=============================================================================
"""

import os
import sys

try:
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__))
                    if '__file__' in globals() else os.getcwd())
    from pipeline_config import *  # noqa: F401,F403
except ModuleNotFoundError:
    if 'H5_DCM_CLEAN' not in globals():
        raise RuntimeError(
            'Config not loaded. Either run the pipeline_config cell at the top '
            'of this notebook, or save pipeline_config.py somewhere on sys.path.'
        ) from None
    print('  Using config from the notebook session.')

# %%
banner('STEP 4 - MARKER SETS')

adata_combined = load_step_input(H5_INTEGRATED, 'step3_integrate.py')

# %%
# --- literature --------------------------------------------------------------
if LITERATURE_CSV is not None:
    lit_df = pd.read_csv(LITERATURE_CSV)
    lit_dict = {ct: g['gene'].tolist() for ct, g in lit_df.groupby('cell_type')}
    lit_dict = {NAME_MAP.get(k, k): v for k, v in lit_dict.items()}
elif LITERATURE_MARKERS is not None:
    lit_dict = {NAME_MAP.get(k, k): list(v) for k, v in LITERATURE_MARKERS.items()}
else:
    lit_dict = {}
print(f'  Literature sets provided for: {sorted(lit_dict)}')

# %%
# --- DE markers from the reference half --------------------------------------
ad_de = adata_combined[adata_combined.obs['dataset'] == 'reference'].copy()
ad_de.X = ad_de.layers['lognorm'].copy()
# rebuild categories from the cells actually present, so rank_genes_groups is
# not handed empty groups left over from the combined object
ad_de.obs[LABEL_COL] = ad_de.obs[LABEL_COL].astype(str).astype('category')
print(f'  Reference cells for DE: {ad_de.n_obs} x {ad_de.n_vars} genes')
print(ad_de.obs[LABEL_COL].value_counts().to_string())

print(f'\n  Running rank_genes_groups ({DE_METHOD})...')
sc.tl.rank_genes_groups(ad_de, groupby=LABEL_COL, method=DE_METHOD, pts=True)

res = ad_de.uns['rank_genes_groups']
de_df = pd.concat([
    pd.DataFrame({
        'gene': res['names'][ct],
        'cell_type': ct,
        'logfc': res['logfoldchanges'][ct],
        'pvals_adj': res['pvals_adj'][ct],
    })
    for ct in res['names'].dtype.names
], ignore_index=True)

sig = de_df[(de_df['logfc'] >= DE_LOGFC_MIN) &
            (de_df['pvals_adj'] < DE_PADJ_MAX)].copy()
print(f'  Significant DE genes: {len(sig)}')

if DE_UNIQUE_ONLY:
    counts_per_gene = sig['gene'].value_counts()
    unique_genes = counts_per_gene[counts_per_gene == 1].index
    sig = sig[sig['gene'].isin(unique_genes)].copy()
    print(f'  After keeping type-unique genes: {len(sig)}')

top_de = (sig.sort_values(['cell_type', 'logfc'], ascending=[True, False])
             .groupby('cell_type').head(DE_TOP_N))
de_dict = {ct: g['gene'].tolist() for ct, g in top_de.groupby('cell_type')}
top_de.to_csv(os.path.join(TAB_DIR, 'reference_de_markers.csv'), index=False)

# %%
# --- union, restricted to genes present in the combined object ---------------
present = set(adata_combined.var_names)
gene_sets = {}
for ct in LABEL_ORDER:
    genes = list(lit_dict.get(ct, [])) + list(de_dict.get(ct, []))
    genes = [g for g in dict.fromkeys(genes) if g in present]
    if genes:
        gene_sets[ct] = genes

CLASSES = [ct for ct in LABEL_ORDER if ct in gene_sets]
dropped = [ct for ct in LABEL_ORDER if ct not in gene_sets]
if dropped:
    print(f'  WARNING: no usable genes for {dropped} - dropped from the model')

print('\n  Final gene sets:')
for ct in CLASSES:
    n_lit = len([g for g in lit_dict.get(ct, []) if g in present])
    n_tot = len(gene_sets[ct])
    warn = '  <-- thin' if n_tot < 10 else ''
    print(f'    {ct:>12}: {n_tot:>3} genes ({n_lit} literature){warn}')

pd.DataFrame([{'cell_type': ct, 'gene': g}
              for ct in CLASSES for g in gene_sets[ct]]).to_csv(
    CSV_GENESETS, index=False)

print(f'\n  Saved {os.path.basename(CSV_GENESETS)}')
print('  Next: step5_classify.py')

In [ ]:
"""
=============================================================================
STEP 4C - Gene sets from a literature marker CSV
=============================================================================

  in : LITERATURE_GENESET_CSV, 02b_combined_integrated.h5ad
  out: tables/final_gene_sets.csv          (what step5 / step5b read)
       tables/literature_gene_match_report.csv
       figures/step4c_geneset_sizes.pdf

Use this INSTEAD of step4_markers.py when the gene sets should come from the
CSV rather than being re-derived by differential expression. Run step5b_gmm_fdr
(or step5_classify) afterwards.

Column handling
---------------
Expects one row per gene x cell type. Column names are detected from common
variants: cell_type / celltype / cluster / type, and gene / gene_symbol /
marker / symbol. If the file has 'source' and 'is_specific' columns - as
specific_markers.csv did - they are used for optional filtering.

Rows whose cell_type contains ';' are the non-specific entries the original
code wrote for genes shared by several types. They are always dropped: a gene
assigned to three cell types cannot contribute to distinguishing them.

The important output is the match report. A gene named in the CSV but absent
from the DCM data cannot be scored on sciMeDseq cells, and silent dropping is
how a 15-gene literature set becomes a 3-gene module score without anyone
noticing.
=============================================================================
"""

import os
import sys

try:
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__))
                    if '__file__' in globals() else os.getcwd())
    from pipeline_config import *  # noqa: F401,F403
except ModuleNotFoundError:
    if 'H5_DCM_CLEAN' not in globals():
        raise RuntimeError(
            'Config not loaded. Either run the pipeline_config cell at the top '
            'of this notebook, or save pipeline_config.py somewhere on sys.path.'
        ) from None
    print('  Using config from the notebook session.')

# %%
# =============================================================================
# 4C.1 - READ AND PARSE THE CSV
# =============================================================================
banner('STEP 4C.1 - LITERATURE MARKER CSV')

require_input(LITERATURE_GENESET_CSV, 'nothing - check the path in the config')
lit = pd.read_csv(LITERATURE_GENESET_CSV)
print(f'  File: {LITERATURE_GENESET_CSV}')
print(f'  Shape: {lit.shape[0]} rows x {lit.shape[1]} columns')
print(f'  Columns: {lit.columns.tolist()}')
print('\n  First rows:')
print(lit.head(8).to_string())

# --- detect the two columns we need ------------------------------------------
CT_CANDIDATES = ['cell_type', 'celltype', 'cluster', 'type', 'cell type',
                 'CellType', 'group']
G_CANDIDATES = ['gene', 'gene_symbol', 'marker', 'symbol', 'genes', 'Gene']

lower = {c.lower().strip(): c for c in lit.columns}
ct_col = next((lower[c] for c in [x.lower() for x in CT_CANDIDATES]
               if c in lower), None)
g_col = next((lower[c] for c in [x.lower() for x in G_CANDIDATES]
              if c in lower), None)

if ct_col is None or g_col is None:
    raise ValueError(
        f'Could not identify the cell-type and gene columns.\n'
        f'  Found columns: {lit.columns.tolist()}\n'
        f'  Set them by hand, e.g.:  ct_col, g_col = "my_type", "my_gene"\n'
        f'  then rerun from the parsing section.'
    )
print(f'\n  Using cell_type column: "{ct_col}"')
print(f'  Using gene column:      "{g_col}"')

df = lit[[ct_col, g_col]].copy()
df.columns = ['cell_type', 'gene']
for extra in ('source', 'is_specific'):
    if extra in lit.columns:
        df[extra] = lit[extra].values

df['cell_type'] = df['cell_type'].astype(str).str.strip()
df['gene'] = df['gene'].astype(str).str.strip()
n_start = len(df)

# --- drop the non-specific rows ----------------------------------------------
multi = df['cell_type'].str.contains(';', na=False)
if multi.any():
    print(f'\n  Dropping {int(multi.sum())} rows whose cell_type lists several '
          f'types (non-specific markers)')
    df = df[~multi].copy()

if 'is_specific' in df.columns and LIT_REQUIRE_SPECIFIC:
    spec = df['is_specific'].astype(str).str.lower().isin(['true', '1', 'yes'])
    print(f'  is_specific column present: keeping {int(spec.sum())} of '
          f'{len(df)} rows')
    df = df[spec].copy()

if 'source' in df.columns:
    print(f'\n  source values: {df["source"].value_counts().to_dict()}')
    if LIT_SOURCE_FILTER is not None:
        df = df[df['source'].astype(str) == LIT_SOURCE_FILTER].copy()
        print(f'  LIT_SOURCE_FILTER="{LIT_SOURCE_FILTER}": {len(df)} rows kept')

# --- harmonise the cell-type names -------------------------------------------
df['cell_type'] = df['cell_type'].replace(NAME_MAP)
unknown_types = sorted(set(df['cell_type']) - set(LABEL_ORDER))
if unknown_types:
    print(f'\n  Cell types in the CSV that are not in LABEL_ORDER, dropped: '
          f'{unknown_types}')
    print(f'  (LABEL_ORDER = {LABEL_ORDER}; add them to NAME_MAP if these are '
          f'alternative spellings)')
    df = df[df['cell_type'].isin(LABEL_ORDER)].copy()

df = df.drop_duplicates(subset=['cell_type', 'gene'])
print(f'\n  Rows after parsing: {len(df)} (from {n_start})')
print(df.groupby('cell_type').size().reindex(LABEL_ORDER).to_string())


# %%
# =============================================================================
# 4C.2 - MATCH AGAINST THE DATA
# =============================================================================
banner('STEP 4C.2 - GENE NAME MATCHING')

adata = load_step_input(H5_INTEGRATED, 'step3_integrate.py')
present = set(adata.var_names)
print(f'  Genes shared by reference and DCM: {len(present)}')

df['in_data'] = df['gene'].isin(present)

# case-insensitive rescue, for symbol-case mismatches (LGR5 vs Lgr5)
ci = {g.lower(): g for g in present}
rescued = {}
for g in df.loc[~df['in_data'], 'gene'].unique():
    hit = ci.get(g.lower())
    if hit is not None:
        rescued[g] = hit
if rescued:
    print(f'\n  {len(rescued)} gene(s) matched only after ignoring case; '
          f'renaming them:')
    for a, b in list(rescued.items())[:15]:
        print(f'    {a} -> {b}')
    if len(rescued) > 15:
        print(f'    ... and {len(rescued) - 15} more')
    df['gene'] = df['gene'].replace(rescued)
    df['in_data'] = df['gene'].isin(present)

match = df.groupby('cell_type').agg(
    n_in_csv=('gene', 'size'), n_in_data=('in_data', 'sum')
).reindex(LABEL_ORDER).fillna(0).astype(int)
match['n_missing'] = match['n_in_csv'] - match['n_in_data']
match['pct_found'] = (100 * match['n_in_data'] /
                      match['n_in_csv'].replace(0, np.nan)).round(1)
print('\n  Per cell type:')
print(match.to_string())

missing = df[~df['in_data']]
if len(missing):
    print(f'\n  {len(missing)} gene x cell-type rows reference genes absent '
          f'from the data.')
    print(f'  Examples: {sorted(missing["gene"].unique())[:20]}')
    if match['pct_found'].min() < 50:
        worst = match['pct_found'].idxmin()
        print(f'\n  WARNING: {worst} only matched '
              f'{match.loc[worst, "pct_found"]}% of its genes. That usually '
              f'means a naming convention mismatch (Ensembl IDs vs symbols, or '
              f'a different genome build), not biology.')

df.to_csv(os.path.join(TAB_DIR, 'literature_gene_match_report.csv'), index=False)


# %%
# =============================================================================
# 4C.3 - BUILD AND WRITE THE GENE SETS
# =============================================================================
banner('STEP 4C.3 - GENE SETS')

kept = df[df['in_data']].copy()
gene_sets = {}
for ct in LABEL_ORDER:
    genes = kept.loc[kept['cell_type'] == ct, 'gene'].tolist()
    genes = list(dict.fromkeys(genes))
    if LIT_MAX_PER_TYPE is not None:
        genes = genes[:LIT_MAX_PER_TYPE]
    if genes:
        gene_sets[ct] = genes

CLASSES = [ct for ct in LABEL_ORDER if ct in gene_sets]
dropped = [ct for ct in LABEL_ORDER if ct not in gene_sets]
if dropped:
    print(f'  No usable genes for {dropped} - these classes are dropped')

print('\n  Final gene sets:')
thin = []
for ct in CLASSES:
    n = len(gene_sets[ct])
    flag = ''
    if n < 5:
        flag = '  <-- very thin, score_genes will be noisy'
        thin.append(ct)
    elif n < 10:
        flag = '  <-- thin'
        thin.append(ct)
    print(f'    {ct:>12}: {n:>3} genes{flag}')

pd.DataFrame([{'cell_type': ct, 'gene': g}
              for ct in CLASSES for g in gene_sets[ct]]).to_csv(
    CSV_GENESETS, index=False)
print(f'\n  Wrote {os.path.basename(CSV_GENESETS)} ({len(CLASSES)} classes)')

fig, ax = plt.subplots(figsize=(7, 4.5))
xpos = np.arange(len(match))
ax.bar(xpos, match['n_in_csv'], color='#DDDDDD', label='in the CSV')
ax.bar(xpos, match['n_in_data'],
       color=[CT_COLORS.get(c, '#777777') for c in match.index],
       label='present in the data')
ax.set_xticks(xpos)
ax.set_xticklabels(match.index, rotation=45, ha='right')
ax.set_ylabel('Genes')
ax.set_title('Literature marker sets, before and after matching',
             fontweight='bold')
ax.legend(fontsize=8)
for i, (a, b) in enumerate(zip(match['n_in_csv'], match['n_in_data'])):
    ax.text(i, a, f'{b}/{a}', ha='center', va='bottom', fontsize=7)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
fig.tight_layout()
save_fig(fig, 'step4c_geneset_sizes.pdf')

print('\n  Next: step5b_gmm_fdr.py  (or step5_classify.py)')
if thin:
    print(f'\n  With {thin} under 10 genes, set GMM_COVARIANCE = "tied" in the')
    print('  config - per-class full covariances are unstable on thin sets.')

In [ ]:
#Based on gmm

In [ ]:
"""
=============================================================================
STEP 5B - GMM on the module scores with Bayesian FDR control
=============================================================================

  in : 02b_combined_integrated.h5ad, tables/final_gene_sets.csv
  out: tables/gmm_fdr_classification.csv
       tables/gmm_fdr_summary.csv
       tables/gmm_call_vs_<INTEG_COL>.csv
       figures/step5b_gmm_fdr.pdf
       figures/step5b_gmm_score_space.pdf
       nothing is written back to the h5ad

WHAT THIS DOES DIFFERENTLY FROM THE EARLIER GMM ATTEMPTS
--------------------------------------------------------
1. One joint GMM over the full score vector, not one 1-D two-component fit per
   module. Posteriors sum to 1 across classes, so the classes compete and each
   cell gets exactly one label by construction.

2. No component -> class collision. In GMM_MODE='reference_anchored' the
   components ARE the classes: each class Gaussian is estimated from the
   labelled reference cells and only the mixture weights are re-fit on the
   query by EM. In 'unsupervised' mode the mapping is solved as an assignment
   problem (linear_sum_assignment against the reference class centroids), which
   forces a bijection - the earlier run mapped two components to Paneth and
   left Stem unassigned.

3. A real Bayesian FDR. local fdr for a cell is (1 - max posterior); the
   expected false-discovery rate over a set of calls is the mean of those local
   fdrs. Sort ascending, keep the largest prefix whose running mean stays at or
   below BAYES_FDR. This is the direct posterior-probability approach (Newton
   et al. 2004) and needs no permutation null.

4. A separate density-based outlier test. A GMM gives a likelihood, so a query
   cell whose log-density falls below the reference's GMM_OUTLIER_PCT
   percentile is flagged not_in_reference. Being confidently classified and
   being a good fit to the reference are different questions.

CAVEAT
------
With SCALE_MODE='zscore' the reference and query scores are standardised
within their own dataset, so the reference Gaussians live in reference-z space
and the query points in query-z space. That is the usual workaround for the
assay shift, but it forces every module to mean 0 in the query: if the
sciMeDseq cells are genuinely skewed toward one type, the scaling partly
erases that. Run once with SCALE_MODE='none' to see how much it matters.

Safe to rerun; it changes nothing upstream.
=============================================================================
"""

import os
import sys

try:
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__))
                    if '__file__' in globals() else os.getcwd())
    from pipeline_config import *  # noqa: F401,F403
except ModuleNotFoundError:
    if 'H5_DCM_CLEAN' not in globals():
        raise RuntimeError(
            'Config not loaded. Either run the pipeline_config cell at the top '
            'of this notebook, or save pipeline_config.py somewhere on sys.path.'
        ) from None
    print('  Using config from the notebook session.')

from sklearn.mixture import GaussianMixture
from scipy.optimize import linear_sum_assignment
from scipy.stats import multivariate_normal

# %%
# =============================================================================
# 5B.1 - MODULE SCORES
# =============================================================================
banner('STEP 5B.1 - MODULE SCORES')

adata = load_step_input(H5_INTEGRATED, 'step3_integrate.py')
gene_sets, CLASSES = load_gene_sets()
K = len(CLASSES)
print(f'  Classes ({K}): {CLASSES}')

is_q = (adata.obs['dataset'] == 'sciMeDseq').values
is_r = ~is_q

ad_r = adata[is_r].copy()
ad_q = adata[is_q].copy()
ad_r.X = ad_r.layers['lognorm'].copy()
ad_q.X = ad_q.layers['lognorm'].copy()


def score_block(ad, tag):
    for ct in CLASSES:
        sc.tl.score_genes(ad, gene_list=gene_sets[ct],
                          score_name=f'{ct}_score', use_raw=False)
    df = ad.obs[[f'{ct}_score' for ct in CLASSES]].astype(float).copy()
    df.columns = CLASSES
    print(f'  {tag}: {df.shape[0]} cells x {df.shape[1]} scores')
    return df


S_ref = scale_block(score_block(ad_r, 'reference'))
S_qry = scale_block(score_block(ad_q, 'query    '))
print(f'  Scaling mode: {SCALE_MODE}')

X_ref = S_ref.to_numpy(dtype=np.float64)
X_qry = S_qry.to_numpy(dtype=np.float64)

y_ref = pd.Categorical(ad_r.obs[LABEL_COL].astype(str).values,
                       categories=CLASSES).codes.astype(int)
ok = y_ref >= 0
if (~ok).sum():
    print(f'  Dropping {int((~ok).sum())} reference cells outside CLASSES')
    X_ref, y_ref = X_ref[ok], y_ref[ok]

print(f'  Reference cells per class: '
      f'{dict(zip(CLASSES, np.bincount(y_ref, minlength=K)))}')

# %%
# --- reference cross-validation on the current gene sets ---------------------
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

banner('REFERENCE CROSS-VALIDATION')

min_class = pd.Series(y_ref).value_counts().min()
n_splits = int(min(N_SPLITS, max(2, min_class)))
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
oof = cross_val_predict(
    LogisticRegression(max_iter=5000, C=1.0, class_weight='balanced',
                       random_state=SEED),
    X_ref, y_ref, cv=cv, method='predict_proba', n_jobs=-1)
oof_pred = oof.argmax(axis=1)
cv_acc = float((oof_pred == y_ref).mean())

print(f'  Reference cells: {len(y_ref)}   folds: {n_splits}')
print(f'  Out-of-fold accuracy: {cv_acc:.3f}   (chance {1/K:.3f})')
print('\n' + classification_report(y_ref, oof_pred, target_names=CLASSES,
                                   digits=3, zero_division=0))

cm = pd.DataFrame(confusion_matrix(y_ref, oof_pred,
                                   labels=np.arange(len(CLASSES))),
                  index=CLASSES, columns=CLASSES)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
sns.heatmap(cm.div(cm.sum(axis=1), axis=0), annot=True, fmt='.2f',
            cmap='Blues', vmin=0, vmax=1, ax=ax,
            cbar_kws={'label': 'Fraction of true class'})
ax.set_xlabel('Predicted'); ax.set_ylabel('Reference label')
ax.set_title(f'Reference {n_splits}-fold CV (accuracy {cv_acc:.3f})',
             fontweight='bold')
fig.tight_layout()
save_fig(fig, 'step5b_reference_cv.pdf')

if cv_acc < 0.60:
    print('\n  The gene sets do not separate these types even in the reference.')
    print('  Nothing downstream is interpretable until they are revised.')
else:
    print(f'\n  The gene sets work in the reference ({cv_acc:.3f}). The failure')
    print('  on sciMeDseq is therefore specific to that assay.')
    
# %%
# =============================================================================
# 5B.2 - FIT THE MIXTURE
# =============================================================================
banner(f'STEP 5B.2 - GMM ({GMM_MODE}, {GMM_COVARIANCE} covariance)')


def class_gaussians(X, y, k, reg):
    """Per-class mean and covariance from labelled data."""
    mus, covs = [], []
    for c in range(k):
        Xi = X[y == c]
        if len(Xi) < 2:
            raise ValueError(f'Class {CLASSES[c]} has {len(Xi)} reference cells '
                             f'- cannot estimate a covariance.')
        mus.append(Xi.mean(axis=0))
        covs.append(np.cov(Xi, rowvar=False) + reg * np.eye(X.shape[1]))
    return np.array(mus), np.array(covs)


if GMM_MODE == 'reference_anchored':
    mus, covs = class_gaussians(X_ref, y_ref, K, GMM_REG_COVAR)
    if GMM_COVARIANCE == 'tied':
        pooled = np.average(covs, axis=0,
                            weights=np.bincount(y_ref, minlength=K))
        covs = np.array([pooled] * K)
    elif GMM_COVARIANCE == 'diag':
        covs = np.array([np.diag(np.diag(c)) for c in covs])

    # EM on the query with only the mixture weights free
    logpdf = np.column_stack([
        multivariate_normal(mean=mus[c], cov=covs[c], allow_singular=True)
        .logpdf(X_qry) for c in range(K)
    ])
    w = np.full(K, 1.0 / K)
    for it in range(GMM_MAX_ITER):
        lw = np.log(np.maximum(w, 1e-300))[None, :] + logpdf
        lw -= lw.max(axis=1, keepdims=True)
        R = np.exp(lw)
        R /= R.sum(axis=1, keepdims=True)
        w_new = R.mean(axis=0)
        if np.max(np.abs(w_new - w)) < 1e-8:
            break
        w = w_new
    post_q = R
    print(f'  EM converged in {it + 1} iterations')
    print('  Fitted mixture weights on the query:')
    for c, ct in enumerate(CLASSES):
        print(f'    {ct:>12}: {w[c]:.3f}')

    # reference densities under the same mixture, for the outlier cut
    logpdf_ref = np.column_stack([
        multivariate_normal(mean=mus[c], cov=covs[c], allow_singular=True)
        .logpdf(X_ref) for c in range(K)
    ])
    from scipy.special import logsumexp
    lo_ref = logsumexp(np.log(np.maximum(w, 1e-300))[None, :] + logpdf_ref, axis=1)
    lo_qry = logsumexp(np.log(np.maximum(w, 1e-300))[None, :] + logpdf, axis=1)

elif GMM_MODE == 'unsupervised':
    gmm = GaussianMixture(
        n_components=K, covariance_type=GMM_COVARIANCE,
        reg_covar=GMM_REG_COVAR, n_init=GMM_N_INIT,
        max_iter=GMM_MAX_ITER, random_state=SEED,
    ).fit(X_qry)
    raw_post = gmm.predict_proba(X_qry)
    print(f'  Converged: {gmm.converged_}   BIC: {gmm.bic(X_qry):.1f}')

    # bijective component -> class mapping: minimise distance between component
    # means and reference class centroids
    ref_centroids = np.vstack([X_ref[y_ref == c].mean(axis=0) for c in range(K)])
    cost = np.linalg.norm(gmm.means_[:, None, :] - ref_centroids[None, :, :],
                          axis=2)
    comp_idx, class_idx = linear_sum_assignment(cost)
    mapping = dict(zip(comp_idx, class_idx))
    print('  Component -> class (assignment problem, one-to-one):')
    for comp in range(K):
        print(f'    component {comp} -> {CLASSES[mapping[comp]]:<12} '
              f'(distance {cost[comp, mapping[comp]]:.2f})')
    assert len(set(mapping.values())) == K, 'mapping is not a bijection'

    post_q = np.zeros_like(raw_post)
    for comp, cls in mapping.items():
        post_q[:, cls] = raw_post[:, comp]

    lo_qry = gmm.score_samples(X_qry)
    lo_ref = gmm.score_samples(X_ref)

else:
    raise ValueError("GMM_MODE must be 'reference_anchored' or 'unsupervised'")

pred_q = post_q.argmax(axis=1)
max_post = post_q.max(axis=1)
local_fdr = 1.0 - max_post


# %%
# =============================================================================
# 5B.3 - BAYESIAN FDR
# =============================================================================
banner(f'STEP 5B.3 - BAYESIAN FDR AT {BAYES_FDR:.0%}')

order = np.argsort(local_fdr)              # most confident first
running = np.cumsum(local_fdr[order]) / np.arange(1, len(local_fdr) + 1)
n_pass = int(np.sum(running <= BAYES_FDR))

passes = np.zeros(len(local_fdr), dtype=bool)
if n_pass > 0:
    passes[order[:n_pass]] = True
    post_cut = float(max_post[order[n_pass - 1]])
    achieved = float(running[n_pass - 1])
else:
    post_cut, achieved = np.nan, np.nan

print(f'  Cells passing global Bayesian FDR <= {BAYES_FDR:.0%}: '
      f'{n_pass}/{len(local_fdr)} ({100 * n_pass / len(local_fdr):.1f}%)')
if n_pass > 0:
    print(f'  Achieved FDR in that set: {achieved:.4f}')
    print(f'  Implied max-posterior cutoff: {post_cut:.4f}')
    print(f'  Expected false calls among them: ~{achieved * n_pass:.1f}')
else:
    print('  No cell reaches the target. The mixture is not separating these')
    print('  classes on the sciMeDseq scores.')
    print(f'  Best local fdr: {local_fdr.min():.3f} '
          f'(max posterior {max_post.max():.3f})')

# per-class FDR, controlled within each class separately
passes_within = np.zeros(len(local_fdr), dtype=bool)
per_class = []
for c in range(K):
    idx = np.where(pred_q == c)[0]
    if len(idx) == 0:
        per_class.append({'class': CLASSES[c], 'n_assigned': 0,
                          'n_pass_within_class': 0, 'achieved_fdr': np.nan})
        continue
    o = idx[np.argsort(local_fdr[idx])]
    run = np.cumsum(local_fdr[o]) / np.arange(1, len(o) + 1)
    n_c = int(np.sum(run <= BAYES_FDR))
    if n_c:
        passes_within[o[:n_c]] = True
    per_class.append({
        'class': CLASSES[c],
        'n_assigned': len(idx),
        'n_pass_within_class': n_c,
        'achieved_fdr': float(run[n_c - 1]) if n_c else np.nan,
        'median_local_fdr': float(np.median(local_fdr[idx])),
    })

per_class_df = pd.DataFrame(per_class)
print('\n  Per-class control:')
print(per_class_df.to_string(index=False))

# density-based outlier test
lo_cut = float(np.percentile(lo_ref, GMM_OUTLIER_PCT))
in_ref = lo_qry >= lo_cut
print(f'\n  Log-density cutoff (reference {GMM_OUTLIER_PCT}th pct): {lo_cut:.2f}')
print(f'  Query cells inside the reference density: '
      f'{int(in_ref.sum())}/{len(in_ref)} ({100 * in_ref.mean():.1f}%)')

call = np.where(~in_ref, 'not_in_reference',
                np.where(passes, np.array(CLASSES)[pred_q], 'unassigned'))

res = pd.DataFrame(index=ad_q.obs_names)
res['gmm_pred_label'] = np.array(CLASSES)[pred_q]
res['gmm_max_posterior'] = max_post
res['gmm_local_fdr'] = local_fdr
res['gmm_pass_global_fdr'] = passes
res['gmm_pass_within_class_fdr'] = passes_within
res['gmm_log_density'] = lo_qry
res['gmm_in_reference'] = in_ref
res['gmm_call'] = pd.Categorical(
    call, categories=CLASSES + ['unassigned', 'not_in_reference'])
for c, ct in enumerate(CLASSES):
    res[f'gmm_post_{ct}'] = post_q[:, c]
for ct in CLASSES:
    res[f'{ct}_score_scaled'] = S_qry[ct].values

print('\n  Final calls:')
print(res['gmm_call'].value_counts().to_string())

res.to_csv(os.path.join(TAB_DIR, 'gmm_fdr_classification.csv'))
per_class_df.to_csv(os.path.join(TAB_DIR, 'gmm_fdr_summary.csv'), index=False)

# --- how decisive is each call? ----------------------------------------------
MIN_MARGIN = 0.20   # top posterior must exceed the runner-up by this much
srt = np.sort(post_q, axis=1)
second = srt[:, -2]
margin = max_post - second
ratio = max_post / np.maximum(second, 1e-12)

res['gmm_second_posterior'] = second
res['gmm_margin'] = margin
res['gmm_odds_ratio'] = ratio
res['gmm_runner_up'] = np.array(CLASSES)[np.argsort(post_q, axis=1)[:, -2]]

print('\n  Decisiveness of the calls:')
print(f'    median margin (top - second): {np.median(margin):.3f}')
print(f'    cells with margin < 0.10:     {int((margin < 0.10).sum())} '
      f'({100 * (margin < 0.10).mean():.1f}%)')
print(f'    cells with second > 0.30:     {int((second > 0.30).sum())} '
      f'({100 * (second > 0.30).mean():.1f}%)')
print(f'    median top:second odds:       {np.median(ratio):.1f}x')

amb = margin < 0.10
if amb.sum():
    pairs = pd.Series([
        ' / '.join(sorted([a, b])) for a, b in
        zip(np.array(CLASSES)[pred_q][amb], res['gmm_runner_up'].values[amb])
    ]).value_counts()
    print('\n  Most contested pairs (margin < 0.10):')
    print(pairs.head(8).to_string())
# %%
# =============================================================================
# 5B.4 - PLOTS
# =============================================================================
banner('STEP 5B.4 - PLOTS')

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

ax = axes[0, 0]
ax.plot(np.arange(1, len(running) + 1), running, lw=1.5, color='#4169E1')
ax.axhline(BAYES_FDR, color='#DC143C', ls='--', lw=1.5,
           label=f'target {BAYES_FDR:.0%}')
if n_pass > 0:
    ax.axvline(n_pass, color='k', ls=':', lw=1.2, label=f'{n_pass} cells pass')
ax.set_xlabel('Cells called, ordered by confidence')
ax.set_ylabel('Running Bayesian FDR')
ax.set_title('FDR as calls are added', fontweight='bold')
ax.legend(fontsize=8)

ax = axes[0, 1]
ax.hist(max_post, bins=40, color='#7B2D8E', alpha=0.8)
if n_pass > 0:
    ax.axvline(post_cut, color='#DC143C', ls='--', lw=1.5,
               label=f'cutoff {post_cut:.3f}')
    ax.legend(fontsize=8)
ax.axvline(1.0 / K, color='k', ls=':', lw=1, label=f'chance {1/K:.2f}')
ax.set_xlabel('Max posterior'); ax.set_ylabel('Cells')
ax.set_title('Posterior confidence', fontweight='bold')

ax = axes[1, 0]
ax.hist(lo_ref, bins=50, density=True, alpha=0.6, color='#4169E1',
        label='reference')
ax.hist(lo_qry, bins=50, density=True, alpha=0.6, color='#DC143C',
        label='sciMeDseq')
ax.axvline(lo_cut, color='k', ls='--', lw=1.5,
           label=f'{GMM_OUTLIER_PCT}th pct')
ax.set_xlabel('GMM log-density'); ax.set_ylabel('Density')
ax.set_title('Fit to the reference mixture', fontweight='bold')
ax.legend(fontsize=8)

ax = axes[1, 1]
counts = res['gmm_call'].value_counts().reindex(
    CLASSES + ['unassigned', 'not_in_reference']).fillna(0)
cols = [CT_COLORS.get(c, '#777777') for c in counts.index]
ax.bar(np.arange(len(counts)), counts.values, color=cols)
ax.set_xticks(np.arange(len(counts)))
ax.set_xticklabels(counts.index, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Cells')
ax.set_title(f'Calls at FDR <= {BAYES_FDR:.0%}', fontweight='bold')
for i, v in enumerate(counts.values):
    ax.text(i, v, f'{int(v)}', ha='center', va='bottom', fontsize=8)

for ax in axes.ravel():
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
fig.tight_layout()
save_fig(fig, 'step5b_gmm_fdr.pdf')

# --- score space, first two modules, reference vs query ----------------------
if K >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    for c, ct in enumerate(CLASSES):
        m = y_ref == c
        axes[0].scatter(X_ref[m, 0], X_ref[m, 1], s=4, alpha=0.4,
                        color=CT_COLORS.get(ct, '#777777'), label=ct,
                        rasterized=True)
    axes[0].legend(fontsize=7, markerscale=2, frameon=False)
    axes[0].set_title('Reference, by true label', fontweight='bold')

    for ct in [c for c in CLASSES if c in set(res['gmm_call'])]:
        m = (res['gmm_call'] == ct).values
        axes[1].scatter(X_qry[m, 0], X_qry[m, 1], s=22, alpha=0.85,
                        color=CT_COLORS.get(ct, '#777777'), label=ct,
                        edgecolors='white', linewidths=0.3, rasterized=True)
    m = res['gmm_call'].isin(['unassigned', 'not_in_reference']).values
    if m.sum():
        axes[1].scatter(X_qry[m, 0], X_qry[m, 1], s=14, alpha=0.5,
                        color='#BBBBBB', label='unassigned / outlier',
                        rasterized=True)
    axes[1].legend(fontsize=7, markerscale=1.5, frameon=False)
    axes[1].set_title(f'sciMeDseq, GMM call at FDR <= {BAYES_FDR:.0%}',
                      fontweight='bold')

    for ax in axes:
        ax.set_xlabel(f'{CLASSES[0]} score (z)')
        ax.set_ylabel(f'{CLASSES[1]} score (z)')
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    fig.tight_layout()
    save_fig(fig, 'step5b_gmm_score_space.pdf')


# %%
# =============================================================================
# 5B.5 - COMPARE WITH THE INTEGRATION LABEL AND WITH STEP 5
# =============================================================================
banner('STEP 5B.5 - COMPARISONS')

if INTEG_COL in ad_q.obs.columns:
    integ = ad_q.obs[INTEG_COL].astype(str)
    xt = pd.crosstab(res['gmm_call'], integ)
    print(f'\n  gmm_call vs {INTEG_COL}:')
    print(xt.to_string())
    xt.to_csv(os.path.join(TAB_DIR, f'gmm_call_vs_{INTEG_COL}.csv'))

    ev = res['gmm_call'].isin(CLASSES).values & integ.isin(CLASSES).values
    if ev.sum():
        match = res['gmm_call'].astype(str).values == integ.values
        print(f'\n  Evaluable: {int(ev.sum())}/{len(res)}   '
              f'Agreement: {int((ev & match).sum())}/{int(ev.sum())} '
              f'({100 * (ev & match).sum() / ev.sum():.1f}%)')

lr_path = os.path.join(TAB_DIR, 'sciMeDseq_reference_classification.csv')
if os.path.exists(lr_path):
    lr = pd.read_csv(lr_path, index_col=0)
    common = res.index.intersection(lr.index)
    if len(common):
        both = pd.crosstab(res.loc[common, 'gmm_call'].astype(str),
                           lr.loc[common, 'ref_call'].astype(str))
        print('\n  GMM call vs logistic-regression call (step5):')
        print(both.to_string())
        agree = (res.loc[common, 'gmm_call'].astype(str).values ==
                 lr.loc[common, 'ref_call'].astype(str).values)
        print(f'  Identical calls: {int(agree.sum())}/{len(common)} '
              f'({100 * agree.mean():.1f}%)')
        print('  Two methods agreeing is weak evidence - they share the same')
        print('  module scores. Disagreement is the informative direction.')
else:
    print('\n  step5 output not found; skipping the classifier comparison.')

print(f'\n  Tables: {TAB_DIR}')
print(f'  Figures: {FIG_DIR}')
# %%
# =============================================================================
# 5B.6 - COMPOSITION BARS AND CONFIDENT / MATCHING CELLS
# =============================================================================
banner('STEP 5B.6 - COMPOSITION')

CATS = CLASSES + ['unassigned', 'not_in_reference']
BAR_COLORS = [CT_COLORS.get(c, '#777777') for c in CATS]

gmm_call = res['gmm_call'].astype(str)
has_integ = INTEG_COL in ad_q.obs.columns
integ = (ad_q.obs[INTEG_COL].astype(str) if has_integ
         else pd.Series('Unknown', index=res.index))
integ.index = res.index

# --- the groups each bar summarises -----------------------------------------
groups = {}
groups[f'GMM, all cells\n(n={len(res)})'] = gmm_call

fdr_mask = res['gmm_pass_global_fdr'].values & res['gmm_in_reference'].values
groups[f'GMM, FDR<={BAYES_FDR:.0%}\n(n={int(fdr_mask.sum())})'] = \
    gmm_call.where(fdr_mask)

if has_integ:
    groups[f'Integration label\n(n={int(integ.isin(CLASSES).sum())})'] = \
        integ.where(integ.isin(CLASSES))

    agree = (gmm_call.values == integ.values) & gmm_call.isin(CLASSES).values
    groups[f'GMM = integration\n(n={int(agree.sum())})'] = \
        gmm_call.where(agree)

    agree_fdr = agree & fdr_mask
    groups[f'GMM = integration\nand FDR<={BAYES_FDR:.0%}\n'
           f'(n={int(agree_fdr.sum())})'] = gmm_call.where(agree_fdr)

comp = pd.DataFrame(
    {name: s.value_counts().reindex(CATS).fillna(0).astype(int)
     for name, s in groups.items()}
).T[CATS]

print('\n  Counts per group:')
print(comp.to_string())
comp.to_csv(os.path.join(TAB_DIR, 'gmm_composition_counts.csv'))

pct = comp.div(comp.sum(axis=1).replace(0, np.nan), axis=0) * 100
print('\n  Percentages per group:')
print(pct.round(1).to_string())
pct.round(2).to_csv(os.path.join(TAB_DIR, 'gmm_composition_percent.csv'))


def stacked_bars(table, title, xlabel, fname, as_pct):
    """One horizontal stacked bar per group, coloured by cell type."""
    fig, ax = plt.subplots(figsize=(11, 0.85 * len(table) + 2.2))
    ypos = np.arange(len(table))
    left = np.zeros(len(table))
    for cat, colour in zip(CATS, BAR_COLORS):
        vals = table[cat].values.astype(float)
        ax.barh(ypos, vals, left=left, color=colour, edgecolor='white',
                linewidth=0.6, label=cat, height=0.62)
        for i, (v, l) in enumerate(zip(vals, left)):
            # only annotate segments wide enough to hold the text
            span = table.sum(axis=1).max() if not as_pct else 100.0
            if v > 0.045 * span:
                txt = f'{v:.0f}%' if as_pct else f'{int(v)}'
                ax.text(l + v / 2, ypos[i], txt, ha='center', va='center',
                        fontsize=7.5, color='white', fontweight='bold')
        left += vals

    ax.set_yticks(ypos)
    ax.set_yticklabels(table.index, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    ax.set_title(title, fontweight='bold')
    if as_pct:
        ax.set_xlim(0, 100)
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8,
              frameon=False, title='cell type', title_fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    fig.tight_layout()
    save_fig(fig, fname)


stacked_bars(comp, 'sciMeDseq composition, cell counts', 'Cells',
             'step5b_composition_counts.pdf', as_pct=False)
stacked_bars(pct.fillna(0), 'sciMeDseq composition, percentage',
             'Percent of the group', 'step5b_composition_percent.pdf',
             as_pct=True)


# %%
# --- where the confident and matching cells sit ------------------------------
# --- where the confident and matching cells sit ------------------------------
if 'X_umap' in adata.obsm:
    umap_all = adata.obsm['X_umap']
    bg = umap_all[is_r]                    # reference cells
    coords = umap_all[is_q]                # query cells, same embedding
    coord_lab = ('UMAP1', 'UMAP2')
    bg_lab = f'reference (n={int(is_r.sum())})'
else:
    bg = None
    coords = X_qry[:, :2]
    coord_lab = (f'{CLASSES[0]} score (z)', f'{CLASSES[1]} score (z)')
    bg_lab = None
    print('  No UMAP on the object; using the first two module scores.')

panels = [
    ('All sciMeDseq, GMM call', np.ones(len(res), dtype=bool), gmm_call.values),
    (f'Passing FDR <= {BAYES_FDR:.0%}  (n={int(fdr_mask.sum())})',
     fdr_mask, gmm_call.values),
]
if has_integ:
    panels.append((f'GMM = integration  (n={int(agree.sum())})',
                   agree, gmm_call.values))
    disagree = (gmm_call.isin(CLASSES).values & integ.isin(CLASSES).values
                & ~agree)
    panels.append((f'GMM != integration  (n={int(disagree.sum())})',
                   disagree, gmm_call.values))

ncol = 2
nrow = int(np.ceil(len(panels) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(6.5 * ncol, 5.5 * nrow))
axes = np.atleast_1d(axes).ravel()

for ax, (title, mask, lab) in zip(axes, panels):
    if bg is not None:
        ax.scatter(bg[:, 0], bg[:, 1], s=2, color='#E4E4E4', alpha=0.6,
                   rasterized=True, label=bg_lab, zorder=1)
    # query cells not in this panel's selection, faint
    ax.scatter(coords[~mask, 0], coords[~mask, 1], s=9, color='#9E9E9E',
               alpha=0.45, rasterized=True, zorder=2)
    for cat in CATS:
        m = mask & (lab == cat)
        if m.sum():
            ax.scatter(coords[m, 0], coords[m, 1], s=28, alpha=0.95,
                       color=CT_COLORS.get(cat, '#777777'),
                       edgecolors='white', linewidths=0.4,
                       label=f'{cat} ({int(m.sum())})', rasterized=True, zorder=3)
    ax.legend(fontsize=7, markerscale=1.3, frameon=False, loc='best')
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlabel(coord_lab[0], fontsize=8)
    ax.set_ylabel(coord_lab[1], fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)

for ax in axes[len(panels):]:
    ax.axis('off')

fig.tight_layout()
save_fig(fig, 'step5b_confident_and_matching_cells.pdf')

# --- per-class agreement, for the record -------------------------------------
if has_integ:
    rows = []
    for ct in CLASSES:
        in_integ = (integ == ct).values
        n_gmm = int((gmm_call == ct).sum())
        n_agree = int((agree & (gmm_call == ct).values).sum())
        n_agree_fdr = int((agree_fdr & (gmm_call == ct).values).sum())
        rows.append({
            'cell_type': ct,
            'n_integration': int(in_integ.sum()),
            'n_gmm': n_gmm,
            'n_agree': n_agree,
            'pct_of_gmm_calls': round(100 * n_agree / n_gmm, 1) if n_gmm else np.nan,
            'pct_of_integration': (round(100 * n_agree / in_integ.sum(), 1)
                                   if in_integ.sum() else np.nan),
            'n_agree_and_fdr': n_agree_fdr,
        })
    agree_df = pd.DataFrame(rows)
    print('\n  Per-class agreement with the integration label:')
    print(agree_df.to_string(index=False))
    agree_df.to_csv(os.path.join(TAB_DIR, 'gmm_integration_agreement.csv'),
                    index=False)

print(f'\n  Figures: {FIG_DIR}')


# %%
# =============================================================================
# 5B.8 - POSTERIOR HEATMAP, ONE ROW PER CELL
# =============================================================================
banner('STEP 5B.8 - POSTERIOR HEATMAP')

P = pd.DataFrame(post_q, index=res.index, columns=CLASSES)

# order cells by assigned class, then by confidence within class
ordering = (pd.DataFrame({'cls': pred_q, 'conf': max_post})
            .sort_values(['cls', 'conf'], ascending=[True, False]).index)
P_ord = P.iloc[ordering]
call_ord = res['gmm_call'].astype(str).values[ordering]
lab_ord = np.array(CLASSES)[pred_q][ordering]
fdr_ord = res['gmm_pass_global_fdr'].values[ordering]
integ_ord = integ.values[ordering] if has_integ else None

# row boundaries between assigned classes
bounds, labels_at = [], []
prev, start = lab_ord[0], 0
for i, v in enumerate(lab_ord):
    if v != prev:
        bounds.append(i)
        labels_at.append((prev, (start + i) / 2))
        prev, start = v, i
labels_at.append((prev, (start + len(lab_ord)) / 2))

# --- annotation strips: assigned class, FDR pass, integration label ----------
def strip(values, palette, default='#EEEEEE'):
    return np.array([[*plt.matplotlib.colors.to_rgb(palette.get(v, default))]
                     for v in values])[:, None, :]

n_strips = 2 + (1 if has_integ else 0)
widths = [1.0] * n_strips + [len(CLASSES) * 1.8]
fig, axes = plt.subplots(
    1, n_strips + 1, figsize=(3.2 + 1.1 * len(CLASSES), 9),
    gridspec_kw={'width_ratios': widths, 'wspace': 0.06})

k = 0
axes[k].imshow(strip(lab_ord, CT_COLORS), aspect='auto', interpolation='nearest')
axes[k].set_title('GMM\ncall', fontsize=8, fontweight='bold')
k += 1

axes[k].imshow(strip(np.where(fdr_ord, 'pass', 'fail'),
                     {'pass': '#2E8B57', 'fail': '#DDDDDD'}),
               aspect='auto', interpolation='nearest')
axes[k].set_title(f'FDR\n<={BAYES_FDR:.0%}', fontsize=8, fontweight='bold')
k += 1

if has_integ:
    axes[k].imshow(strip(integ_ord, CT_COLORS), aspect='auto',
                   interpolation='nearest')
    axes[k].set_title('integ.\nlabel', fontsize=8, fontweight='bold')
    k += 1

for ax in axes[:k]:
    ax.set_xticks([]); ax.set_yticks([])
    for b in bounds:
        ax.axhline(b, color='white', lw=0.8)

hm = axes[k]
im = hm.imshow(P_ord.values, aspect='auto', cmap='magma', vmin=0, vmax=1,
               interpolation='nearest')
hm.set_xticks(np.arange(len(CLASSES)))
hm.set_xticklabels(CLASSES, rotation=45, ha='right', fontsize=9)
hm.set_yticks([pos for _, pos in labels_at])
hm.set_yticklabels([f'{lab}  (n={int((lab_ord == lab).sum())})'
                    for lab, _ in labels_at], fontsize=8)
hm.set_ylabel('sciMeDseq cells, grouped by GMM call', fontsize=9)
hm.set_title(f'Posterior per cell  (n={len(P_ord)}, {GMM_MODE})',
             fontweight='bold')
for b in bounds:
    hm.axhline(b, color='white', lw=1.0)
plt.colorbar(im, ax=hm, label='posterior', fraction=0.035, pad=0.02)

fig.tight_layout()
save_fig(fig, 'step5b_posterior_heatmap.pdf')

# --- the ambiguous cells on their own, since they vanish above ---------------
amb_idx = np.where(margin < 0.30)[0]
if len(amb_idx):
    order_amb = amb_idx[np.argsort(-max_post[amb_idx])]
    Pa = P.iloc[order_amb]
    h = max(3.0, 0.22 * len(order_amb))
    fig, ax = plt.subplots(figsize=(2.0 + 1.0 * len(CLASSES), h))
    im = ax.imshow(Pa.values, aspect='auto', cmap='magma', vmin=0, vmax=1,
                   interpolation='nearest')
    for i in range(Pa.shape[0]):
        for j in range(Pa.shape[1]):
            v = Pa.values[i, j]
            if v > 0.08:
                ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=6,
                        color='white' if v < 0.6 else 'black')
    ax.set_xticks(np.arange(len(CLASSES)))
    ax.set_xticklabels(CLASSES, rotation=45, ha='right', fontsize=8)
    lbl = [f'{c[:14]}  [{integ.values[i]}]' if has_integ else c[:18]
           for c, i in zip(Pa.index, order_amb)]
    ax.set_yticks(np.arange(len(order_amb)))
    ax.set_yticklabels(lbl, fontsize=6)
    ax.set_title(f'Contested cells, margin < 0.30 (n={len(order_amb)})'
                 + ('   [integration label in brackets]' if has_integ else ''),
                 fontweight='bold', fontsize=9)
    plt.colorbar(im, ax=ax, label='posterior', fraction=0.04, pad=0.02)
    fig.tight_layout()
    save_fig(fig, 'step5b_posterior_heatmap_contested.pdf')
    print(f'  Contested cells (margin < 0.30): {len(order_amb)}')
else:
    print('  No cells with margin < 0.30.')

P.to_csv(os.path.join(TAB_DIR, 'gmm_posteriors_per_cell.csv'))
print(f'  Saved gmm_posteriors_per_cell.csv and the heatmaps to {FIG_DIR}')

    # %%
# =============================================================================
# 5B.9 - CELL CYCLE GENES: MATRIXPLOT
# =============================================================================
banner('STEP 5B.9c - CELL CYCLE')

"""
Checks whether any GMM-called module is confounded with proliferation.
Stem/TA are expected to cycle; Enterocyte/Paneth/Goblet/Tuft/EEC should be
largely post-mitotic. Uses the standard Tirosh et al. 2015 / Seurat cc.genes
S- and G2M-phase lists, cased for mouse (mm10/GRCm38) gene symbols.
"""

S_GENES_MOUSE = [
    'Mcm5', 'Pcna', 'Tyms', 'Fen1', 'Mcm2', 'Mcm4', 'Rrm1', 'Ung', 'Gins2',
    'Mcm6', 'Cdca7', 'Dtl', 'Prim1', 'Uhrf1', 'Hells', 'Rfc2', 'Rpa2', 'Nasp',
    'Rad51ap1', 'Gmnn', 'Wdr76', 'Slbp', 'Ccne2', 'Ubr7', 'Pold3', 'Msh2',
    'Atad2', 'Rad51', 'Rrm2', 'Cdc45', 'Cdc6', 'Exo1', 'Tipin', 'Dscc1',
    'Blm', 'Casp8ap2', 'Usp1', 'Clspn', 'Pola1', 'Chaf1b', 'Brip1', 'E2f8',
]
G2M_GENES_MOUSE = [
    'Hmgb2', 'Cdk1', 'Nusap1', 'Ube2c', 'Birc5', 'Tpx2', 'Top2a', 'Ndc80',
    'Cks2', 'Nuf2', 'Cks1b', 'Mki67', 'Tmpo', 'Cenpf', 'Tacc3', 'Smc4',
    'Ccnb2', 'Ckap2l', 'Ckap2', 'Aurkb', 'Bub1', 'Kif11', 'Anp32e', 'Tubb4b',
    'Gtse1', 'Kif20b', 'Hjurp', 'Cdca3', 'Cdc20', 'Ttk', 'Cdc25c', 'Kif2c',
    'Rangap1', 'Ncapd2', 'Dlgap5', 'Cdca2', 'Cdca8', 'Ect2', 'Kif23', 'Hmmr',
    'Aurka', 'Psrc1', 'Anln', 'Lbr', 'Ckap5', 'Cenpe', 'Ctcf', 'Nek2', 'G2e3',
    'Gas2l3', 'Cbx5', 'Cenpa',
]

s_genes = [g for g in S_GENES_MOUSE if g in ad_q.var_names]
g2m_genes = [g for g in G2M_GENES_MOUSE if g in ad_q.var_names]
missing_s = sorted(set(S_GENES_MOUSE) - set(s_genes))
missing_g2m = sorted(set(G2M_GENES_MOUSE) - set(g2m_genes))
print(f'  S-phase genes found: {len(s_genes)}/{len(S_GENES_MOUSE)} '
      f'(missing: {missing_s[:8]}{"..." if len(missing_s) > 8 else ""})')
print(f'  G2M-phase genes found: {len(g2m_genes)}/{len(G2M_GENES_MOUSE)} '
      f'(missing: {missing_g2m[:8]}{"..." if len(missing_g2m) > 8 else ""})')
if len(s_genes) < 3 or len(g2m_genes) < 3:
    raise ValueError('Too few cell-cycle genes present in ad_q.var_names to '
                     'score cell cycle reliably.')

ad_cc = ad_q.copy()
ad_cc.X = ad_cc.layers['lognorm'].copy()
sc.tl.score_genes_cell_cycle(ad_cc, s_genes=s_genes, g2m_genes=g2m_genes,
                             use_raw=False)

print('\n  Phase composition, all query cells:')
print(ad_cc.obs['phase'].value_counts().to_string())

# --- attach to the res/long tables already in scope --------------------------
res['cc_phase'] = ad_cc.obs['phase'].values
res['cc_S_score'] = ad_cc.obs['S_score'].values
res['cc_G2M_score'] = ad_cc.obs['G2M_score'].values
res[['cc_phase', 'cc_S_score', 'cc_G2M_score']].to_csv(
    os.path.join(TAB_DIR, 'cell_cycle_scores.csv'))

ad_cc.obs['gmm_call'] = res['gmm_call'].astype(str).values
if has_integ:
    ad_cc.obs['integration'] = integ.values
    gmm_eq_integ_lab = np.where(long['agrees'].values,
                               res['gmm_call'].astype(str).values, 'no_match')
    ad_cc.obs['gmm_eq_integ'] = pd.Categorical(
        gmm_eq_integ_lab,
        categories=[c for c in CLASSES if (gmm_eq_integ_lab == c).any()] + ['no_match'])
    res['gmm_eq_integ'] = gmm_eq_integ_lab

# %%
# --- matrixplot of the raw cycling genes, same style as 5B.9b ---------------
banner('5B.9c.1 - CELL CYCLE MATRIXPLOT')

cc_var_groups = {'S phase': s_genes, 'G2M phase': g2m_genes}
groupings_cc = [('gmm_call', 'GMM call')]
if has_integ:
    groupings_cc.append(('integration', 'integration label'))
    groupings_cc.append(('gmm_eq_integ', 'GMM = integration'))

for key, title in groupings_cc:
    ad_sub_cc = (ad_cc[ad_cc.obs[key] != 'no_match'].copy()
                if key == 'gmm_eq_integ' else ad_cc)
    if ad_sub_cc.obs[key].nunique() < 2:
        print(f'  skipping {title}: <2 groups')
        continue
    for ext in ('pdf', 'png'):
        sc.pl.matrixplot(
            ad_sub_cc, cc_var_groups, groupby=key, use_raw=False,
            standard_scale='var', cmap='magma',
            colorbar_title='scaled mean\nexpression',
            title=f'Cell cycle genes by {title}',
            save=f'_step5b_cellcycle_matrix_{key}.{ext}', show=False,
        )
print(f'  Matrixplots saved to {FIG_DIR}')

# %%
# --- S/G2M score boxplots by GMM call ----------------------------------------
banner('5B.9c.2 - CELL CYCLE SCORE BOXPLOTS BY GMM CALL')

order = [c for c in CLASSES if (res['gmm_call'] == c).any()]
order += [c for c in ['unassigned', 'not_in_reference']
          if (res['gmm_call'].astype(str) == c).any()]

fig, axes = plt.subplots(1, 2, figsize=(5.4 * 2, 4.6), sharex=True)
for ax, score_col, score_lab in zip(
        axes, ['cc_S_score', 'cc_G2M_score'], ['S-phase score', 'G2M-phase score']):
    pal = [CT_COLORS.get(g, '#777777') for g in order]
    sns.boxplot(data=res.reset_index(), x='gmm_call', y=score_col, order=order,
                hue='gmm_call', hue_order=order, legend=False,
                palette=pal, fliersize=1.5, linewidth=0.9, ax=ax)
    ax.axhline(0, color='k', ls=':', lw=0.8)
    for i, grp in enumerate(order):
        n = int((res['gmm_call'].astype(str) == grp).sum())
        ax.text(i, ax.get_ylim()[0], f'{n}', ha='center', va='bottom',
                fontsize=6.5, color='#555555')
    ax.set_title(score_lab, fontweight='bold')
    ax.set_xlabel(''); ax.set_ylabel(score_col)
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    for lbl in ax.get_xticklabels():
        lbl.set_ha('right')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
fig.suptitle('Cell cycle scores by GMM call', fontweight='bold', y=1.03)
fig.tight_layout()
save_fig(fig, 'step5b_cellcycle_scores_by_gmm_call.pdf')


# %%
# =============================================================================
# 5B.9d - UMAP: S/G2M SCORES AND GMM = INTEGRATION MARKER GENES
# =============================================================================
banner('STEP 5B.9d - S/G2M AND MARKER GENE UMAPS')

if 'ad_cc' not in globals():
    raise RuntimeError('ad_cc not found - run 5B.9c (cell cycle scoring) first.')
if 'ad_plot' not in globals():
    raise RuntimeError('ad_plot not found - run 5B.9b first.')

DIM_PLOT = globals().get('DIM_PLOT', 20)
_ad = sc.read_h5ad(H5_INTEGRATED)
emb_key = f'X_umap_dim{DIM_PLOT}'
if emb_key not in _ad.obsm:
    raise KeyError(f'{emb_key} not in obsm. Rerun step3_integrate.py with '
                   f'{DIM_PLOT} in DIMENSIONS.')
is_q_all = (_ad.obs['dataset'] == 'sciMeDseq').values
q_names = _ad.obs_names[is_q_all]
q_order = pd.Index(q_names).get_indexer(res.index)
if (q_order < 0).any():
    raise ValueError('Query cells in res are missing from the h5ad.')
U_q = _ad.obsm[emb_key][is_q_all][q_order]   # aligned to res.index order
del _ad

# --- S and G2M score UMAPs, diverging around 0 (same convention as the -----
#     component-mean heatmaps elsewhere, since these are signed scores) -----
fig, axes = plt.subplots(1, 2, figsize=(13, 5.6))
for ax, score_col, title in zip(axes, ['cc_S_score', 'cc_G2M_score'],
                                ['S-phase score', 'G2M-phase score']):
    vals = res[score_col].values
    lim = float(np.abs(vals).max())
    sca = ax.scatter(U_q[:, 0], U_q[:, 1], s=10, c=vals, cmap='RdBu_r',
                     vmin=-lim, vmax=lim, linewidths=0, rasterized=True)
    plt.colorbar(sca, ax=ax, label=score_col, fraction=0.04, pad=0.02)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
fig.suptitle(f'Cell cycle scores, dim-{DIM_PLOT} UMAP', fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, f'step5b_dim{DIM_PLOT}_cellcycle_umap.pdf')

# %%
# --- marker genes from the gmm_eq_integ matrixplot, one UMAP per gene -------
ad_sub_ge = ad_plot[ad_plot.obs['gmm_eq_integ'] != 'no_match'].copy()
var_groups_ge, sel_ge = select_genes(ad_sub_ge, 'gmm_eq_integ', 'GMM = integration')
if not var_groups_ge:
    raise ValueError('No genes survived selection for the gmm_eq_integ grouping.')

genes_ge = [g for kept in var_groups_ge.values() for g in kept]
gene_to_class = {g: ct for ct, kept in var_groups_ge.items() for g in kept}
print(f'  Plotting {len(genes_ge)} marker genes from the gmm_eq_integ matrixplot')

expr = ad_plot[ad_sub_ge.obs_names, genes_ge].X
expr = np.asarray(expr.todense()) if sp.issparse(expr) else np.asarray(expr)
U_ge = U_q[pd.Index(res.index).get_indexer(ad_sub_ge.obs_names)]

ncol = 4
nrow = int(np.ceil(len(genes_ge) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.8 * nrow))
axes = np.atleast_1d(axes).ravel()
for ax, gene, vals in zip(axes, genes_ge, expr.T):
    sca = ax.scatter(U_ge[:, 0], U_ge[:, 1], s=8, c=vals, cmap='RdBu_r',
                     linewidths=0, rasterized=True)
    plt.colorbar(sca, ax=ax, fraction=0.045, pad=0.02)
    ax.set_title(f'{gene}\n({gene_to_class[gene]})', fontsize=9,
                fontweight='bold', color=CT_COLORS.get(gene_to_class[gene],
                                                         '#333333'))
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
for ax in axes[len(genes_ge):]:
    ax.axis('off')
fig.suptitle(f'Marker gene expression, GMM = integration cells, '
             f'dim-{DIM_PLOT} UMAP', fontweight='bold', y=1.005)
fig.tight_layout()
save_fig(fig, f'step5b_dim{DIM_PLOT}_gmm_eq_integ_marker_umaps.pdf')

# %%
# --- redo the gmm_eq_integ matrixplot with blue-to-red gradient -------------
banner('5B.9b REDO - GMM = INTEGRATION MATRIXPLOT, BLUE TO RED')

for ext in ('pdf', 'png'):
    sc.pl.matrixplot(
        ad_sub_ge, var_groups_ge, groupby='gmm_eq_integ', use_raw=False,
        standard_scale='var', cmap='RdBu_r',
        colorbar_title='scaled mean\nexpression',
        title='Genes specific within GMM = integration',
        save=f'_step5b_matrix_gmm_eq_integ_redblue.{ext}', show=False,
    )

print(f'\n  Figures: {FIG_DIR}')


#### Fig.1E

In [ ]:
# %%
# =============================================================================
# 5B.10 - UMAP: ONLY sciMeDseq CELLS WHERE GMM MATCHES DIM-20 INTEGRATION
# =============================================================================
banner('STEP 5B.10 - GMM = DIM-20 INTEGRATION, ON UMAP')

DIM_PLOT = 20
_ad = sc.read_h5ad(H5_INTEGRATED)

emb_key = f'X_umap_dim{DIM_PLOT}'
type_col = f'leiden_annot_{DIM_PLOT}_type'
if emb_key not in _ad.obsm:
    raise KeyError(f'{emb_key} not in obsm. Rerun step3_integrate.py with '
                   f'{DIM_PLOT} in DIMENSIONS.')
if type_col not in _ad.obs.columns:
    raise KeyError(f'{type_col} not in obs. Rerun step3_integrate.py with '
                   f'{DIM_PLOT} in DIMENSIONS.')

U = _ad.obsm[emb_key]
is_r_all = (_ad.obs['dataset'] != 'sciMeDseq').values
is_q_all = ~is_r_all

# align query rows to res
q_names = _ad.obs_names[is_q_all]
q_order = pd.Index(q_names).get_indexer(res.index)
if (q_order < 0).any():
    raise ValueError('Query cells in res are missing from the h5ad.')

integ20 = _ad.obs.loc[res.index, type_col].astype(str)
gmm_call = res['gmm_call'].astype(str)
match20 = (gmm_call.values == integ20.values) & gmm_call.isin(CLASSES).values
print(f'  sciMeDseq cells where GMM = dim-{DIM_PLOT} integration: '
      f'{int(match20.sum())}/{len(res)} '
      f'({100 * match20.mean():.1f}%)')

U_q = U[is_q_all][q_order]   # query coords, aligned to res order

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(U[is_r_all, 0], U[is_r_all, 1], s=6, color='#B0B0B0', alpha=0.45,
           linewidths=0, rasterized=True, zorder=1,
           label=f'reference (n={int(is_r_all.sum())})')
ax.scatter(U_q[~match20, 0], U_q[~match20, 1], s=7, color='#DDDDDD', alpha=0.4,
           linewidths=0, rasterized=True, zorder=2,
           label=f'sciMeDseq, no match (n={int((~match20).sum())})')

for ct in CLASSES:
    m = match20 & (gmm_call.values == ct)
    if not m.sum():
        continue
    ax.scatter(U_q[m, 0], U_q[m, 1], s=45, marker='s',
               color=CT_COLORS.get(ct, '#000000'),
               alpha=0.95, edgecolors='white', linewidths=0.4,
               rasterized=True, zorder=3, label=f'{ct} (n={int(m.sum())})')

ax.legend(fontsize=8, frameon=False, markerscale=1.6, loc='best')
ax.set_title(f'sciMeDseq cells where GMM = dim-{DIM_PLOT} integration label',
             fontweight='bold')
ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')
ax.set_xticks([]); ax.set_yticks([])
for s in ax.spines.values():
    s.set_visible(False)
fig.tight_layout()
save_fig(fig, f'step5b_dim{DIM_PLOT}_gmm_match_highlight.pdf')

del _ad
print(f'\n  Figures: {FIG_DIR}')
# %%
# =============================================================================
# 5B.10b - UMAP: SECRETORY-LINEAGE sciMeDseq CELLS MATCHING DIM-20 INTEGRATION
# =============================================================================
banner('STEP 5B.10b - SECRETORY, GMM = DIM-20 INTEGRATION, ON UMAP')

SECRETORY = [c for c in ['Paneth', 'Goblet', 'Tuft', 'EEC'] if c in CLASSES]
print(f'  Secretory classes present: {SECRETORY}')
if not SECRETORY:
    raise ValueError('None of Paneth/Goblet/Tuft/EEC are in CLASSES - check '
                     'the secretory-lineage definition against this dataset.')

_ad = sc.read_h5ad(H5_INTEGRATED)
is_r_all = (_ad.obs['dataset'] != 'sciMeDseq').values
U = _ad.obsm[f'X_umap_dim{DIM_PLOT}']
del _ad

secretory_match = match20 & gmm_call.isin(SECRETORY).values
print(f'  Secretory cells matching dim-{DIM_PLOT} integration: '
      f'{int(secretory_match.sum())}/{int(match20.sum())} matching cells '
      f'({100 * secretory_match.sum() / max(match20.sum(), 1):.1f}%)')
for ct in SECRETORY:
    n = int((secretory_match & (gmm_call.values == ct)).sum())
    print(f'    {ct:>10}: {n}')

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(U[is_r_all, 0], U[is_r_all, 1], s=6, color='#B0B0B0', alpha=0.4,
           linewidths=0, rasterized=True, zorder=1,
           label=f'reference (n={int(is_r_all.sum())})')
ax.scatter(U_q[~secretory_match, 0], U_q[~secretory_match, 1], s=6,
           color='#E8E8E8', alpha=0.3, linewidths=0, rasterized=True, zorder=2,
           label='sciMeDseq, other/no match')

for ct in SECRETORY:
    m = secretory_match & (gmm_call.values == ct)
    if not m.sum():
        continue
    ax.scatter(U_q[m, 0], U_q[m, 1], s=45, marker='s',
               color=CT_COLORS.get(ct, '#000000'),
               alpha=0.95, edgecolors='white', linewidths=0.4,
               rasterized=True, zorder=3, label=f'{ct} (n={int(m.sum())})')
ax.legend(fontsize=8, frameon=False, markerscale=1.6, loc='best')
ax.set_title(f'Secretory-lineage sciMeDseq cells matching dim-{DIM_PLOT} '
             f'integration', fontweight='bold')
ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')
ax.set_xticks([]); ax.set_yticks([])
for s in ax.spines.values():
    s.set_visible(False)
fig.tight_layout()
save_fig(fig, f'step5b_dim{DIM_PLOT}_secretory_match_highlight.pdf')

print(f'\n  Figures: {FIG_DIR}')

#### Fig1C

In [ ]:
# %%
# =============================================================================
# 5B.8b - POSTERIOR HEATMAP, PAPER-READY (RdBu, GMM annotation only)
# =============================================================================
banner('STEP 5B.8b - POSTERIOR HEATMAP (paper-ready)')

P = pd.DataFrame(post_q, index=res.index, columns=CLASSES)

ordering = (pd.DataFrame({'cls': pred_q, 'conf': max_post})
            .sort_values(['cls', 'conf'], ascending=[True, False]).index)
P_ord = P.iloc[ordering]
lab_ord = np.array(CLASSES)[pred_q][ordering]

bounds, labels_at = [], []
prev, start = lab_ord[0], 0
for i, v in enumerate(lab_ord):
    if v != prev:
        bounds.append(i)
        labels_at.append((prev, (start + i) / 2))
        prev, start = v, i
labels_at.append((prev, (start + len(lab_ord)) / 2))


def strip(values, palette, default='#EEEEEE'):
    return np.array([[*mpl.colors.to_rgb(palette.get(v, default))]
                     for v in values])[:, None, :]


fig, axes = plt.subplots(
    1, 2, figsize=(2.0 + 1.1 * len(CLASSES), 9),
    gridspec_kw={'width_ratios': [1.0, len(CLASSES) * 1.8], 'wspace': 0.05})

strip_ax, hm = axes
strip_ax.imshow(strip(lab_ord, CT_COLORS), aspect='auto',
                interpolation='nearest')
strip_ax.set_title('GMM\ncall', fontsize=9, fontweight='bold')
strip_ax.set_xticks([]); strip_ax.set_yticks([])
for b in bounds:
    strip_ax.axhline(b, color='white', lw=1.0)

im = hm.imshow(P_ord.values, aspect='auto', cmap='RdBu_r', vmin=0, vmax=1,
               interpolation='nearest')
hm.set_xticks(np.arange(len(CLASSES)))
hm.set_xticklabels(CLASSES, rotation=45, ha='right', fontsize=10)
hm.set_yticks([pos for _, pos in labels_at])
hm.set_yticklabels([f'{lab}  (n={int((lab_ord == lab).sum())})'
                    for lab, _ in labels_at], fontsize=9)
hm.set_ylabel('sciMeDseq cells, grouped by GMM call', fontsize=10)
hm.set_title(f'Posterior per cell  (n={len(P_ord)}, {GMM_MODE})',
             fontweight='bold', fontsize=12)
for b in bounds:
    hm.axhline(b, color='white', lw=1.2)

cbar = plt.colorbar(im, ax=hm, label='posterior probability', fraction=0.035,
                    pad=0.02)
cbar.ax.tick_params(labelsize=9)

fig.tight_layout()
save_fig(fig, 'step5b_posterior_heatmap_paperready.pdf')
print(f'  Figures: {FIG_DIR}')

#### Fig 1E

In [ ]:
# %%
# =============================================================================
# 5B.11 - GMM = INTEGRATION COMPOSITION vs. FACS-SORTED ENTEROCYTE CONTROL
# =============================================================================
banner('STEP 5B.21b - COMPOSITION BAR WITH FACS CONTROL')

if not has_integ:
    raise RuntimeError('No integration label available (has_integ is False); '
                       'this plot needs the GMM = integration mask from 5B.9b.')
if 'agree' not in globals():
    raise RuntimeError('`agree` (GMM = integration mask) not found - run 5B.9b '
                       'first.')

# --- composition of the GMM = integration cells ------------------------------
gmm_call = res['gmm_call'].astype(str)
n_agree_by_class = gmm_call[agree].value_counts().reindex(CLASSES, fill_value=0)
n_agree_total = int(n_agree_by_class.sum())
pct_agree_by_class = (n_agree_by_class / n_agree_total * 100
                      if n_agree_total else n_agree_by_class.astype(float))

print(f'  GMM = integration cells: {n_agree_total}')
print(n_agree_by_class.to_string())

# --- FACS-sorted Enterocyte control: 100% Enterocyte, by construction -------
FACS_CT = 'Enterocyte'
if FACS_CT not in CLASSES:
    raise ValueError(f'{FACS_CT!r} not in CLASSES: {CLASSES}')
n_facs = n_agree_total  # bar height is percentage either way; count is informational
pct_facs = pd.Series({ct: (100.0 if ct == FACS_CT else 0.0) for ct in CLASSES})

# --- assemble the two-bar table ---------------------------------------------
bar_labels = [f'GMM = integration\n(n={n_agree_total})',
             f'FACS-sorted Enterocytes\n(control, 100% {FACS_CT})']
pct_table = pd.DataFrame([pct_agree_by_class, pct_facs], index=bar_labels)[CLASSES]

fig, ax = plt.subplots(figsize=(6.5, 6))
xpos = np.arange(len(pct_table))
bottom = np.zeros(len(pct_table))
for ct in CLASSES:
    vals = pct_table[ct].values.astype(float)
    n_lab = int(n_agree_by_class.get(ct, 0))
    ax.bar(xpos, vals, bottom=bottom, width=0.6, color=CT_COLORS.get(ct, '#777777'),
          edgecolor='white', linewidth=0.8,
          label=f'{ct} (n={n_lab})')
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 4:
            ax.text(xpos[i], b + v / 2, f'{v:.0f}%', ha='center', va='center',
                    fontsize=9, color='white', fontweight='bold')
    bottom += vals

ax.set_xticks(xpos)
ax.set_xticklabels(bar_labels, fontsize=9)
ax.set_ylabel('Percent of cells')
ax.set_ylim(0, 105)
ax.set_title('Cell type composition: GMM = integration vs. FACS control',
             fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8, frameon=False,
          title='cell type (n in GMM=integration)', title_fontsize=8)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
fig.tight_layout()
save_fig(fig, 'step5b_gmm_eq_integ_vs_facs_control_percent.pdf')

pct_table.to_csv(os.path.join(TAB_DIR, 'gmm_eq_integ_vs_facs_control_percent.csv'))
n_agree_by_class.to_frame('n').to_csv(
    os.path.join(TAB_DIR, 'gmm_eq_integ_composition_counts.csv'))

print(f'\n  Figures: {FIG_DIR}\n  Tables: {TAB_DIR}')

#### Fig S3A

In [ ]:
# %%
# =============================================================================
# 5B.12 - REFERENCE MODULE SCORES: UMAP (ONE PER MODULE) + BOXPLOTS
# =============================================================================
banner('STEP 5B.12 - REFERENCE MODULE SCORES')

if 'S_ref' not in globals() or 'ad_r' not in globals():
    raise RuntimeError('S_ref/ad_r not found - run 5B.1 first.')
if 'X_umap' not in adata.obsm:
    raise KeyError('No X_umap in adata.obsm; rerun step3_integrate.py.')

U_ref = adata.obsm['X_umap'][is_r]
print(f'  Reference cells: {U_ref.shape[0]}')
print(f'  Modules: {CLASSES}')

# --- one UMAP per module, blue-to-red, centered on the module's own mean ---
for ct in CLASSES:
    vals = S_ref[ct].values
    center = float(vals.mean())
    lim = float(np.abs(vals - center).max())

    fig, ax = plt.subplots(figsize=(6.5, 5.6))
    sca = ax.scatter(U_ref[:, 0], U_ref[:, 1], s=6, c=vals, cmap='RdBu_r',
                     vmin=center - lim, vmax=center + lim,
                     linewidths=0, rasterized=True)
    plt.colorbar(sca, ax=ax, label=f'{ct} module score (z)', fraction=0.045,
                pad=0.02)
    ax.set_title(f'{ct} module score, reference cells', fontweight='bold',
                fontsize=10, color=CT_COLORS.get(ct, '#333333'))
    ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    fig.tight_layout()
    save_fig(fig, f'step5b_reference_module_umap_{ct}.pdf')
    plt.close(fig)

print(f'  Wrote {len(CLASSES)} reference module-score UMAPs')



#### Fig S2E

In [ ]:
# %%
# =============================================================================
# 5B.14 - LEIDEN CLUSTERS vs REFERENCE CELL TYPE, FRACTION HEATMAP
# =============================================================================
import re
banner('STEP 5B.14 - LEIDEN CLUSTER COMPOSITION, REFERENCE CELLS')

LEIDEN_DIM = globals().get('DIM_PLOT', 25)   # falls back to the conventions'
                                             # default RNA clustering (25) if
                                             # no dim-specific run is in scope
print(f'  Using leiden_annot_{LEIDEN_DIM} (LEIDEN_DIM={LEIDEN_DIM})')

_ad = sc.read_h5ad(H5_INTEGRATED)
leiden_col = f'leiden_annot_{LEIDEN_DIM}'
if leiden_col not in _ad.obs.columns:
    raise KeyError(f'{leiden_col} not in obs. obs holds columns matching '
                   f'leiden_annot_*: '
                   f'{[c for c in _ad.obs.columns if c.startswith("leiden_annot")]}')

is_r_all = (_ad.obs['dataset'] != 'sciMeDseq').values
ref_clusters = _ad.obs.loc[_ad.obs_names[is_r_all], leiden_col].astype(str)
ref_labels = _ad.obs.loc[_ad.obs_names[is_r_all], LABEL_COL].astype(str)
del _ad

print(f'  Reference cells: {len(ref_clusters)}')
print(f'  Clusters: {ref_clusters.nunique()}   True labels: {ref_labels.nunique()}')

# --- fraction of each true cell type within each cluster (rows sum to 1) ---
ct_order = [c for c in CLASSES if c in ref_labels.unique()]
ct_order += sorted(c for c in ref_labels.unique() if c not in ct_order)

cm = pd.crosstab(ref_clusters, ref_labels).reindex(columns=ct_order, fill_value=0)
cm = cm.loc[sorted(cm.index, key=lambda x: (len(x), x))]   # numeric-ish sort
frac = cm.div(cm.sum(axis=1).replace(0, np.nan), axis=1 if False else 0)
# --- reorder clusters: by dominant cell type (CLASSES order), then by number
dominant_ct = frac.idxmax(axis=1)   # dominant true type per cluster
ct_rank = {ct: i for i, ct in enumerate(ct_order)}


def cluster_num(cid):
    m = re.search(r'\d+', cid)
    return int(m.group()) if m else 10**9   # non-numeric labels sort last


row_order = sorted(frac.index,
                   key=lambda cid: (ct_rank.get(dominant_ct[cid], 10**9),
                                    cluster_num(cid)))
frac = frac.loc[row_order]
cm = cm.loc[row_order]

print('\n  Clusters ordered by dominant type, then number:')
print(pd.DataFrame({'dominant_type': dominant_ct.loc[row_order]}).to_string())
print('\n  Cluster composition (fraction of true label per cluster):')
print(frac.round(2).to_string())
frac.to_csv(os.path.join(TAB_DIR, f'leiden_{LEIDEN_DIM}_ref_composition_fraction.csv'))
cm.to_csv(os.path.join(TAB_DIR, f'leiden_{LEIDEN_DIM}_ref_composition_counts.csv'))

fig, ax = plt.subplots(figsize=(0.9 * len(ct_order) + 3,
                                0.42 * len(frac) + 3))
sns.heatmap(frac, annot=True, fmt='.2f', cmap='RdBu_r', vmin=0, vmax=1,
            linewidths=0.4, linecolor='white', ax=ax,
            cbar_kws={'label': 'fraction of cluster'})
ax.set_xlabel('true cell type (reference)')
ax.set_ylabel(f'{leiden_col} cluster')
ax.set_title(f'Leiden cluster composition, reference cells (dim {LEIDEN_DIM})',
             fontweight='bold')
fig.tight_layout()
save_fig(fig, f'step5b_leiden_{LEIDEN_DIM}_ref_composition_heatmap.pdf')

print(f'\n  Figures: {FIG_DIR}\n  Tables: {TAB_DIR}')

#### Marker genes and cycling scores in umap Fig1D, Fig S3B

In [ ]:
marker_genes = {
    'Stem':       ['Lgr5', 'Lrig1', 'Lgr4', 'Bmi1', 'Hopx'],
    'Proliferation': ['Mki67', 'Pcna'],
    'Secretory':  ['Atoh1', 'Dll1'],
    'Paneth':     ['Lyz1'],
    'Goblet':     ['Muc2', 'Agr2'],
    'Tuft':       ['Gfi1b'],
    'Enterocyte': ['Alpi', 'Slc2a2', 'Vil1'],
    'EEC':        ['Chga', "Epcam" ], #'Neurog3'
}

# flat list if you just need all genes
all_markers = [g for genes in marker_genes.values() for g in genes]

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
markers = all_markers
sc.settings.figdir = "./"  # save in current directory# --- UMAP plots ---
blue_red = LinearSegmentedColormap.from_list(
    "white_blue_red",
    [(0, "#FFFFFF"), (0.3, "#FFFFFF"), (0.6, "#0000FF"), (1.0, "#FF0000")]
)
sc.pl.umap(_ad, color=markers, cmap="Reds", use_raw=False, size=12, save="umap_markers_epcam_glut2.png")
plt.close()
# --- Violin plot ---
sc.pl.violin(_ad, keys=markers, groupby="leiden_annot_20", save="violin_markers_epcam_glut2.png")
plt.close()
# --- Dot plot ---
sc.pl.dotplot(_ad, var_names=markers, groupby="leiden_annot_20",save="dotplot_markers_epcam_glut2.png")
plt.close()

In [ ]:
# Option 1: coolwarm — the classic publication diverging colormap
cmap = plt.cm.coolwarm

# Option 2: RdBu_r — deep blue to deep red, very common in papers
#cmap = plt.cm.RdBu_r

# Option 3: RdYlBu_r — blue → yellow → red, popular in Nature/Cell papers
#cmap = plt.cm.RdYlBu_r

# Option 4: custom white → blue → red (clean, no yellow)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list(
    "pub_blue_red",
    ["#053061", "#2166AC", "#4393C3", "#F7F7F7", "#D6604D", "#B2182B", "#67001F"]
)

# Option 5: viridis-style but blue-red (used in Seurat/Signac papers)
cmap = LinearSegmentedColormap.from_list(
    "seurat_like",
    ["#E8E8E8", "#4A78B0", "#8B0000"]
)

In [ ]:
# %%
# =============================================================================
# CELL-CYCLE SCORING ON THE INTEGRATED OBJECT
# =============================================================================
banner('CELL-CYCLE SCORES')

S_GENES_HS = [
    'MCM5', 'PCNA', 'TYMS', 'FEN1', 'MCM2', 'MCM4', 'RRM1', 'UNG', 'GINS2',
    'MCM6', 'CDCA7', 'DTL', 'PRIM1', 'UHRF1', 'MLF1IP', 'HELLS', 'RFC2',
    'RPA2', 'NASP', 'RAD51AP1', 'GMNN', 'WDR76', 'SLBP', 'CCNE2', 'UBR7',
    'POLD3', 'MSH2', 'ATAD2', 'RAD51', 'RRM2', 'CDC45', 'CDC6', 'EXO1',
    'TIPIN', 'DSCC1', 'BLM', 'CASP8AP2', 'USP1', 'CLSPN', 'POLA1', 'CHAF1B',
    'BRIP1', 'E2F8',
]
G2M_GENES_HS = [
    'HMGB2', 'CDK1', 'NUSAP1', 'UBE2C', 'BIRC5', 'TPX2', 'TOP2A', 'NDC80',
    'CKS2', 'NUF2', 'CKS1B', 'MKI67', 'TMPO', 'CENPF', 'TACC3', 'FAM64A',
    'SMC4', 'CCNB2', 'CKAP2L', 'CKAP2', 'AURKB', 'BUB1', 'KIF11', 'ANP32E',
    'TUBB4B', 'GTSE1', 'KIF20B', 'HJURP', 'CDCA3', 'HN1', 'CDC20', 'TTK',
    'CDC25C', 'KIF2C', 'RANGAP1', 'NCAPD2', 'DLGAP5', 'CDCA2', 'CDCA8',
    'ECT2', 'KIF23', 'HMMR', 'AURKA', 'PSRC1', 'ANLN', 'LBR', 'CKAP5',
    'CENPE', 'CTCF', 'NEK2', 'G2E3', 'GAS2L3', 'CBX5', 'CENPA',
]

# map to whatever casing the object uses
lookup = {g.upper(): g for g in _ad.var_names}
s_genes = [lookup[g] for g in S_GENES_HS if g in lookup]
g2m_genes = [lookup[g] for g in G2M_GENES_HS if g in lookup]
print(f'  S phase:  {len(s_genes)}/{len(S_GENES_HS)} genes found')
print(f'  G2/M:     {len(g2m_genes)}/{len(G2M_GENES_HS)} genes found')
if len(s_genes) < 10 or len(g2m_genes) < 10:
    print('  WARNING: few genes matched. Check the gene naming convention '
          f'(examples in the object: {list(_ad.var_names[:5])})')

_ad.X = _ad.layers['lognorm'].copy()
sc.tl.score_genes_cell_cycle(_ad, s_genes=s_genes, g2m_genes=g2m_genes,
                             use_raw=False)
_ad.obs['cc_difference'] = _ad.obs['S_score'] - _ad.obs['G2M_score']

_isq = (_ad.obs['dataset'] == 'sciMeDseq').values
print('\n  Phase assignment:')
print(pd.crosstab(_ad.obs['phase'], _ad.obs['dataset']).to_string())

print('\n  Scores by dataset:')
print(_ad.obs.groupby('dataset')[['S_score', 'G2M_score']]
      .agg(['mean', 'std']).round(3).to_string())

if LABEL_COL in _ad.obs.columns:
    ref = _ad.obs[~_isq]
    print(f'\n  Reference: phase by {LABEL_COL}:')
    print(pd.crosstab(ref[LABEL_COL].astype(str), ref['phase'],
                      normalize='index').round(3).to_string())

integ_col = f'leiden_annot_{BEST_DIM}_type'
if integ_col in _ad.obs.columns:
    q = _ad.obs[_isq]
    print(f'\n  sciMeDseq: phase by {integ_col}:')
    print(pd.crosstab(q[integ_col].astype(str), q['phase']).to_string())

# --- the requested UMAP plot -------------------------------------------------
sc.settings.figdir = FIG_DIR
cmap = 'RdBu_r'
all_markers = ['S_score', 'G2M_score', 'cc_difference', 'phase',
               'dataset', LABEL_COL, integ_col]
all_markers = [c for c in all_markers if c in _ad.obs.columns]

for d in sorted({BEST_DIM, 35}):
    key = f'X_umap_dim{d}'
    if key not in _ad.obsm:
        continue
    _ad.obsm['X_umap'] = _ad.obsm[key].copy()
    for ext in ('png', 'pdf'):
        sc.pl.umap(_ad, color=all_markers, cmap=cmap, use_raw=False, size=12,
                   ncols=3, frameon=False,
                   save=f'_markers_all_dim{d}.{ext}', show=False)
    print(f'  Wrote umap_markers_all_dim{d}')

# --- query cells only, so 516 don't vanish under 17k reference ---------------
_ad.obsm['X_umap'] = _ad.obsm[f'X_umap_dim{BEST_DIM}'].copy()
U = _ad.obsm['X_umap']
q_idx = np.where(_isq)[0]

fig, axes = plt.subplots(1, 3, figsize=(17, 5.2))
for ax, col in zip(axes, ['S_score', 'G2M_score', 'cc_difference']):
    v = _ad.obs[col].values[q_idx]
    lim = float(np.nanpercentile(np.abs(v), 98))
    ax.scatter(U[~_isq, 0], U[~_isq, 1], s=2, color='#EEEEEE', alpha=0.5,
               linewidths=0, rasterized=True, zorder=1)
    sca = ax.scatter(U[q_idx, 0], U[q_idx, 1], s=26, c=v, cmap=cmap,
                     vmin=-lim, vmax=lim, edgecolors='white', linewidths=0.3,
                     rasterized=True, zorder=3)
    plt.colorbar(sca, ax=ax, label=col)
    ax.set_title(f'{col}, sciMeDseq cells', fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
    for s_ in ax.spines.values():
        s_.set_visible(False)
fig.tight_layout()
save_fig(fig, f'cellcycle_query_dim{BEST_DIM}.pdf')

# --- does it track depth, and does it track the proliferative compartment? ---
from scipy.stats import spearmanr
depth_q = pd.to_numeric(_ad.obs['total_counts'], errors='coerce').values[q_idx]
ld = np.log10(np.maximum(depth_q, 1))
for col in ['S_score', 'G2M_score']:
    rho = spearmanr(ld, _ad.obs[col].values[q_idx], nan_policy='omit')[0]
    print(f'  {col} vs log10 depth in the query: rho = {rho:+.3f}')

if integ_col in _ad.obs.columns:
    qdf = pd.DataFrame({
        'integration': _ad.obs[integ_col].astype(str).values[q_idx],
        'S_score': _ad.obs['S_score'].values[q_idx],
        'G2M_score': _ad.obs['G2M_score'].values[q_idx],
    })
    order = [c for c in LABEL_ORDER if (qdf['integration'] == c).any()]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, col in zip(axes, ['S_score', 'G2M_score']):
        sns.boxplot(data=qdf, x='integration', y=col, order=order,
                    hue='integration', hue_order=order, legend=False,
                    palette=[CT_COLORS.get(c, '#777777') for c in order],
                    fliersize=1.5, linewidth=0.9, ax=ax)
        ax.axhline(0, color='k', ls=':', lw=0.8)
        ax.set_title(f'{col} by {integ_col}', fontweight='bold')
        ax.set_xlabel(''); ax.tick_params(axis='x', rotation=45)
        for lbl in ax.get_xticklabels():
            lbl.set_ha('right')
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    fig.tight_layout()
    save_fig(fig, 'cellcycle_by_integration.pdf')

_ad.obs[['S_score', 'G2M_score', 'cc_difference', 'phase']].to_csv(
    os.path.join(TAB_DIR, 'cellcycle_scores.csv'))
print(f'\n  Figures: {FIG_DIR}\n  Tables: {TAB_DIR}')

In [ ]:
# %%
# =============================================================================
# 5B.18 - CLUSTER sciMeDseq ALONE, THEN TEST THE MARKERS AGAINST THE REFERENCE
# =============================================================================
banner('STEP 5B.18 - INDEPENDENT CLUSTERING OF THE DCM CELLS')

from scipy.stats import hypergeom, spearmanr
from statsmodels.stats.multitest import multipletests

DCM_RESOLUTIONS = [0.4, 0.6, 0.8]
DCM_RES = 0.4              # the one used for the marker analysis
DCM_N_HVG = 2000
DCM_N_PCS = 20
CLUSTER_MARKER_PADJ = 0.05
CLUSTER_MARKER_LOGFC = 0.25
CLUSTER_MARKER_TOP = 1000   # cap per cluster before the overlap test

_ad = sc.read_h5ad(H5_INTEGRATED)
_isq = (_ad.obs['dataset'] == 'sciMeDseq').values

# ------------------------------------------ reference DE, ALL significant ----
banner('5B.18a - REFERENCE DE (all significant, no uniqueness filter)')

ad_de = _ad[~_isq].copy()
ad_de.X = ad_de.layers['lognorm'].copy()
ad_de.obs[LABEL_COL] = ad_de.obs[LABEL_COL].astype(str).astype('category')
print(f'  Reference: {ad_de.n_obs} cells x {ad_de.n_vars} genes')
print(ad_de.obs[LABEL_COL].value_counts().to_string())

sc.tl.rank_genes_groups(ad_de, groupby=LABEL_COL, method='t-test', pts=True)
r = ad_de.uns['rank_genes_groups']
de_all = pd.concat([
    pd.DataFrame({'gene': r['names'][ct], 'cell_type': ct,
                  'logfc': r['logfoldchanges'][ct],
                  'pvals_adj': r['pvals_adj'][ct]})
    for ct in r['names'].dtype.names], ignore_index=True)

sig = de_all[(de_all['pvals_adj'] < DE_PADJ_MAX) &
             (de_all['logfc'] >= DE_LOGFC_MIN)].copy()
ref_sets = {ct: set(gg['gene']) for ct, gg in sig.groupby('cell_type')}
print(f'\n  Significant DE genes (padj < {DE_PADJ_MAX}, logFC >= '
      f'{DE_LOGFC_MIN}), NOT restricted to unique:')
for ct in LABEL_ORDER:
    n = len(ref_sets.get(ct, set()))
    print(f'    {ct:>12}: {n:>5}')
n_shared_genes = sum(len(v) for v in ref_sets.values())
n_uniq = len(set().union(*ref_sets.values())) if ref_sets else 0
print(f'    total assignments {n_shared_genes}, distinct genes {n_uniq} '
      f'(a gene can belong to several sets)')
sig.to_csv(os.path.join(TAB_DIR, 'reference_de_all_significant.csv'),
           index=False)
del ad_de

# --------------------------------------------- cluster the DCM cells alone ---
banner('5B.18b - CLUSTERING sciMeDseq ON ITS OWN')

ad_q2 = _ad[_isq].copy()
ad_q2.X = ad_q2.layers['lognorm'].copy()
for k in list(ad_q2.obsm.keys()):
    del ad_q2.obsm[k]
print(f'  {ad_q2.n_obs} cells x {ad_q2.n_vars} genes')

sc.pp.highly_variable_genes(ad_q2, n_top_genes=DCM_N_HVG)
print(f'  HVGs: {int(ad_q2.var["highly_variable"].sum())}')
ad_q2.raw = ad_q2
sc.pp.scale(ad_q2, max_value=10)
sc.tl.pca(ad_q2, n_comps=min(50, ad_q2.n_obs - 1))
sc.pp.neighbors(ad_q2, n_pcs=DCM_N_PCS)
sc.tl.umap(ad_q2)
for rr in DCM_RESOLUTIONS:
    sc.tl.leiden(ad_q2, resolution=rr, key_added=f'dcm_leiden_{rr}',
                 random_state=SEED)
    print(f'  resolution {rr}: {ad_q2.obs[f"dcm_leiden_{rr}"].nunique()} clusters')
CL = f'dcm_leiden_{DCM_RES}'
ad_q2.obs['dcm_cluster'] = ad_q2.obs[CL].astype(str)
ad_q2.X = ad_q2.raw.X.copy()          # back to lognorm for DE
clusters = sorted(ad_q2.obs['dcm_cluster'].unique(),
                  key=lambda x: int(x) if x.isdigit() else x)
print(f'\n  Using {CL}: {len(clusters)} clusters')
print(ad_q2.obs['dcm_cluster'].value_counts().sort_index().to_string())

# is the clustering just depth?
depth = pd.to_numeric(ad_q2.obs['total_counts'], errors='coerce').values
ld = np.log10(np.maximum(depth, 1))
dep = pd.DataFrame({'cluster': ad_q2.obs['dcm_cluster'].values,
                    'log_depth': ld}).groupby('cluster')['log_depth'].median()
spread = float(dep.max() - dep.min())
print(f'\n  Median log10 depth per cluster (spread {spread:.2f} logs):')
print(dep.round(2).to_string())
if spread > 0.5:
    print('  WARNING: clusters differ by more than 3x in median depth. The')
    print('  clustering may be separating cells by coverage, not identity.')

# ---------------------------------------------------- cluster marker genes ---
banner('5B.18c - CLUSTER MARKERS')

sc.tl.rank_genes_groups(ad_q2, groupby='dcm_cluster', method='wilcoxon',
                        pts=True)
rq = ad_q2.uns['rank_genes_groups']
cl_de = pd.concat([
    pd.DataFrame({'gene': rq['names'][c], 'cluster': c,
                  'logfc': rq['logfoldchanges'][c],
                  'pvals_adj': rq['pvals_adj'][c]})
    for c in rq['names'].dtype.names], ignore_index=True)
cl_sig = cl_de[(cl_de['pvals_adj'] < CLUSTER_MARKER_PADJ) &
               (cl_de['logfc'] >= CLUSTER_MARKER_LOGFC)].copy()
cl_top = (cl_sig.sort_values(['cluster', 'logfc'], ascending=[True, False])
          .groupby('cluster').head(CLUSTER_MARKER_TOP))
cl_sets = {c: set(gg['gene']) for c, gg in cl_top.groupby('cluster')}
print(f'  Markers per cluster (padj < {CLUSTER_MARKER_PADJ}, '
      f'logFC >= {CLUSTER_MARKER_LOGFC}, top {CLUSTER_MARKER_TOP}):')
for c in clusters:
    print(f'    cluster {c:>3}: {len(cl_sets.get(c, set())):>4} markers')
cl_top.to_csv(os.path.join(TAB_DIR, 'dcm_cluster_markers.csv'), index=False)

empty = [c for c in clusters if len(cl_sets.get(c, set())) == 0]
if empty:
    print(f'  Clusters with no significant markers: {empty}')

# --------------------------------------------------- overlap with reference --
banner('5B.18d - OVERLAP WITH THE REFERENCE DE SETS')

universe = set(ad_q2.var_names) & set().union(*ref_sets.values()) \
    if ref_sets else set()
M = len(set(ad_q2.var_names))
print(f'  Gene universe: {M}')

rows = []
for c in clusters:
    cs = cl_sets.get(c, set()) & set(ad_q2.var_names)
    for ct in LABEL_ORDER:
        rs = ref_sets.get(ct, set()) & set(ad_q2.var_names)
        k = len(cs & rs)
        exp = len(cs) * len(rs) / M if M else np.nan
        p = hypergeom.sf(k - 1, M, len(rs), len(cs)) if cs and rs else 1.0
        rows.append({'cluster': c, 'cell_type': ct,
                     'n_cluster_markers': len(cs), 'n_ref_de': len(rs),
                     'overlap': k, 'expected': round(exp, 1),
                     'fold': round(k / exp, 2) if exp and exp > 0 else np.nan,
                     'pval': p,
                     'genes': ','.join(sorted(cs & rs)[:20])})
ov = pd.DataFrame(rows)
ov['padj'] = multipletests(ov['pval'], method='fdr_bh')[1]
ov.to_csv(os.path.join(TAB_DIR, 'dcm_cluster_vs_reference_overlap.csv'),
          index=False)

fold = ov.pivot(index='cluster', columns='cell_type', values='fold') \
    .reindex(index=clusters, columns=LABEL_ORDER)
padj = ov.pivot(index='cluster', columns='cell_type', values='padj') \
    .reindex(index=clusters, columns=LABEL_ORDER)
ovn = ov.pivot(index='cluster', columns='cell_type', values='overlap') \
    .reindex(index=clusters, columns=LABEL_ORDER)

print('\n  Fold enrichment (observed / expected overlap):')
print(fold.round(2).to_string())
print('\n  Significant at BH < 0.05:')
hits = ov[(ov['padj'] < 0.05) & (ov['overlap'] > 0)].sort_values('padj')
if len(hits):
    print(hits[['cluster', 'cell_type', 'overlap', 'expected', 'fold',
                'padj']].head(25).to_string(index=False))
else:
    print('  None. No cluster marker set is enriched for any reference cell '
          'type beyond chance.')

annot = np.where(padj.values < 0.05,
                 ovn.values.astype(int).astype(str) + '*',
                 ovn.values.astype(int).astype(str))
fig, ax = plt.subplots(figsize=(1.1 * len(LABEL_ORDER) + 5,
                                0.45 * len(clusters) + 3.5))
sns.heatmap(fold, annot=annot, fmt='', cmap='Reds', vmin=0,
            vmax=max(2.0, float(np.nanpercentile(fold.values, 95))),
            linewidths=0.5, linecolor='white', ax=ax,
            cbar_kws={'label': 'fold enrichment'})
ax.set_xlabel('reference cell type'); ax.set_ylabel('sciMeDseq cluster')
ax.set_title('Cluster markers vs reference DE genes\n'
             '(numbers = overlapping genes, * = BH < 0.05)', fontweight='bold')
fig.tight_layout()
save_fig(fig, 'step5b_dcm_cluster_vs_reference_overlap.pdf')

# ------------------------------------------------------------------ context --
banner('5B.18e - CLUSTER CONTEXT')

best = {}
for c in clusters:
    sub = ov[(ov['cluster'] == c) & (ov['overlap'] > 0)]
    if len(sub) and sub['padj'].min() < 0.05:
        row = sub.loc[sub['padj'].idxmin()]
        best[c] = f"{row['cell_type']} (fold {row['fold']}, padj {row['padj']:.1e})"
    else:
        best[c] = 'no significant match'
print('  Best reference match per cluster:')
for c in clusters:
    print(f'    cluster {c:>3}: {best[c]}')

ad_q2.obs['gmm_call'] = res.loc[ad_q2.obs_names, 'gmm_call'].astype(str).values
integ_col = f'leiden_annot_{BEST_DIM}_type'
if integ_col in _ad.obs.columns:
    ad_q2.obs['integration'] = _ad.obs.loc[ad_q2.obs_names,
                                           integ_col].astype(str).values
    xt_i = pd.crosstab(ad_q2.obs['dcm_cluster'], ad_q2.obs['integration'])
    print(f'\n  DCM cluster vs {integ_col}:')
    print(xt_i.to_string())
    xt_i.to_csv(os.path.join(TAB_DIR, 'dcm_cluster_vs_integration.csv'))
xt_g = pd.crosstab(ad_q2.obs['dcm_cluster'], ad_q2.obs['gmm_call'])
print('\n  DCM cluster vs GMM call:')
print(xt_g.to_string())
xt_g.to_csv(os.path.join(TAB_DIR, 'dcm_cluster_vs_gmm.csv'))

ad_q2.obs['log_depth'] = ld
sc.settings.figdir = FIG_DIR
cols = ['dcm_cluster', 'log_depth', 'n_genes_by_counts', 'gmm_call'] + \
       (['integration'] if 'integration' in ad_q2.obs else [])
cols = [c for c in cols if c in ad_q2.obs.columns]
for ext in ('pdf', 'png'):
    sc.pl.umap(ad_q2, color=cols, frameon=False, ncols=3,
               save=f'_step5b_dcm_only.{ext}', show=False)

r_dep = spearmanr(ld, ad_q2.obsm['X_umap'][:, 0])[0]
r_dep2 = spearmanr(ld, ad_q2.obsm['X_umap'][:, 1])[0]
print(f'\n  Depth vs the DCM-only UMAP axes: rho = {r_dep:+.3f} / {r_dep2:+.3f}')

ad_q2.write_h5ad(os.path.join(OBJ_DIR, '04_dcm_only_clustered.h5ad'))
del _ad
print(f'\n  Saved 04_dcm_only_clustered.h5ad')
print(f'  Figures: {FIG_DIR}\n  Tables: {TAB_DIR}')l

In [ ]:
# %%
# =============================================================================
# 5B.19 - THE OVERLAPPING GENES, AND MODULE SCORES BUILT FROM THEM
# =============================================================================
banner('STEP 5B.19 - OVERLAP GENE TABLE AND DATA-DRIVEN MODULE SCORES')

OVERLAP_PADJ = 1       # a pair counts as enriched below this BH-adjusted p
OVERLAP_MIN_GENES = 1    # minimum genes for a set to be scored

if OVERLAP_PADJ > 0.05:
    print(f'  NOTE: OVERLAP_PADJ = {OVERLAP_PADJ}. At this cutoff a large share '
          f'of pairs pass by chance, so the resulting sets are exploratory and '
          f'not statistically supported.')
if OVERLAP_MIN_GENES < 3:
    print(f'  NOTE: OVERLAP_MIN_GENES = {OVERLAP_MIN_GENES}. Module scores over '
          f'1-2 genes are extremely noisy.')

# ---------------------------------------------- the full overlap gene table --
rows = []
for _, r_ in ov.iterrows():
    c, ct = r_['cluster'], r_['cell_type']
    shared = sorted(cl_sets.get(c, set()) & ref_sets.get(ct, set()))
    if not shared:
        continue
    cm = cl_top[cl_top['cluster'] == c].set_index('gene')
    rm = sig[sig['cell_type'] == ct].set_index('gene')
    for gene in shared:
        rows.append({
            'gene': gene, 'cluster': c, 'cell_type': ct,
            'cluster_logfc': round(float(cm.loc[gene, 'logfc']), 3)
                             if gene in cm.index else np.nan,
            'cluster_padj': float(cm.loc[gene, 'pvals_adj'])
                            if gene in cm.index else np.nan,
            'ref_logfc': round(float(rm.loc[gene, 'logfc']), 3)
                         if gene in rm.index else np.nan,
            'ref_padj': float(rm.loc[gene, 'pvals_adj'])
                        if gene in rm.index else np.nan,
            'pair_fold': r_['fold'], 'pair_padj': r_['padj'],
            'pair_enriched': bool(r_['padj'] < OVERLAP_PADJ),
            'pair_enriched_strict': bool(r_['padj'] < 0.05),
        })
ovg = pd.DataFrame(rows)
if ovg.empty:
    raise ValueError('No overlapping genes between the cluster markers and the '
                     'reference DE sets.')
ovg = ovg.sort_values(['cell_type', 'pair_padj', 'cluster_logfc'],
                      ascending=[True, True, False])
ovg.to_csv(os.path.join(TAB_DIR, 'overlap_genes_cluster_vs_reference.csv'),
           index=False)

print(f'\n  Overlapping gene x cluster x cell-type rows: {len(ovg)}')
print(f'  From pairs at BH < {OVERLAP_PADJ}: '
      f'{int(ovg["pair_enriched"].sum())}')
print(f'  From pairs at BH < 0.05:          '
      f'{int(ovg["pair_enriched_strict"].sum())}')

wide = (ovg.groupby(['cell_type', 'cluster'])['gene']
        .apply(lambda s: ', '.join(sorted(s)))
        .unstack().reindex(index=LABEL_ORDER, columns=clusters))
wide.to_csv(os.path.join(TAB_DIR, 'overlap_genes_wide.csv'))

print(f'\n  Overlapping genes per cell type '
      f'(* = BH < {OVERLAP_PADJ}, ** = BH < 0.05):')
for ct in LABEL_ORDER:
    sub = ovg[ovg['cell_type'] == ct]
    if sub.empty:
        print(f'\n    {ct}: no overlapping genes')
        continue
    print(f'\n    {ct}: {sub["gene"].nunique()} distinct genes across '
          f'{sub["cluster"].nunique()} cluster(s)')
    for c, g_ in sub.groupby('cluster'):
        star = ('**' if g_['pair_enriched_strict'].iloc[0]
                else '* ' if g_['pair_enriched'].iloc[0] else '  ')
        genes = sorted(g_['gene'].unique())
        shown = ', '.join(genes[:12]) + (f' ... (+{len(genes) - 12})'
                                         if len(genes) > 12 else '')
        print(f'      {star} cluster {c:>3} (fold {g_["pair_fold"].iloc[0]}, '
              f'padj {g_["pair_padj"].iloc[0]:.2e}, n={len(genes)}): {shown}')

# ------------------------------------------- build the data-driven gene sets --
banner('5B.19b - GENE SETS FROM THE OVERLAPS')

src = ovg[ovg['pair_enriched']] if ovg['pair_enriched'].any() else ovg
if not ovg['pair_enriched'].any():
    print(f'  No pair reached BH < {OVERLAP_PADJ}; using every overlapping '
          f'gene instead.')

overlap_sets, dropped_sets = {}, []
for ct in LABEL_ORDER:
    g_ = sorted(src.loc[src['cell_type'] == ct, 'gene'].unique())
    if g_ and len(g_) >= max(OVERLAP_MIN_GENES, 1):
        overlap_sets[ct] = g_
    else:
        dropped_sets.append((ct, len(g_)))

print(f'\n  Sets built: {list(overlap_sets)}')
for ct in LABEL_ORDER:
    if ct in overlap_sets:
        n_strict = len(ovg.loc[ovg['pair_enriched_strict'] &
                               (ovg['cell_type'] == ct), 'gene'].unique())
        note = f'  ({n_strict} of them from BH < 0.05 pairs)' if n_strict else \
               '  (none from a BH < 0.05 pair)'
        print(f'    {ct:>12}: {len(overlap_sets[ct]):>4} genes{note}')
    else:
        n_any = dict(dropped_sets).get(ct, 0)
        print(f'    {ct:>12}: not scored ({n_any} overlapping gene(s))')

if not overlap_sets:
    raise ValueError('No cell type has any overlapping genes to score.')

pd.DataFrame([{'cell_type': ct, 'gene': g_}
              for ct, gl in overlap_sets.items() for g_ in gl]).to_csv(
    os.path.join(TAB_DIR, 'overlap_derived_gene_sets.csv'), index=False)

print('\n  How many of the new genes were already literature markers:')
for ct in overlap_sets:
    lit = set(gene_sets.get(ct, []))
    new = set(overlap_sets[ct])
    print(f'    {ct:>12}: {len(new & lit):>3} of {len(new)} '
          f'({100 * len(new & lit) / len(new):.0f}%)')

# --------------------------------------------------- score on the full object -
banner('5B.19c - SCORING AND PLOTTING')

_ad = sc.read_h5ad(H5_INTEGRATED)
_isq = (_ad.obs['dataset'] == 'sciMeDseq').values
OSET = [c for c in LABEL_ORDER if c in overlap_sets]
print(f'  Scoring {len(OSET)} sets: {OSET}')

S_ov_q = None
for tag, mask in [('reference', ~_isq), ('query', _isq)]:
    sub = _ad[mask].copy()
    sub.X = sub.layers['lognorm'].copy()
    for ct in OSET:
        gl = [g_ for g_ in overlap_sets[ct] if g_ in sub.var_names]
        sc.tl.score_genes(sub, gene_list=gl, score_name=f'ov_{ct}',
                          use_raw=False)
    S = sub.obs[[f'ov_{ct}' for ct in OSET]].astype(float)
    S.columns = OSET
    Sz = scale_block(S)
    for ct in OSET:
        col = f'ov_{ct}_z'
        if col not in _ad.obs:
            _ad.obs[col] = np.nan
        _ad.obs.loc[sub.obs_names, col] = Sz[ct].values
    print(f'  {tag}: scored {sub.n_obs} cells')
    if tag == 'query':
        S_ov_q = Sz.copy()

zcols = [f'ov_{ct}_z' for ct in OSET]
sc.settings.figdir = FIG_DIR
ncol = min(3, len(OSET))
nrow = int(np.ceil(len(OSET) / ncol))

for d in sorted({BEST_DIM, 35}):
    key = f'X_umap_dim{d}'
    if key not in _ad.obsm:
        print(f'  {key} not stored; skipping')
        continue
    _ad.obsm['X_umap'] = _ad.obsm[key].copy()
    for ext in ('pdf', 'png'):
        sc.pl.umap(_ad, color=zcols, cmap='RdBu_r', vmin=-3, vmax=3,
                   frameon=False, ncols=ncol,
                   save=f'_step5b_overlap_scores_dim{d}.{ext}', show=False)
    print(f'  Feature plots written for dim {d}')

_ad.obsm['X_umap'] = _ad.obsm[f'X_umap_dim{BEST_DIM}'].copy()
U = _ad.obsm['X_umap']
q_ord = pd.Index(_ad.obs_names[_isq]).get_indexer(res.index)
Cq = U[_isq][q_ord]

fig, axes = plt.subplots(nrow, ncol, figsize=(5.2 * ncol, 4.6 * nrow),
                         squeeze=False)
axes = axes.ravel()
for ax, ct in zip(axes, OSET):
    ax.scatter(U[~_isq, 0], U[~_isq, 1], s=2, color='#EEEEEE', alpha=0.5,
               linewidths=0, rasterized=True, zorder=1)
    sca = ax.scatter(Cq[:, 0], Cq[:, 1], s=26, c=S_ov_q[ct].values,
                     cmap='RdBu_r', vmin=-3, vmax=3, edgecolors='white',
                     linewidths=0.3, rasterized=True, zorder=3)
    plt.colorbar(sca, ax=ax, label='z')
    ax.set_title(f'{ct}  ({len(overlap_sets[ct])} genes)',
                 fontweight='bold', fontsize=10,
                 color=CT_COLORS.get(ct, '#333333'))
    ax.set_xticks([]); ax.set_yticks([])
    for s_ in ax.spines.values():
        s_.set_visible(False)
for ax in axes[len(OSET):]:
    ax.axis('off')
fig.suptitle(f'Overlap-derived module scores, sciMeDseq cells '
             f'(dim-{BEST_DIM} embedding)', fontweight='bold', y=1.005)
fig.tight_layout()
save_fig(fig, 'step5b_overlap_scores_query.pdf')

# ---------------------------------------------- do these scores separate? -----
banner('5B.19d - DO THE NEW SCORES SEPARATE ANYTHING?')

integ = _ad.obs.loc[res.index, f'leiden_annot_{BEST_DIM}_type'].astype(str)
long_ov = S_ov_q.copy()
long_ov['integration'] = integ.values
long_ov['dcm_cluster'] = ad_q2.obs.loc[res.index, 'dcm_cluster'].values

for grp_col, title in [('integration', f'integration (dim {BEST_DIM})'),
                       ('dcm_cluster', 'DCM-only cluster')]:
    order = ([c for c in LABEL_ORDER if (long_ov[grp_col] == c).any()]
             if grp_col == 'integration'
             else sorted(long_ov[grp_col].unique(),
                         key=lambda x: int(x) if x.isdigit() else x))
    fig, axes = plt.subplots(nrow, ncol, figsize=(5.4 * ncol, 4.2 * nrow),
                             sharey=True, squeeze=False)
    axes = axes.ravel()
    for ax, ct in zip(axes, OSET):
        pal = ['#CFCFCF'] * len(order)
        if ct in order:
            pal[order.index(ct)] = CT_COLORS.get(ct, '#777777')
        sns.boxplot(data=long_ov, x=grp_col, y=ct, order=order,
                    hue=grp_col, hue_order=order, legend=False, palette=pal,
                    fliersize=1.5, linewidth=0.9, ax=ax)
        ax.axhline(0, color='k', ls=':', lw=0.8)
        ymin = ax.get_ylim()[0]
        for i, grp in enumerate(order):
            ax.text(i, ymin, f'{int((long_ov[grp_col] == grp).sum())}',
                    ha='center', va='bottom', fontsize=6.5, color='#555555')
        if ct in order:
            ax.get_xticklabels()[order.index(ct)].set_fontweight('bold')
        ax.set_title(f'{ct} overlap module', fontweight='bold', fontsize=10,
                     color=CT_COLORS.get(ct, '#333333'))
        ax.set_xlabel(''); ax.set_ylabel('z-score')
        ax.tick_params(axis='x', rotation=45, labelsize=7.5)
        for lbl in ax.get_xticklabels():
            lbl.set_ha('right')
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    for ax in axes[len(OSET):]:
        ax.axis('off')
    fig.suptitle(f'Overlap-derived scores grouped by {title}',
                 fontweight='bold', y=1.005)
    fig.tight_layout()
    save_fig(fig, f'step5b_overlap_scores_by_{grp_col}.pdf')

diag = []
for ct in OSET:
    if ct not in set(long_ov['integration']):
        diag.append({'cell_type': ct, 'n': 0, 'own_median': np.nan,
                     'best_other_type': None, 'best_other': np.nan,
                     'gap': np.nan})
        continue
    g_ = long_ov[long_ov['integration'] == ct]
    own = float(g_[ct].median())
    others = {o: float(g_[o].median()) for o in OSET if o != ct}
    diag.append({'cell_type': ct, 'n': len(g_), 'own_median': round(own, 3),
                 'best_other_type': max(others, key=others.get) if others else None,
                 'best_other': round(max(others.values()), 3) if others else np.nan,
                 'gap': round(own - max(others.values()), 3) if others else np.nan})
dg = pd.DataFrame(diag)
print('  Diagonal against the integration labels:')
print(dg.to_string(index=False))
dg.to_csv(os.path.join(TAB_DIR, 'overlap_score_diagonal.csv'), index=False)

good = dg[(dg['gap'] > 0) & (dg['n'] >= 5)]
print(f'\n  {len(good)}/{len(dg)} scored types are highest on their own module '
      f'(n>=5): {good["cell_type"].tolist()}')
print('  NOTE: the DCM-cluster panel is circular - these genes were selected')
print('  from those clusters. The integration panel is the real test.')

_ad.write_h5ad(H5_INTEGRATED)
del _ad
print(f'\n  Overlap scores stored as {zcols}')
print(f'  Figures: {FIG_DIR}\n  Tables: {TAB_DIR}')

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
markers = [ "Slc2a2"]
sc.settings.figdir = "./"  # save in current directory# --- UMAP plots ---
blue_red = LinearSegmentedColormap.from_list(
    "white_blue_red",
    [(0, "#FFFFFF"), (0.3, "#FFFFFF"), (0.6, "#0000FF"), (1.0, "#FF0000")]
)
sc.pl.umap(ad_q2, color=markers, cmap="Reds", use_raw=False, size=12, save="umap_markers_epcam_glut2.png")
plt.close()
# --- Violin plot ---
sc.pl.violin(ad_q2, keys=markers, groupby="leiden_annot_35", save="violin_markers_epcam_glut2.png")
plt.close()
# --- Dot plot ---
sc.pl.dotplot(ad_q2, var_names=markers, groupby="leiden_annot_35",save="dotplot_markers_epcam_glut2.png")
plt.close()


In [ ]:
marker_genes = {
    'Stem':       ['Lgr5', 'Lrig1', 'Lgr4', 'Bmi1', 'Hopx'],
    'Proliferation': ['Mki67', 'Pcna'],
    'Secretory':  ['Atoh1', 'Dll1'],
    'Paneth':     ['Lyz1'],
    'Goblet':     ['Muc2', 'Agr2'],
    'Tuft':       ['Gfi1b'],
    'Enterocyte': ['Alpi', 'Slc2a2', 'Vil1'],
    'EEC':        ['Chga', "Epcam" ], #'Neurog3'
}

# flat list if you just need all genes
all_markers = [g for genes in marker_genes.values() for g in genes]

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
markers = ["Neurog3", "Slc2a2"]
sc.settings.figdir = "./"  # save in current directory# --- UMAP plots ---
blue_red = LinearSegmentedColormap.from_list(
    "white_blue_red",
    [(0, "#FFFFFF"), (0.3, "#FFFFFF"), (0.6, "#0000FF"), (1.0, "#FF0000")]
)
sc.pl.umap(ad_q2, color=markers, cmap="Reds", use_raw=False, size=12, save="umap_markers_epcam_glut2.png")
plt.close()
# --- Violin plot ---
sc.pl.violin(ad_q2, keys=markers, groupby="leiden_annot_20", save="violin_markers_epcam_glut2.png")
plt.close()
# --- Dot plot ---
sc.pl.dotplot(ad_q2, var_names=markers, groupby="leiden_annot_20",save="dotplot_markers_epcam_glut2.png")
plt.close()

In [ ]:
# Option 1: coolwarm — the classic publication diverging colormap
cmap = plt.cm.coolwarm

# Option 2: RdBu_r — deep blue to deep red, very common in papers
#cmap = plt.cm.RdBu_r

# Option 3: RdYlBu_r — blue → yellow → red, popular in Nature/Cell papers
#cmap = plt.cm.RdYlBu_r

# Option 4: custom white → blue → red (clean, no yellow)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list(
    "pub_blue_red",
    ["#053061", "#2166AC", "#4393C3", "#F7F7F7", "#D6604D", "#B2182B", "#67001F"]
)

# Option 5: viridis-style but blue-red (used in Seurat/Signac papers)
cmap = LinearSegmentedColormap.from_list(
    "seurat_like",
    ["#E8E8E8", "#4A78B0", "#8B0000"]
)

In [ ]:
sc.pl.umap(ad_q2, color=all_markers, cmap=cmap, use_raw=False, size=12,
           save="umap_markers_all_scimedseq.png")